## 1. 安裝環境與匯入套件


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import subprocess
import sys
from pathlib import Path
import torch

'''
Set the local ComfyUI installation directory.
設定本地 ComfyUI 安裝目錄。
'''
ROOT = Path('/kaggle/working')

'''
Isolate MiniMax H3 from existing ComfyUI projects.
將 MiniMax H3 與其他 ComfyUI 專案隔離。
'''
COMFY = ROOT / 'ComfyUI-MiniMaxH3'

'''
Keep large weights in scratch storage.
大型模型權重使用暫存磁碟。
'''
STORE = Path('/tmp/minimax_h3_models')

'''
Reserve a dedicated local service port.
使用獨立的本地服務連接埠。
'''
PORT = 8193

'''
Stop early if Kaggle GPU T4 x2 is not enabled.
未啟用 Kaggle T4 雙 GPU 時提前停止。
'''
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, '請啟用 Kaggle T4 x2 GPU。'

'''
Show the actual GPU model and VRAM size.
顯示 GPU 型號與顯示記憶體容量。
'''
for i in range(2):

    '''
    Report the hardware used by this notebook.
    顯示 Notebook 實際使用的 GPU。
    '''
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}, {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GiB')

'''
Define the repositories needed for the local workflow.
定義本地 Workflow 需要的程式庫。
'''
repositories = {
    COMFY: 'https://github.com/Comfy-Org/ComfyUI.git',
    COMFY / 'custom_nodes/ComfyUI-GGUF': 'https://github.com/leejet/ComfyUI-GGUF.git',
    COMFY / 'custom_nodes/ComfyUI-ClipProj': 'https://github.com/nicolab28/ComfyUI-ClipProj.git',
}

'''
Use the verified stable ComfyUI release instead of its moving default branch.
使用已確認的 ComfyUI 穩定發行版，不跟隨隨時變動的預設分支。
'''
H3_COMFY_REF = 'v0.45.0'

'''
Preserve prior commits to detect code changes while a model server is running.
記錄舊版 commit，以判斷是否需重啟已運作的模型服務。
'''
old_commits = {}

'''
Install missing sources and update only clean repositories.
安裝缺少的程式庫，僅更新沒有本機修改的倉庫。
'''
for directory, url in repositories.items():

    '''
    Reject existing non-Git directories to avoid deleting user files.
    拒絕來源不明的現有資料夾，避免刪除使用者資料。
    '''
    if directory.exists() and not (directory / '.git').is_dir():

        '''
        Prevent overwriting an unrecognized checkout.
        不覆寫無法辨識的專案目錄。
        '''
        raise RuntimeError(f'目錄不是 Git 倉庫：{directory}')

    '''
    Record a previously installed version.
    記錄已安裝版本。
    '''
    if (directory / '.git').is_dir():

        '''
        Read the current commit for change detection.
        取得目前 commit 以判斷版本是否變更。
        '''
        old_commits[directory] = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()
    else:

        '''
        Create the destination's parent directory.
        建立目標目錄的上層資料夾。
        '''
        directory.parent.mkdir(parents=True, exist_ok=True)

        '''
        Clone the repository; pin ComfyUI to the stable release.
        Clone 程式庫，並讓 ComfyUI 固定於穩定發行版。
        '''
        command = ['git', 'clone', '--depth', '1']

        '''
        Choose the release tag only for ComfyUI core.
        只有 ComfyUI 核心指定發行版標籤。
        '''
        if directory == COMFY:

            '''
            Add the verified ComfyUI release tag.
            加入經查證的 ComfyUI 版本標籤。
            '''
            command += ['--branch', H3_COMFY_REF]

        '''
        Install the source without touching model files.
        安裝原始碼，不變更模型權重。
        '''
        subprocess.run(command + [url, str(directory)], check=True)

    '''
    The archived or patched GGUF loader may contain local compatibility changes.
    GGUF 載入器可能含有必要的本機相容性修正。
    '''
    if directory.name == 'ComfyUI-GGUF':

        '''
        Inspect local modifications before a fast-forward update.
        先檢查本機修改，再進行 fast-forward 更新。
        '''
        dirty = subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True).strip()

        '''
        Update only when no manual compatibility patch would be lost.
        只有不會遺失手動修正時才更新。
        '''
        if not dirty:

            '''
            Bring the GGUF loader to its latest upstream commit.
            將 GGUF 載入器更新至上游最新 commit。
            '''
            subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', '-q'], check=True)
        else:

            '''
            Avoid overwriting a custom H3 architecture patch.
            避免覆寫自訂的 H3 架構相容修正。
            '''
            print('GGUF 載入器已有本機修改，本次略過自動更新。')

'''
Retrieve the official ComfyUI release on repeat runs.
重跑時同樣取得官方 ComfyUI 發行版本。
'''
subprocess.run(['git', '-C', str(COMFY), 'fetch', '--depth', '1', 'origin', 'tag', H3_COMFY_REF], check=True)

'''
Check out the exact release without touching model checkpoints.
切換到指定發行版，不變更模型權重。
'''
subprocess.run(['git', '-C', str(COMFY), 'checkout', '--quiet', '--detach', H3_COMFY_REF], check=True)

'''
Capture code changes for managed-server restarts.
記錄程式碼更新，以便重啟 Notebook 管理的服務。
'''
H3_CORE_UPDATED = any(
    directory in old_commits and old_commits[directory] != subprocess.check_output(
        ['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True,
    ).strip()
    for directory in (COMFY, COMFY / 'custom_nodes/ComfyUI-GGUF')
)

'''
Update ClipProj to its current compatible implementation if this folder already existed.
若 ClipProj 資料夾原本就存在，同步更新到支援新版投影矩陣的程式。
'''
clipproj_dir = COMFY / 'custom_nodes/ComfyUI-ClipProj'

'''
Record the active commit to detect changes that require a service restart.
記錄更新前的版本，以判斷是否需要重新啟動服務。
'''
clipproj_before = subprocess.check_output(['git', '-C', str(clipproj_dir), 'rev-parse', 'HEAD'], text=True).strip()

'''
Download the current ClipProj code without touching the much larger model weights.
更新 ClipProj 程式碼，不重新下載大型模型權重。
'''
subprocess.run(['git', '-C', str(clipproj_dir), 'fetch', '--quiet', '--depth', '1', 'origin', 'main'], check=True)

'''
Switch the local custom node to the retrieved revision for consistent code and weights.
切換到取得的版本，確保節點程式與投影矩陣相容。
'''
subprocess.run(['git', '-C', str(clipproj_dir), 'checkout', '--quiet', '--force', 'FETCH_HEAD'], check=True)

'''
Mark a running local service for restart only when the node code changed.
只有節點程式變動時，才標記需要重啟本地服務。
'''
H3_CLIPPROJ_UPDATED = clipproj_before != subprocess.check_output(['git', '-C', str(clipproj_dir), 'rev-parse', 'HEAD'], text=True).strip()

'''
Check for the automatic 4B architecture selector and the residual projection implementation.
驗證 ClipProj 支援 4B 自動辨識與 v3.1 MLP 投影功能。
'''
clipproj_source = (clipproj_dir / 'clipproj_nodes.py').read_text(encoding='utf-8')

'''
Fail before loading models if an outdated fork lacks the required interfaces.
若外掛版本過舊且缺少必要介面，在載入模型前停止。
'''
assert 'def clip_types(' in clipproj_source and 'build_residual' in clipproj_source, 'ClipProj 缺少新版必要功能，請檢查安裝版本。'

'''
Keep Kaggle's installed PyTorch versions when resolving dependencies.
安裝依賴時保留 Kaggle 內建的 PyTorch 版本。
'''
constraint = ROOT / 'minimax_h3_torch_constraints.txt'

'''
Preserve the installed Torch version to avoid replacing CUDA libraries.
固定既有 Torch 版本，避免意外更換 CUDA 函式庫。
'''
constraint.write_text(f'torch=={torch.__version__}\ntransformers>=4.50.3,<5.0\n', encoding='utf-8')

'''
Install ComfyUI dependencies and the GGUF backend.
安裝 ComfyUI 相依套件與 GGUF 後端。
'''
for requirements in [COMFY / 'requirements.txt', COMFY / 'custom_nodes/ComfyUI-GGUF/requirements.txt']:

    '''
    Only install requirement files that exist.
    只安裝確實存在的套件清單。
    '''
    if requirements.is_file():

        '''
        Install dependencies into the running Python environment.
        將相依套件安裝到目前 Python 環境。
        '''
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(requirements)], check=True)

'''
Install remaining download and communication tools.
安裝模型下載與本地通訊工具。
'''
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), 'huggingface_hub', 'hf_xet', 'requests', 'websocket-client', 'tqdm', 'ipywidgets>=8,<9'], check=True)

'''
Confirm local installation is ready.
確認本地安裝程序完成。
'''
print('ComfyUI 與 MiniMax H3 節點安裝完成。')

'''
Import a small regex helper for the H3 GGUF architecture registration.
載入簡單的正規表示式工具，以註冊 H3 GGUF 架構。
'''
import re

'''
Locate the upstream architecture allow-list in the installed GGUF loader.
找到目前 GGUF 載入器中允許的模型架構清單。
'''
gguf_loader_path = COMFY / 'custom_nodes/ComfyUI-GGUF/loader.py'

'''
Stop if the custom node installation did not produce the expected loader file.
如果自訂節點安裝後沒有產生載入器檔案，立即顯示錯誤。
'''
assert gguf_loader_path.is_file(), f'找不到 GGUF 載入器：{gguf_loader_path}'

'''
Read the exact installed GGUF implementation instead of assuming a fixed version.
讀取實際安裝的 GGUF 原始碼，不假設版本永遠固定。
'''
gguf_source = gguf_loader_path.read_text(encoding='utf-8')

'''
Find the image-model architecture set in the installed loader.
在載入器中尋找影像模型架構允許清單。
'''
arch_match = re.search(r'IMG_ARCH_LIST\s*=\s*\{[^}]*\}', gguf_source)

'''
Avoid silently patching the wrong location if upstream changes its internals.
如果上游修改內部結構，不要在錯誤位置盲目修改。
'''
assert arch_match, '無法定位 IMG_ARCH_LIST；請檢查 ComfyUI-GGUF 版本。'

'''
Record whether the running ComfyUI process would require a reload.
記錄是否需要讓已啟動的 ComfyUI 重新載入節點。
'''
H3_RESTART_REQUIRED = H3_CLIPPROJ_UPDATED or H3_CORE_UPDATED

'''
Apply the MiniMax-H3 compatibility entry only when absent.
只有當清單尚未包含 MiniMax-H3 時才補上相容項目。
'''
if '"minimax_h3"' not in arch_match.group(0):

    '''
    Preserve the existing entries while adding the missing GGUF architecture.
    保留原本的架構列表，僅補上缺少的 GGUF 架構。
    '''
    previous_arch_list = arch_match.group(0)[:-1].rstrip()
    separator = '' if previous_arch_list.endswith(('{', ',')) else ','
    updated_arch_list = f'{previous_arch_list}{separator} "minimax_h3"}}'

    '''
    Update the installed plugin without modifying model weights.
    更新安裝的外掛相容清單，不修改模型權重。
    '''
    gguf_source = gguf_source[:arch_match.start()] + updated_arch_list + gguf_source[arch_match.end():]
    gguf_loader_path.write_text(gguf_source, encoding='utf-8')

    '''
    Mark the service as needing a restart only if it was previously running.
    僅在服務原本已啟動的情況下，標記需要重新啟動。
    '''
    H3_RESTART_REQUIRED = True
    print('已加入 MiniMax-H3 GGUF 架構相容設定。')

'''
Confirm that environment setup finished successfully.
顯示環境安裝完成，避免使用者無法確認 Cell 是否執行。
'''
print('✓ 第 1 節完成：ComfyUI、GGUF、ClipProj 與 WebSocket 工具已安裝／檢查。')


## 2. 下載量化模型（Q4／Q5／Q6／Q8，權重保留於 `/tmp`）


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import shutil

'''
Store Hugging Face metadata and downloads in /tmp to avoid filling Kaggle working.
將 Hugging Face 快取及下載資訊放到 /tmp，避免占滿 Kaggle working。
'''
os.environ['HF_HOME'] = '/tmp/minimax_h3_hf'

'''
Keep the hub cache on the same scratch filesystem as the model files.
讓 Hub 快取和模型檔案位於相同的暫存磁碟。
'''
os.environ['HF_HUB_CACHE'] = '/tmp/minimax_h3_hf/hub'

'''
Disable extra Xet chunk caching to conserve temporary disk space.
停用額外的 Xet 分塊快取，節省暫存空間。
'''
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'

'''
Download only after configuring the Hub cache environment.
先設定 Hub 快取環境，再匯入下載函式，避免路徑設定失效。
'''
from huggingface_hub import hf_hub_download

'''
Choose the generation mode from the four supported task strings. flf2v = first-and-last-frame-to-video; r2v = multi-reference-to-video (Ref2VA model). Available choices: t2v, i2v, flf2v, r2v.
從四種合法字串中選擇生成模式。 t2v = text-to-video (文字生成影片); i2v = image-to-video (首張圖片生成影片). t2v 為文生影片；i2v 為首張圖片生成影片。 flf2v 為首尾影格生成影片；r2v 為多張參考圖生成影片（需使用 Ref2VA 權重）。 可選值：t2v、i2v、flf2v、r2v。
'''
TASK = 't2v'

'''
Reject unsupported task names before downloading any weights.
下載權重前先確認任務名稱有效。
'''
assert TASK in ('t2v', 'i2v', 'flf2v', 'r2v'), f'不支援的 TASK：{TASK}'

'''
Use the distinct Ref2VA denoiser only for reference-to-video tasks.
只有 R2V 使用獨立的 Ref2VA 去噪模型，其餘模式維持 FL2VA。
'''
MODEL_VARIANT = 'ref2va_pruned' if TASK == 'r2v' else 'fl2va_pruned'

'''
Use edited task parameters when the corresponding settings cell has run.
若已執行對應的參數 Cell，則優先採用使用者修改後的量化選項。
'''
task_settings = globals().get('TASK_CONFIGS', {}).get(TASK, {})

'''
Supply first-run defaults before the task settings cells are executed.
第一次由上往下執行時，先使用各模式預設的量化格式。
'''
QUANT = task_settings.get('quant', 'Q4_K' if TASK == 'r2v' else 'Q5_0')

'''
Record the published FL2VA model sizes in GiB.
記錄 FL2VA 各量化模型公開的檔案大小（GiB）。
'''
FL2VA_QUANT_SIZES_GIB = {'Q4_K': 10.64, 'Q5_0': 12.97, 'Q6_K': 15.45, 'Q8_0': 19.97}

'''
Record the separate Ref2VA model sizes in GiB.
記錄 Ref2VA 各量化模型公開的檔案大小（GiB）。
'''
REF2VA_QUANT_SIZES_GIB = {'Q4_K': 10.60, 'Q5_0': 12.94, 'Q6_K': 15.42, 'Q8_0': 19.94}

'''
Select the size table matching the actual diffusion model family.
依照真正要使用的模型種類選擇大小對照表。
'''
QUANT_SIZES_GIB = REF2VA_QUANT_SIZES_GIB if TASK == 'r2v' else FL2VA_QUANT_SIZES_GIB

'''
Reject unsupported quantization names before any download.
下載前先排除不支援的量化名稱。
'''
assert QUANT in QUANT_SIZES_GIB, f'不支援 QUANT={QUANT!r}；請選擇 {list(QUANT_SIZES_GIB)}。'

'''
Keep both model families from the same GGUF publishing repository.
FL2VA 和 Ref2VA 都使用同一個 GGUF 發布來源。
'''
GGUF_REPO = 'robertbschuette/MiniMax-H3-GGUF'

'''
Build the actual published filename, never reusing FL2VA weights for R2V.
根據實際發布檔名選擇模型，R2V 絕不沿用 FL2VA 權重。
'''
DIT_NAME = f'minimax_h3_{MODEL_VARIANT}-{QUANT}.gguf'

'''
List all required assets and their loader-visible model directories.
列出本次需要的完整模型組合及 ComfyUI 載入資料夾。
'''
models = [
    (GGUF_REPO, DIT_NAME, 'diffusion_models'),
    ('Comfy-Org/Krea-2', 'text_encoders/qwen3vl_4b_fp8_scaled.safetensors', 'text_encoders'),
    ('NicoLab28/ClipProj-MiniMax-H3', 'mmh3-4b-ClipProj-v3.1-mlp.safetensors', 'clip_projections'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_fp16.safetensors', 'vae'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae'),
]

'''
Estimate auxiliary download sizes in GiB from published model file sizes.
根據已發布檔案大小估算共用輔助模型的下載容量（GiB）。
'''
AUX_SIZES_GIB = {
    'qwen3vl_4b_fp8_scaled.safetensors': 4.88,
    'mmh3-4b-ClipProj-v3.1-mlp.safetensors': 0.47,
    'minimax_h3_video_vae_fp16.safetensors': 4.85,
    'minimax_h3_audio_vae_fp32.safetensors': 0.56,
}

'''
Keep the valid downloaded GGUF from either the current or historical folder.
可沿用目前或舊版資料夾中已完整下載的 GGUF 權重。
'''
GGUF_CACHE_PATHS = (STORE / 'diffusion_models' / DIT_NAME, STORE / 'unet_gguf' / DIT_NAME)

'''
Include every selected model in the disk preflight, not only the denoiser.
磁碟容量預檢必須計入所有需要下載的模型，不只計算去噪權重。
'''
missing_gib = 0.0

'''
Reuse existing files only when their byte sizes agree with published estimates.
只有檔案大小符合發布資訊，才將既有權重視為完整快取。
'''
for repo, filename, folder in models:

    '''
    Read the published-size estimate for the requested asset.
    取得所選權重的公開檔案大小估計值。
    '''
    size_gib = QUANT_SIZES_GIB[QUANT] if folder == 'diffusion_models' else AUX_SIZES_GIB[Path(filename).name]

    '''
    Accept the legacy GGUF path as well as the canonical download path.
    GGUF 同時檢查舊路徑與目前的標準下載路徑。
    '''
    candidates = GGUF_CACHE_PATHS if folder == 'diffusion_models' else (STORE / folder / Path(filename).name,)

    '''
    Allow a 3% size tolerance for rounding of published file sizes.
    允許發布檔案大小四捨五入產生的 3% 誤差。
    '''
    complete = any(path.is_file() and 0.97 * size_gib <= path.stat().st_size / 1024**3 <= 1.03 * size_gib for path in candidates)

    '''
    Count only components that still need to be downloaded.
    只累計尚未完整下載的模型大小。
    '''
    if not complete:

        '''
        Include this missing model in the capacity estimate.
        將尚未下載完整的模型加入預估容量。
        '''
        missing_gib += size_gib

'''
Read the free space of the scratch filesystem used for all model files.
讀取實際存放所有模型權重的暫存磁碟可用容量。
'''
tmp_free_gib = shutil.disk_usage('/tmp').free / 1024**3

'''
Reserve additional capacity for Hub metadata and temporary download files.
額外預留 Hugging Face 中繼資料與暫存下載空間。
'''
assert tmp_free_gib >= missing_gib + 3, (
    f'/tmp 空間不足：仍需下載約 {missing_gib:.2f} GiB，'
    f'目前剩餘 {tmp_free_gib:.2f} GiB（另需預留 3 GiB）。'
)

'''
Show the complete download estimate before accessing large remote files.
開始下載大型權重前，顯示整套模型預估所需的空間。
'''
print(f'選擇 {MODEL_VARIANT}/{QUANT}｜去噪權重約 {QUANT_SIZES_GIB[QUANT]:.2f} GiB｜'
      f'尚需下載約 {missing_gib:.2f} GiB｜/tmp 可用 {tmp_free_gib:.1f} GiB')

'''
Warn when high-bit quantizations will require greater GPU/CPU offload.
高精度量化需要較多 GPU／CPU 卸載時，事先提醒使用者。
'''
if QUANT in ('Q6_K', 'Q8_0'):

    '''
    Do not imply the weight file can reside fully in a 16 GiB T4.
    不誤導使用者認為這些權重能完整常駐 16 GiB 的 T4。
    '''
    print('注意：Q6／Q8 可能需要大量 CPU Offload，速度較慢且有記憶體不足風險。')

'''
Download once and link each model to its loader-visible ComfyUI directory.
權重只下載一次，再連結到對應載入器可辨識的資料夾。
'''
for repo, filename, folder in models:

    '''
    Name each model file for a readable progress report.
    顯示正在處理的模型檔案名稱。
    '''
    print('模型：', Path(filename).name, flush=True)

    '''
    Reuse a complete GGUF in either scratch path when its size is plausible.
    GGUF 權重大小符合發布資訊時，才沿用任一暫存路徑的快取。
    '''
    source = next((path for path in GGUF_CACHE_PATHS
                   if folder == 'diffusion_models' and path.is_file()
                   and 0.97 * QUANT_SIZES_GIB[QUANT] <= path.stat().st_size / 1024**3 <= 1.03 * QUANT_SIZES_GIB[QUANT]), None)

    '''
    Download files with Hub-managed validation when no complete cache exists.
    沒有完整快取時，交由 Hugging Face Hub 管理下載及檔案驗證。
    '''
    if source is None:

        '''
        Detect a suspicious local file and force its replacement.
        如果本地權重大小異常，強制重新下載以避免沿用損壞檔案。
        '''
        local_file = STORE / folder / Path(filename).name

        '''
        Select the size estimate matching the asset family.
        依照權重種類選擇對應的檔案大小估計。
        '''
        expected_gib = QUANT_SIZES_GIB[QUANT] if folder == 'diffusion_models' else AUX_SIZES_GIB[Path(filename).name]

        '''
        Decide whether the existing file is incomplete or unexpectedly large.
        判斷現有權重是否下載不完整或大小異常。
        '''
        invalid_cache = local_file.is_file() and not (0.97 * expected_gib <= local_file.stat().st_size / 1024**3 <= 1.03 * expected_gib)

        '''
        Keep models under /tmp without copying gigabytes into Kaggle working.
        模型實體仍放在 /tmp，避免複製大型檔案到 Kaggle working。
        '''
        source = Path(hf_hub_download(repo_id=repo, filename=filename,
                                      local_dir=STORE / folder, force_download=invalid_cache))

    '''
    Confirm the final file is consistent with its published size.
    下載完成後再檢查大小，避免將異常檔案交給 ComfyUI 載入。
    '''
    expected_gib = QUANT_SIZES_GIB[QUANT] if folder == 'diffusion_models' else AUX_SIZES_GIB[Path(filename).name]

    '''
    Report a clear error if the download did not produce the expected file.
    若下載結果不符合預期，直接顯示錯誤而不是等到模型載入才失敗。
    '''
    assert 0.97 * expected_gib <= source.stat().st_size / 1024**3 <= 1.03 * expected_gib, f'模型檔案大小異常：{source}'

    '''
    Create the destination path inside ComfyUI's registered model folders.
    建立 ComfyUI 已註冊的模型資料夾內的目標路徑。
    '''
    target = COMFY / 'models' / folder / source.name
    target.parent.mkdir(parents=True, exist_ok=True)

    '''
    Keep an already-correct symbolic link to avoid unnecessary service restarts.
    已指向正確權重的符號連結直接沿用，避免不必要的服務重啟。
    '''
    if target.is_symlink() and (not target.exists() or not target.samefile(source)):

        '''
        Replace only an outdated or broken symbolic link, never a real model file.
        只替換指向錯誤位置或已失效的符號連結，不刪除實體權重。
        '''
        target.unlink()

    '''
    Add a new loader-visible link only when the correct model is not already linked.
    只有 ComfyUI 尚未連結正確模型時才建立符號連結。
    '''
    if not target.exists():

        '''
        Link the scratch-space model without copying gigabytes of weights.
        使用符號連結指向 /tmp 權重，不複製數 GB 的大型檔案。
        '''
        target.symlink_to(source.resolve())

        '''
        Notify an existing ComfyUI process to rescan every changed model dropdown.
        提醒已啟動的 ComfyUI 重新掃描所有更新過的模型選單。
        '''
        H3_RESTART_REQUIRED = True

    '''
    Refuse to silently use an unrelated real model file with the same filename.
    若已有同名的實體模型檔案，不要默默載入來源不同的權重。
    '''
    assert target.samefile(source), f'模型檔案衝突：{target} 與預期來源 {source} 不同。請先檢查舊檔。'

    '''
    Check that the loader-visible file is readable and not a broken link.
    確認載入器可見的檔案可讀取，且不是失效的符號連結。
    '''
    assert target.is_file(), f'模型連結失敗：{target}'
    print('ComfyUI 路徑：', target)

'''
Remove only the old, incorrectly placed GGUF symlink.
僅刪除放錯位置的舊 GGUF 符號連結，不動實體權重。
'''
obsolete_link = COMFY / 'models' / 'unet_gguf' / DIT_NAME
if obsolete_link.is_symlink():
    obsolete_link.unlink()

'''
Confirm all requested model paths can be opened.
確認所有所需模型檔案都可正常存取。
'''
assert all((COMFY / 'models' / folder / Path(filename).name).is_file() for _, filename, folder in models)

'''
Show when local assets are ready.
顯示本地模型資源準備完成。
'''
print(f'H3 {MODEL_VARIANT}/{QUANT}、4B ClipProj、Video VAE、Audio VAE 已就緒。')

'''
Confirm where the model files are stored after download and linking.
確認模型下載與連結完成，並顯示大型模型儲存位置。
'''
print(f'✓ 第 2 節完成：{MODEL_VARIANT}/{QUANT} 模型與相關權重已備妥（/tmp）。')

'''
Record which quantization was actually linked into the ComfyUI model directory.
記錄真正完成連結的模型量化格式，防止生成時誤載舊權重。
'''
H3_MODEL_CHOICE = (MODEL_VARIANT, QUANT)

'''
Remind users to revalidate the running loader after changing quantizations.
提醒切換量化格式後，先更新目前 ComfyUI 的模型選單。
'''
print('切換量化格式後：請重新執行第 4 節，再執行第 9 節生成。')


## 3. GPU 記憶體與 CPU Offload 設定


In [ ]:
'''
Set the approximate GPU 0 memory budget managed by ComfyUI.
設定 ComfyUI 管理的 GPU 0 近似記憶體預算（不是模型硬上限）。
'''
GPU0_MODEL_GIB = 11.0

'''
Set a GPU 1 usage warning threshold, not a hard allocator limit.
設定 GPU 1 使用量警示門檻，這不是 GPU 記憶體硬限制。
'''
GPU1_MODEL_GIB = 9.0

'''
Choose one ClipProj memory mode: dynamic, streaming, or resident. dynamic = dynamically move weights; streaming = prioritize paging; resident = keep encoder in GPU memory. Available choices: dynamic, streaming, resident.
ClipProj 載入模式可選：dynamic、streaming、resident。 dynamic 為動態搬移權重；streaming 為分段載入；resident 為盡量常駐 GPU。 可選值：dynamic、streaming、resident。
'''
TEXT_MODE = 'dynamic'

'''
Select cuda:1 (recommended encoder GPU) or cuda:0 (shares the denoiser GPU).
cuda:1 為建議的文字編碼器 GPU；cuda:0 會與影片去噪模型共用顯示記憶體。
'''
TEXT_GPU = 'cuda:1'

'''
Discover GPU capacities instead of using a fixed T4-only ceiling.
動態讀取 GPU 容量，不使用固定的 T4 專用上限。
'''
gpu0_capacity = torch.cuda.get_device_properties(0).total_memory / 1024**3

'''
Read GPU 1 capacity for meaningful warning thresholds.
取得 GPU 1 的容量，確保警示門檻合理。
'''
gpu1_capacity = torch.cuda.get_device_properties(1).total_memory / 1024**3

'''
Reject values that exceed the available GPU memory.
避免設定超過 GPU 實際容量的數值。
'''
assert 0 < GPU0_MODEL_GIB <= gpu0_capacity, f'GPU0_MODEL_GIB 必須介於 0 和 {gpu0_capacity:.2f} GiB 之間。'

'''
Reject impossible GPU 1 monitoring thresholds.
排除無效的 GPU 1 使用量警示門檻。
'''
assert 0 < GPU1_MODEL_GIB <= gpu1_capacity, f'GPU1_MODEL_GIB 必須介於 0 和 {gpu1_capacity:.2f} GiB 之間。'

'''
Use only documented ClipProj offload modes.
僅使用 ClipProj 正式提供的卸載模式。
'''
assert TEXT_MODE in ('dynamic', 'streaming', 'resident'), 'TEXT_MODE 設定無效。'

'''
Reserve the difference for GPU 0 so ComfyUI targets the chosen overall budget.
將 GPU 0 的容量差額保留，讓 ComfyUI 以指定的整體預算運作。
'''
GPU0_RESERVE_GIB = round(gpu0_capacity - GPU0_MODEL_GIB, 3)

'''
Choose classic node caching for unchanged model loaders; use none if RAM is insufficient. Available choices: classic, none.
相同模型載入節點使用 classic 快取；RAM 不足時可改成 none。 可選值：classic、none。
'''
NODE_CACHE_MODE = 'classic'

'''
Reject invalid ComfyUI node-cache modes.
避免輸入 ComfyUI 不支援的節點快取模式。
'''
assert NODE_CACHE_MODE in ('classic', 'none')

'''
Store the desired service configuration for change detection.
儲存預期的服務參數，用來判斷是否需要重新啟動 ComfyUI。
'''
H3_DESIRED_CONFIG = (GPU0_RESERVE_GIB, NODE_CACHE_MODE)

'''
Explain which settings are enforced and which are advisory.
說明哪些設定會套用、哪些僅供監控。
'''
print(f'✓ 第 3 節完成：GPU 0 管理預算約 {GPU0_MODEL_GIB:.2f} GiB；保留 {GPU0_RESERVE_GIB:.2f} GB。')

'''
Clarify that per-model residency is not a hard limit.
明確提醒：這不是單一模型的精確權重常駐限制。
'''
print('GPU 0：ComfyUI Dynamic VRAM / CPU Offload（權重、運算暫存及其它元件共用預算）。')

'''
Explain that ClipProj supports offload modes rather than an independent hard cap.
說明 ClipProj 支援的是卸載模式，無法單獨限制 GPU 1 精確用量。
'''
print(f'GPU 1：ClipProj {TEXT_MODE}，{GPU1_MODEL_GIB:.2f} GiB 為監控警示值，並非硬上限。')


## 4. 啟動本地 ComfyUI 與相依性檢查


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import requests
import time

'''
Keep console logs outside the model cache.
將服務紀錄與模型快取分開存放。
'''
LOG = ROOT / 'minimax_h3_comfyui.log'

'''
Define the private local server address.
定義僅供 Notebook 使用的本地服務網址。
'''
BASE = f'http://127.0.0.1:{PORT}'

'''
Read server readiness without requiring a public tunnel.
不開放公開連線即可檢查伺服器狀態。
'''
def ready():

    '''
    Handle an unavailable local service safely.
    安全處理尚未啟動的服務。
    '''
    try:

        '''
        Check the converter or ComfyUI health endpoint.
        透過 ComfyUI 系統狀態檢查服務。
        '''
        return requests.get(f'{BASE}/system_stats', timeout=3).ok

    except requests.RequestException:

        '''
        Return false when the service is unreachable.
        服務無法連線時回傳 False。
        '''

        '''
        Report unavailable state to the caller.
        將未就緒狀態回傳給呼叫端。
        '''
        return False

'''
Read the ComfyUI loader's actual model dropdown values.
讀取 ComfyUI GGUF Loader 真正提供的模型選項。
'''
def listed_ggufs():

    '''
    Retrieve the registered GGUF model names from the local service.
    從本地服務取得已註冊的 GGUF 模型檔名。
    '''
    payload = requests.get(f'{BASE}/object_info/UnetLoaderGGUF', timeout=30).json()
    info = payload.get('UnetLoaderGGUF', {})

    '''
    Read the registered input field without assuming it is a dropdown list.
    讀取已註冊欄位，不假設其必定有下拉選單資料。
    '''
    spec = info.get('input', {}).get('required', {}).get('unet_name')

    '''
    Return an empty list when model metadata is incomplete or unavailable.
    模型選單資料不完整時回傳空清單，避免 NoneType 錯誤。
    '''
    return spec[0] if isinstance(spec, list) and spec and isinstance(spec[0], list) else []

'''
Restart the notebook-owned service only when the model scan or memory budget changed.
模型選單過期或記憶體預算改變時，才重啟此 Notebook 管理的服務。
'''
need_restart = False

'''
Inspect a running service before deciding whether to reuse it.
先檢查目前服務狀態，再決定是否沿用。
'''
if ready():

    '''
    Detect node updates, missing models and changed startup memory settings.
    偵測節點更新、模型未註冊及啟動記憶體設定改變。
    '''
    need_restart = (
        globals().get('H3_RESTART_REQUIRED', False)
        or DIT_NAME not in listed_ggufs()
        or globals().get('H3_ACTIVE_MODEL') != (MODEL_VARIANT, QUANT)
        or globals().get('H3_ACTIVE_CONFIG') != H3_DESIRED_CONFIG
    )

'''
Never terminate services launched by another notebook or process.
不任意終止其它 Notebook 或程序啟動的 ComfyUI。
'''
if need_restart:

    '''
    Only restart a service for which this kernel holds the process handle.
    只有目前 Kernel 持有程序控制權時才能自動重新啟動。
    '''
    assert 'H3_PROC' in globals() and H3_PROC.poll() is None, (
        '服務需要更新記憶體設定，但不是本 Notebook 可管理的程序。'
        '請先停止連接埠 8193 的舊服務，再執行本 Cell。'
    )

    '''
    Ask the old service to exit before applying the new memory budget.
    讓舊服務結束，再套用新的記憶體預算。
    '''
    H3_PROC.terminate()

    '''
    Bound the shutdown time and force termination only when necessary.
    限制關閉等待時間，必要時才強制結束程序。
    '''
    try:

        '''
        Wait briefly for a graceful service shutdown.
        等待服務正常結束。
        '''
        H3_PROC.wait(timeout=20)

    except subprocess.TimeoutExpired:

        '''
        Handle a service that does not respond to graceful shutdown.
        處理無法正常關閉的服務。
        '''

        '''
        Terminate the notebook-owned stalled process.
        強制終止本 Notebook 所管理且未回應的程序。
        '''
        H3_PROC.kill()

        '''
        Reap the terminated process before restarting.
        等待程序確實結束，避免連接埠衝突。
        '''
        H3_PROC.wait(timeout=10)

    '''
    Confirm the memory configuration will be applied by a fresh process.
    提醒接下來會啟動套用新記憶體設定的程序。
    '''
    print('重新啟動 ComfyUI：模型索引、節點版本或 GPU 0 記憶體預算有變更。')

'''
Launch ComfyUI only when there is no usable local process.
沒有可用服務時才啟動 ComfyUI。
'''
if not ready():

    '''
    Use ComfyUI's documented reserve-vram flag for an approximate GPU 0 budget.
    使用 ComfyUI 官方 reserve-vram 參數調整 GPU 0 的近似整體預算。
    '''
    command = [
        sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(PORT),
        '--enable-dynamic-vram', '--reserve-vram', str(GPU0_RESERVE_GIB),
        '--cache-classic' if NODE_CACHE_MODE == 'classic' else '--cache-none',
        '--disable-pinned-memory',
    ]

    '''
    Keep the local service log available for debugging without blocking Kaggle.
    保留服務 Log 方便除錯，且不阻塞 Notebook。
    '''
    with LOG.open('w', encoding='utf-8') as stream:

        '''
        Start the persistent local inference process.
        啟動可供多次推論的本地服務程序。
        '''
        H3_PROC = subprocess.Popen(command, cwd=COMFY, stdout=stream, stderr=subprocess.STDOUT)

    '''
    Wait for either service readiness or a startup crash.
    等待服務就緒，或偵測啟動階段的錯誤。
    '''
    for attempt in range(180):

        '''
        Report the original service error rather than waiting indefinitely.
        發生啟動錯誤時直接輸出 Log，避免無限等待。
        '''
        if H3_PROC.poll() is not None:

            '''
            Provide the most recent server diagnostics.
            提供最近的服務錯誤紀錄。
            '''
            raise RuntimeError(LOG.read_text(errors='replace')[-4000:])

        '''
        Stop waiting once the server responds to health checks.
        服務通過健康檢查後立即繼續。
        '''
        if ready():

            '''
            Exit startup polling after success.
            啟動成功後結束輪詢。
            '''
            break

        '''
        Prevent excessive requests while the service initializes.
        避免服務初始化期間產生過多查詢。
        '''
        time.sleep(2)

    '''
    Fail with diagnostics when the service never becomes healthy.
    服務未成功啟動時顯示診斷訊息。
    '''
    assert ready(), LOG.read_text(errors='replace')[-4000:]

'''
Remember the actual startup configuration to detect later budget edits.
記錄已套用的啟動參數，以便之後變更記憶體預算時自動重啟。
'''
H3_ACTIVE_CONFIG = H3_DESIRED_CONFIG

'''
Get the exact input schemas registered by the running ComfyUI process.
從目前執行中的 ComfyUI 讀取真正註冊的節點輸入規格。
'''
NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()

'''
List all node types required for text, image, audio and video output.
列出文生、圖生、音訊解碼及影片輸出所需要的全部節點。
'''
REQUIRED_NODES = (
    'ClipProjLoader', 'UnetLoaderGGUF', 'MiniMaxH3SigmaShift',
    'VAELoader', 'MiniMaxH3ImageToVideo', 'BasicGuider', 'KSamplerSelect',
    'BasicScheduler', 'RandomNoise', 'SamplerCustomAdvanced', 'VAEDecode',
    'VAEDecodeAudio', 'CreateVideo', 'SaveVideo', 'LoadImage',
    'MiniMaxH3ReferenceToVideo',
)

'''
Collect missing nodes before an expensive GPU load or sampling step.
在大型模型載入與採樣之前，一次列出全部缺少的節點。
'''
missing = [name for name in REQUIRED_NODES if name not in NODE_INFO]

'''
Show a clear explanation for a failed or incomplete ComfyUI installation.
若 ComfyUI 安裝不完整，直接指出缺少的節點。
'''
assert not missing, f'ComfyUI 缺少節點：{missing}。請檢查 {LOG} 並重新執行安裝與啟動。'

'''
Read a dropdown's allowed values from ComfyUI's live schema.
從 ComfyUI 正在使用的規格讀取下拉選單允許值。
'''
def choices(node_name, input_name):

    '''
    Support both required and optional node inputs.
    同時支援必填與選填的節點輸入。
    '''
    groups = NODE_INFO[node_name].get('input', {})

    '''
    Locate the requested input in the registered schema.
    在已註冊規格中找到指定的輸入欄位。
    '''
    for group in ('required', 'optional'):

        '''
        Read the field's allowed choice list if the input exists.
        若輸入欄位存在，就讀取其允許的選項。
        '''
        spec = groups.get(group, {}).get(input_name)

        '''
        Return None for numeric/string inputs that are not dropdown lists.
        如果不是下拉式選單，就不視為有限選項。
        '''
        if spec is not None:

            '''
            Read fixed dropdown values only when the node exposes an actual choice list.
            只有節點確實提供選項清單時，才讀取固定的下拉選項。
            '''
            if isinstance(spec, (list, tuple)) and spec:

                '''
                ComfyUI normally represents dropdowns as [[choice1, choice2], metadata].
                ComfyUI 通常以 [[選項1, 選項2], 相關設定] 表示下拉選單。
                '''
                if isinstance(spec[0], list):

                    '''
                    Return an explicit enumeration for early validation.
                    回傳明確列舉的選項，供提前檢查。
                    '''
                    return spec[0]

                '''
                Support nodes exposing their enum in a metadata dictionary.
                相容以設定字典提供下拉選項的節點。
                '''
                if len(spec) > 1 and isinstance(spec[1], dict):

                    '''
                    Try recognized metadata keys without inventing choices.
                    僅讀取已知的選項欄位，不自行猜測可用值。
                    '''
                    for key in ('choices', 'options'):

                        '''
                        Check whether this metadata entry contains a list of choices.
                        確認設定欄位是否真的包含選項清單。
                        '''
                        if isinstance(spec[1].get(key), list):

                            '''
                            Use this declared choice list.
                            使用該節點宣告的選項清單。
                            '''
                            return spec[1][key]

            '''
            A scalar input has no enumerated choices; defer to the ComfyUI API.
            一般字串等輸入沒有固定選單，交由 ComfyUI API 驗證。
            '''
            return None

    '''
    Make a missing input fail clearly rather than waiting for prompt validation.
    若輸入欄位不存在，直接報錯，不等到提交 workflow 才發現。
    '''
    raise RuntimeError(f'節點 {node_name} 沒有預期的輸入 {input_name}；可能是節點版本不相容.')

'''
Validate dropdown values without crashing if a node exposes only a scalar input.
驗證下拉選項；如果節點只提供一般輸入型別，就不要因此發生例外。
'''
def check_choice(node_name, input_name, value, fallback=None):

    '''
    Read the node's current explicit choice list if present.
    有明確選項清單時優先讀取目前節點的清單。
    '''
    allowed = choices(node_name, input_name)

    '''
    Use the standard ComfyUI sampler schema when the specialized node has no enum.
    專用節點沒有列舉選項時，改查 ComfyUI 標準採樣器的規格。
    '''
    if allowed is None and fallback and fallback[0] in NODE_INFO:

        '''
        Obtain the comparable fallback list.
        取得備援節點提供的相同選項。
        '''
        allowed = choices(*fallback)

    '''
    Let ComfyUI validate scalar/non-enumerated inputs when submitting the workflow.
    如果沒有固定選項，就由 ComfyUI 在提交工作流程時驗證。
    '''
    if allowed is None:

        '''
        Warn rather than comparing a string against None.
        顯示提示，不再把字串拿來與 None 比對。
        '''
        print(f'預檢提示：{node_name}.{input_name} 沒有選項清單，將交由 ComfyUI 驗證。')
    else:

        '''
        Report an unsupported choice before starting expensive model inference.
        無效選項會在進行耗時模型推論前先被攔下。
        '''
        assert value in allowed, f'{node_name}.{input_name} 不支援 {value!r}；可用選項：{allowed}'

'''
Validate the denoiser exactly as the GGUF dropdown sees it.
用 GGUF Loader 實際提供的檔案選單確認生成模型。
'''
assert DIT_NAME in (choices('UnetLoaderGGUF', 'unet_name') or []), f'GGUF Loader 沒有辨識 {DIT_NAME}。'

'''
Confirm that the 4B text encoder is accessible from the local ComfyUI service.
確認 4B 文字編碼器已被本地 ComfyUI 服務辨識。
'''
assert 'qwen3vl_4b_fp8_scaled.safetensors' in (choices('ClipProjLoader', 'clip_name') or []), 'ClipProj Loader 沒有辨識 4B Encoder。'

'''
Verify that the correct projection matrix is registered, not merely downloaded.
確認正確的投影矩陣已註冊，而不只是檔案下載成功。
'''
assert 'mmh3-4b-ClipProj-v3.1-mlp.safetensors' in (choices('ClipProjLoader', 'projection') or []), 'ClipProj Loader 沒有辨識 v3.1 投影矩陣。'

'''
Verify the fixed automatic architecture detection option is supported.
確認新版 ClipProj 支援 auto 自動辨識文字編碼器架構。
'''
assert 'auto' in (choices('ClipProjLoader', 'type') or []), 'ClipProj Loader 不支援 type=auto，請更新節點。'

'''
The automatic detector must also be able to instantiate the 4B Krea2 architecture.
自動偵測模式也必須有 krea2（Qwen3-VL 4B）載入實作。
'''
assert 'krea2' in (choices('ClipProjLoader', 'type') or []), '目前 ComfyUI 尚未提供 krea2／Qwen3-VL 4B 編碼器載入支援，請更新 ComfyUI。'

'''
Verify the supported video and audio VAE filenames.
確認影片與音訊 VAE 都出現在實際模型選單中。
'''
for vae_name in ('minimax_h3_video_vae_fp16.safetensors', 'minimax_h3_audio_vae_fp32.safetensors'):

    '''
    Stop if either necessary VAE is not visible to the running service.
    若任一必要 VAE 未被目前服務辨識，立即停止。
    '''
    assert vae_name in (choices('VAELoader', 'vae_name') or []), f'ComfyUI 沒有辨識 VAE：{vae_name}'

'''
Keep the active schemas available for checking generation parameters later.
保留目前節點規格，供生成前再次確認參數使用。
'''
print('依賴檢查通過：FL2VA、Ref2VA、GGUF、4B Encoder、ClipProj v3.1、Video/Audio VAE。')

'''
Confirm that the current ComfyUI instance has validated the selected quantization.
確認目前 ComfyUI 服務已驗證本次選取的量化格式。
'''
H3_ACTIVE_QUANT = QUANT

'''
Remember the model family as well as its quantization.
同時記錄模型種類及量化格式，避免混用 FL2VA／Ref2VA。
'''
H3_ACTIVE_MODEL = (MODEL_VARIANT, QUANT)

'''
Clear the previous restart flag once the live loader recognizes the model.
服務已辨識模型後，清除上一次的重新啟動標記。
'''
H3_RESTART_REQUIRED = False

'''
Confirm the local service is ready to accept successive prompts.
確認本地服務可持續接收新的提示詞。
'''
print(f'本地 MiniMax H3 {MODEL_VARIANT}/{QUANT} 已就緒（GGUF 模型選單驗證通過）。')

'''
Confirm that all service and model checks finished without an exception.
所有服務及模型檢查通過後，明確顯示本格完成。
'''
print(f'✓ 第 4 節完成：ComfyUI 服務與模型預檢完成；GPU 0 保留 {GPU0_RESERVE_GIB:.2f} GB。')


## 5. T2V：Prompt 與參數設定


In [ ]:
'''
Describe the text-to-video scene and synchronized sound.
設定文生影片的場景與同步音效描述。
'''
T2V_PROMPT = '''
A cinematic close-up of a red panda in a misty forest. The red panda slowly turns its head.
Natural fur details, gentle camera movement, soft morning light. Sound: birds chirping and light wind.
'''.strip()

'''
Configure the output dimensions for text-to-video.
設定文生影片的畫面解析度。
'''
T2V_WIDTH, T2V_HEIGHT = 384, 224

'''
Configure the target duration in seconds.
設定文生影片的目標秒數。
'''
T2V_SECONDS = 10

'''
Choose sampling iterations for the original model.
設定原始模型的採樣步數。
'''
T2V_STEPS = 20

'''
Set the audio sigma shift for this generation mode (ComfyUI default: 3.0).
設定這個生成模式的音訊 Sigma Shift（ComfyUI 預設：3.0）。
'''
T2V_AUDIO_SHIFT = 6.0

'''
Choose the GGUF quantization independently for this generation mode. Valid FL2VA model quants: Q4_K (~10.64 GiB), Q5_0 (~12.97 GiB), Q6_K (~15.45 GiB), Q8_0 (~19.97 GiB). Higher precision uses more RAM/VRAM and can require slower CPU offload on T4. Available choices: Q4_K, Q5_0, Q6_K, Q8_0.
獨立指定此生成模式的 GGUF 量化格式。 FL2VA 可選：Q4_K（約 10.64 GiB）、Q5_0（12.97 GiB）、Q6_K（15.45 GiB）、Q8_0（19.97 GiB）。 精度越高，RAM／VRAM 需求越大；T4 可能需要更慢的 CPU offload。 可選值：Q4_K、Q5_0、Q6_K、Q8_0。
'''
T2V_QUANT = 'Q5_0'

'''
Register the quantization below in TASK_CONFIGS for the download cell. QUANT = T2V_QUANT Set this task’s seed; use -1 to randomize each generation.
下方 TASK_CONFIGS 會儲存量化選項，供下載 Cell 讀取。 設定此任務的種子；-1 代表每次生成都重新抽取。
'''
T2V_SEED = 42

'''
Set the T2V output filename without overwriting earlier runs.
設定文生影片輸出檔名；若同名會自動編號。
'''
T2V_OUTPUT = 'minimax_h3_t2v.mp4'

'''
Preserve configurations for every task independently.
分開保存各項任務的設定，避免不同 Prompt 互相覆寫。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register this task's complete generation settings.
登記文生影片的完整生成設定。
'''
TASK_CONFIGS['t2v'] = dict(prompt=T2V_PROMPT, width=T2V_WIDTH, height=T2V_HEIGHT,
                           seconds=T2V_SECONDS, steps=T2V_STEPS, seed=T2V_SEED,
                           audio_shift=T2V_AUDIO_SHIFT, quant=T2V_QUANT,
                           output_name=T2V_OUTPUT, first_image='', last_image='')

'''
Provide explicit feedback after configuring the task.
完成任務設定後顯示明確提示。
'''
print(f'✓ 第 5 節完成：T2V | {T2V_WIDTH}×{T2V_HEIGHT} | {T2V_SECONDS} 秒 | {T2V_STEPS} steps | {T2V_QUANT} | Audio Shift {T2V_AUDIO_SHIFT} | {T2V_OUTPUT}')


## 6. I2V：Prompt、起始圖片與參數設定


In [ ]:
'''
Describe how the uploaded first frame should animate.
描述上傳的第一張圖片要如何轉化為動態影片。
'''
I2V_PROMPT = '''
Animate the subject in the reference image with subtle natural motion and smooth cinematic camera movement.
Keep the person's or object's identity, clothing and environment consistent. Include realistic ambient sound.
'''.strip()

'''
Select the initial image from the Kaggle filesystem.
指定 Kaggle 本地的起始圖片路徑。
'''
I2V_FIRST_IMAGE = '/kaggle/working/first_frame.png'

'''
Configure output dimensions for image-to-video.
設定圖生影片的解析度。
'''
I2V_WIDTH, I2V_HEIGHT = 384, 224

'''
Configure output duration for image-to-video.
設定圖生影片的目標秒數。
'''
I2V_SECONDS = 10

'''
Choose the image-conditioned sampling steps.
設定圖生影片的採樣步數。
'''
I2V_STEPS = 20

'''
Set the audio sigma shift for this generation mode (ComfyUI default: 3.0).
設定這個生成模式的音訊 Sigma Shift（ComfyUI 預設：3.0）。
'''
I2V_AUDIO_SHIFT = 6.0

'''
Choose the GGUF quantization independently for this generation mode. Valid FL2VA model quants: Q4_K (~10.64 GiB), Q5_0 (~12.97 GiB), Q6_K (~15.45 GiB), Q8_0 (~19.97 GiB). Higher precision uses more RAM/VRAM and can require slower CPU offload on T4. Available choices: Q4_K, Q5_0, Q6_K, Q8_0.
獨立指定此生成模式的 GGUF 量化格式。 FL2VA 可選：Q4_K（約 10.64 GiB）、Q5_0（12.97 GiB）、Q6_K（15.45 GiB）、Q8_0（19.97 GiB）。 精度越高，RAM／VRAM 需求越大；T4 可能需要更慢的 CPU offload。 可選值：Q4_K、Q5_0、Q6_K、Q8_0。
'''
I2V_QUANT = 'Q5_0'

'''
Register the quantization below in TASK_CONFIGS for the download cell. QUANT = I2V_QUANT Set this task’s seed; use -1 to randomize each generation.
下方 TASK_CONFIGS 會儲存量化選項，供下載 Cell 讀取。 設定此任務的種子；-1 代表每次生成都重新抽取。
'''
I2V_SEED = 42

'''
Set an independent image-to-video output filename.
指定圖生影片自己的輸出檔名。
'''
I2V_OUTPUT = 'minimax_h3_i2v.mp4'

'''
Reuse the task configuration registry without resetting other tasks.
沿用任務設定清單，不重置其他任務。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register the first-frame conditioning input and task parameters.
登記起始影格及圖生影片參數。
'''
TASK_CONFIGS['i2v'] = dict(prompt=I2V_PROMPT, width=I2V_WIDTH, height=I2V_HEIGHT,
                           seconds=I2V_SECONDS, steps=I2V_STEPS, seed=I2V_SEED,
                           audio_shift=I2V_AUDIO_SHIFT, quant=I2V_QUANT,
                           output_name=I2V_OUTPUT, first_image=I2V_FIRST_IMAGE, last_image='')

'''
Confirm the configuration cell has run without attempting to open the image yet.
顯示設定完成；實際圖片檔案會在生成時再驗證。
'''
print(f'✓ 第 6 節完成：I2V | {I2V_WIDTH}×{I2V_HEIGHT} | {I2V_SECONDS} 秒 | {I2V_QUANT} | Audio Shift {I2V_AUDIO_SHIFT} | 圖片：{I2V_FIRST_IMAGE}')


## 7. FLF2V：Prompt、首尾圖片與參數設定


In [ ]:
'''
Describe a video that connects its first and last reference frames.
描述從第一張圖片過渡到最後一張圖片的影片內容。
'''
FLF2V_PROMPT = '''
Create a smooth cinematic transition from the first reference frame to the last reference frame.
Maintain consistent subject details and lighting, with natural intermediate motion and synchronized ambient sound.
'''.strip()

'''
Choose the first image in the start/end-frame sequence.
設定首尾影格模式的起始圖片。
'''
FLF2V_FIRST_IMAGE = '/kaggle/working/first_frame.png'

'''
Choose the last image in the start/end-frame sequence.
設定首尾影格模式的結束圖片。
'''
FLF2V_LAST_IMAGE = '/kaggle/working/last_frame.png'

'''
Set output dimensions for first-and-last-frame generation.
設定首尾影格模式的輸出解析度。
'''
FLF2V_WIDTH, FLF2V_HEIGHT = 384, 224

'''
Set the duration for the start/end-frame transition.
設定首尾影格過渡的目標秒數。
'''
FLF2V_SECONDS = 10

'''
Choose the diffusion sampling steps.
設定首尾影格模式的採樣步數。
'''
FLF2V_STEPS = 20

'''
Set the audio sigma shift for this generation mode (ComfyUI default: 3.0).
設定這個生成模式的音訊 Sigma Shift（ComfyUI 預設：3.0）。
'''
FLF2V_AUDIO_SHIFT = 6.0

'''
Choose the GGUF quantization independently for this generation mode. Valid FL2VA model quants: Q4_K (~10.64 GiB), Q5_0 (~12.97 GiB), Q6_K (~15.45 GiB), Q8_0 (~19.97 GiB). Higher precision uses more RAM/VRAM and can require slower CPU offload on T4. Available choices: Q4_K, Q5_0, Q6_K, Q8_0.
獨立指定此生成模式的 GGUF 量化格式。 FL2VA 可選：Q4_K（約 10.64 GiB）、Q5_0（12.97 GiB）、Q6_K（15.45 GiB）、Q8_0（19.97 GiB）。 精度越高，RAM／VRAM 需求越大；T4 可能需要更慢的 CPU offload。 可選值：Q4_K、Q5_0、Q6_K、Q8_0。
'''
FLF2V_QUANT = 'Q5_0'

'''
Register the quantization below in TASK_CONFIGS for the download cell. QUANT = FLF2V_QUANT Set this task’s seed; use -1 to randomize each generation.
下方 TASK_CONFIGS 會儲存量化選項，供下載 Cell 讀取。 設定此任務的種子；-1 代表每次生成都重新抽取。
'''
FLF2V_SEED = 42

'''
Save output under a dedicated filename.
為首尾影格模式指定獨立輸出檔名。
'''
FLF2V_OUTPUT = 'minimax_h3_flf2v.mp4'

'''
Keep previously configured modes unchanged.
保留先前其它生成模式的設定。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register both conditioning images and all parameters.
登記兩張參考圖片與全部生成參數。
'''
TASK_CONFIGS['flf2v'] = dict(prompt=FLF2V_PROMPT, width=FLF2V_WIDTH, height=FLF2V_HEIGHT,
                             seconds=FLF2V_SECONDS, steps=FLF2V_STEPS, seed=FLF2V_SEED,
                             audio_shift=FLF2V_AUDIO_SHIFT, quant=FLF2V_QUANT,
                             output_name=FLF2V_OUTPUT, first_image=FLF2V_FIRST_IMAGE,
                             last_image=FLF2V_LAST_IMAGE)

'''
Confirm the configuration has been stored without running generation.
確認首尾影格設定已儲存，這一格不會執行生成。
'''
print(f'✓ 第 7 節完成：FLF2V | {FLF2V_WIDTH}×{FLF2V_HEIGHT} | {FLF2V_SECONDS} 秒 | {FLF2V_QUANT} | Audio Shift {FLF2V_AUDIO_SHIFT} | 參考圖 2 張')


## 8. R2V：Prompt、Reference Images 與參數設定


In [ ]:
'''
Describe how multiple reference images define the subject and environment.
使用多張參考圖片指定人物外觀、服裝及環境。
'''
R2V_PROMPT = '''
<Picture 1> defines the subject's face, hairstyle, and identity.
Additional pictures, when provided, help specify the outfit and environment.
Create a cinematic video of the subject walking naturally through the scene.
Preserve identity, clothing, and scene details from the reference pictures.
Natural motion, soft daylight, realistic ambient sounds.
'''.strip()

'''
Provide one to nine local reference images in the same order as the prompt tags.
依照 Prompt 的圖片標籤順序，設定一至九張 Kaggle 本地參考圖片。
'''
R2V_REF_IMAGES = []

'''
Choose reference sizing: match or max. match = downscale to output-video pixel area; max = preserve up to a 2048 px short edge. Available choices: match, max.
參考圖片尺寸模式可選 match、max。 match 會縮至影片像素面積；max 可保留最多約 2048 px 短邊細節，但耗時及 VRAM 較高。 可選值：match、max。
'''
R2V_REF_IMAGE_SIZE = 'match'

'''
Select the R2V-specific ClipProj mode: resident, dynamic, or streaming. resident = prioritize vision-encoder stability; dynamic = lower residency; streaming = page weights. Available choices: resident, dynamic, streaming.
R2V 專用 ClipProj 模式可選 resident、dynamic、streaming。 resident 優先保持影像編碼器常駐；dynamic 動態卸載；streaming 使用分段載入。 可選值：resident、dynamic、streaming。
'''
R2V_TEXT_MODE = 'resident'

'''
Configure the output canvas without an arbitrary aspect-ratio restriction.
設定輸出解析度，不另外限制畫面比例。
'''
R2V_WIDTH, R2V_HEIGHT = 384, 224

'''
Configure the target video duration in seconds.
設定影片目標秒數。
'''
R2V_SECONDS = 10

'''
Set the diffusion sampling steps for the normal Ref2VA model.
設定正常版 Ref2VA 模型的採樣步數。
'''
R2V_STEPS = 20

'''
Configure the audio sigma shift independently of the other tasks.
獨立設定 R2V 音訊 Sigma Shift。
'''
R2V_AUDIO_SHIFT = 3.0

'''
Choose an available Ref2VA quant: Q4_K (10.60 GiB), Q5_0 (12.94 GiB), Q6_K (15.42 GiB), Q8_0 (19.94 GiB). Larger quants may not fit in T4 VRAM and need substantial offload. Available choices: Q4_K, Q5_0, Q6_K, Q8_0.
Ref2VA 量化選項：Q4_K（10.60 GiB）、Q5_0（12.94 GiB）、Q6_K（15.42 GiB）、Q8_0（19.94 GiB）。 高精度權重可能無法完整放入 T4，必須使用較多卸載空間。 可選值：Q4_K、Q5_0、Q6_K、Q8_0。
'''
R2V_QUANT = 'Q4_K'

'''
Set this task’s seed; use -1 to randomize each generation.
設定此任務的種子；-1 代表每次生成都重新抽取。
'''
R2V_SEED = 42

'''
Set a separate output name so R2V never overwrites earlier modes.
設定獨立輸出檔名，避免覆蓋其它模式的影片。
'''
R2V_OUTPUT = 'minimax_h3_r2v.mp4'

'''
Reuse the shared task configuration registry.
沿用各模式共用的任務設定表。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Save the reference media and settings without downloading extra models yet.
儲存參考圖與參數，不在此 Cell 下載額外權重。
'''
TASK_CONFIGS['r2v'] = dict(prompt=R2V_PROMPT, width=R2V_WIDTH, height=R2V_HEIGHT,
                            seconds=R2V_SECONDS, steps=R2V_STEPS, seed=R2V_SEED,
                            audio_shift=R2V_AUDIO_SHIFT, quant=R2V_QUANT,
                            output_name=R2V_OUTPUT, first_image='', last_image='',
                            ref_images=R2V_REF_IMAGES, ref_image_size=R2V_REF_IMAGE_SIZE,
                            text_mode=R2V_TEXT_MODE)

'''
Explain how to activate this task using the shared downloader selector.
提示如何透過共用下載 Cell 選擇 R2V 任務。
'''
print(f'✓ 第 8 節完成：R2V | {R2V_WIDTH}×{R2V_HEIGHT} | {R2V_SECONDS} 秒 | '
      f'{R2V_QUANT} | {len(R2V_REF_IMAGES)} 張圖片；請在第 2 節設定 TASK = "r2v"。')

'''
Provide a multi-file upload control for the supported reference-image task.
為支援參考圖片的模式建立多選上傳介面。
'''
import ipywidgets as widgets
from pathlib import Path
from PIL import Image, ImageOps
from io import BytesIO
from uuid import uuid4
from html import escape
from IPython.display import display, clear_output
import shutil

'''
Keep the selected image batch independent of other generation modes.
參考圖資料夾獨立保存，避免干擾其他生成模式。
'''
_REF_ROOT = Path('/kaggle/working/minimax_h3_r2v_references')
_REF_ROOT.mkdir(parents=True, exist_ok=True)
_REF_MAX = 9
_REF_FILES = sorted(p for p in _REF_ROOT.iterdir() if p.is_file() and p.suffix.lower() in ('.png','.jpg','.jpeg','.webp'))
if len(_REF_FILES) > _REF_MAX or any(p.stem != f'{i:03d}' for i,p in enumerate(_REF_FILES,1)):
    raise RuntimeError('參考圖編號不連續；請清理舊資料夾再執行上傳 Cell。')

'''
Synchronize the numbered paths with the generation configuration.
將最新圖片清單同步到生成工作所使用的設定。
'''
def _ref_sync(paths):
    global R2V_REF_IMAGES
    R2V_REF_IMAGES = [str(path) for path in paths]
    TASK_CONFIGS['r2v']['ref_images'] = list(R2V_REF_IMAGES)

'''
Replace all reference images only after the new upload is fully validated.
新圖片全部通過驗證後才取代上一批；選得更少時也會清除多餘舊檔。
'''
def _ref_update(change):
    items = change['new']
    if not items:
        return
    if isinstance(items, dict):
        items = [dict(name=name, **item) for name, item in items.items()]
    stage = _REF_ROOT.with_name(_REF_ROOT.name + '_stage_' + uuid4().hex)
    stage.mkdir(parents=True)
    backup = _REF_ROOT.with_name(_REF_ROOT.name + '_backup_' + uuid4().hex)
    try:
        if not 1 <= len(items) <= _REF_MAX:
            raise ValueError(f'一次請選擇 1～{_REF_MAX} 張圖片。')
        for index, item in enumerate(items, 1):
            filename = Path(item['name']).name
            suffix = Path(filename).suffix.lower()
            if suffix not in ('.png','.jpg','.jpeg','.webp'):
                raise ValueError(f'不支援的圖片格式：{filename}')
            blob = bytes(item['content'])
            if len(blob) > 20 * 1024 * 1024:
                raise ValueError(f'單張圖片超過 20 MiB：{filename}')
            with Image.open(BytesIO(blob)) as photo:
                photo.verify()
            (stage / f'{index:03d}{suffix}').write_bytes(blob)
        _REF_ROOT.rename(backup)
        try:
            stage.rename(_REF_ROOT)
        except Exception:
            backup.rename(_REF_ROOT)
            raise
        shutil.rmtree(backup)
        current = sorted(_REF_ROOT.iterdir())
        _ref_sync(current)
        print('最新參考圖：', ', '.join(f'{p.name} = <Picture {i}>' for i,p in enumerate(current,1)))
    except Exception as error:
        shutil.rmtree(stage, ignore_errors=True)
        print('上傳未生效，保留上一批參考圖：', error)
    finally:
        _REF_PANEL.children = (_ref_new(),)

'''
Reset the browser picker so selecting the same filenames works again.
每次上傳後重建選檔元件，確保可以重新選相同檔名。
'''
def _ref_new():
    picker = widgets.FileUpload(accept='image/*', multiple=True, description='多選參考圖片')
    picker.observe(_ref_update, names='value')
    return picker

'''
Show a single picker for the complete current selection.
顯示目前整批參考圖的多選介面。
'''
_REF_PANEL = widgets.VBox()
_REF_PANEL.children = (_ref_new(),)
_ref_sync(_REF_FILES)
display(_REF_PANEL)
print('目前圖片：', ', '.join(p.name for p in _REF_FILES) or '尚未上傳')

## 9. 生成影片（沿用第 2 節 `TASK`，顯示即時步數進度）


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import json
import shutil
import uuid
import re
import math
import websocket
from tqdm.auto import tqdm

'''
Select the desired task without overwriting independent prompts. Choose the generation task in Section 2.
選擇本次生成的任務，不覆寫其它任務的 Prompt。 在第 2 節選擇：'t2v', 'i2v', 'flf2v', 'r2v'
'''
TASK = globals().get('TASK', 't2v')

'''
Read the selected task's previously executed configuration cell.
讀取所選任務已執行過的設定 Cell。
'''
assert TASK in globals().get('TASK_CONFIGS', {}), f'請先執行 {TASK.upper()} 的參數設定 Cell。'

'''
Resolve the selected task's independent settings.
取得所選任務的獨立參數。
'''
config = TASK_CONFIGS[TASK]

'''
Keep one submission flow shared across all four tasks.
四種生成模式共用同一份工作流程提交邏輯。
'''
MODE = TASK

'''
Use the prompt stored in the selected task cell.
使用目前任務設定 Cell 裡的 Prompt。
'''
PROMPT = config['prompt']

'''
Use the selected canvas settings.
套用所選任務的輸出尺寸。
'''
WIDTH, HEIGHT = config['width'], config['height']

'''
Use the task-specific duration and sampling count.
套用所選任務的秒數與採樣步數。
'''
SECONDS, STEPS = config['seconds'], config['steps']

'''
Apply the task-specific seed and output filename.
套用所選任務的種子與輸出檔名。
'''
SEED, OUTPUT_NAME = config['seed'], config['output_name']

'''
Resolve task-specific reference images without mixing modes.
依據所選任務使用對應參考圖片，避免不同模式混用。
'''
FIRST_IMAGE, LAST_IMAGE = config['first_image'], config['last_image']

'''
Apply the selected task's audio sigma shift without editing the generation graph.
使用此任務的音訊 Sigma Shift，不必修改生成 Workflow。
'''
AUDIO_SHIFT = config['audio_shift']

'''
Apply the selected task's GGUF quantization for model consistency checks.
使用此任務指定的 GGUF 量化格式，以確保模型檢查一致。
'''
QUANT = config['quant']

'''
Reject unsupported quantization names before attempting generation.
生成前先攔截不支援的量化格式名稱。
'''
assert QUANT in QUANT_SIZES_GIB, f'不支援的 QUANT：{QUANT}'

'''
Load reference-only parameters without affecting frame-based tasks.
讀取 R2V 專用參數，不影響原本的首尾影格模式。
'''
REF_IMAGES = config.get('ref_images', [])

'''
Read the reference size strategy only for the new R2V path.
只有 R2V 使用參考圖縮放策略。
'''
REF_IMAGE_SIZE = config.get('ref_image_size', 'match')

'''
Select the encoder residency strategy for the current task.
選擇目前模式的 Encoder 載入策略。
'''
ACTIVE_TEXT_MODE = config.get('text_mode', TEXT_MODE)

'''
Match the numeric range declared by the actual MiniMaxH3SigmaShift node.
比照 MiniMaxH3SigmaShift 節點宣告的數值範圍進行檢查。
'''
assert 0.01 <= AUDIO_SHIFT <= 100.0, 'AUDIO_SHIFT 必須介於 0.01 與 100.0 之間。'

'''
Ensure edited GPU 0 memory budgets have been applied to the running service.
確認調整過的 GPU 0 記憶體預算已套用到目前服務。
'''
assert globals().get('H3_ACTIVE_CONFIG') == H3_DESIRED_CONFIG, 'GPU 0 預算已變更，請重新執行第 4 節啟動 ComfyUI。'

'''
Prevent generation until the chosen quantization was downloaded and registered. Expect a Ref2VA denoiser only when the reference-image task is selected.
切換量化格式後，必須先完成下載與 ComfyUI 模型註冊。 只有選取 R2V 時才使用 Ref2VA 去噪模型。
'''
EXPECTED_VARIANT = 'ref2va_pruned' if MODE == 'r2v' else 'fl2va_pruned'

'''
Require both the model family and the quantization to match the task.
同時確認模型種類及量化格式，避免錯誤沿用 FL2VA 權重。
'''
assert globals().get('H3_MODEL_CHOICE') == (EXPECTED_VARIANT, QUANT) and globals().get('H3_ACTIVE_MODEL') == (EXPECTED_VARIANT, QUANT), (
    f'目前選擇 {QUANT}，但尚未完成下載或服務預檢。請先執行第 2 節與第 4 節。'
)

'''
Ensure the workflow filename always matches the current quantization.
確認 Workflow 檔名與目前選擇的量化格式一致。
'''
assert DIT_NAME == f'minimax_h3_{EXPECTED_VARIANT}-{QUANT}.gguf', '模型檔名與任務種類／QUANT 不一致，請重跑第 2 節。'

'''
Avoid showing a stale result from an earlier successful run after a failure.
避免本次生成失敗後，最後一格仍誤顯示舊影片。
'''
LAST_OUTPUT = None

'''
Confirm the existing local service is still running.
確認先前啟動的本地推論服務仍正常運作。
'''
assert ready(), 'ComfyUI 已中斷，請先重新執行第 4 節。'

'''
Validate the generation mode, prompt, sampling steps, and positive duration.
檢查生成模式、Prompt、採樣步數及秒數是否有效。
'''
assert MODE in ('t2v', 'i2v', 'flf2v', 'r2v') and PROMPT.strip() and STEPS > 0 and math.isfinite(SECONDS) and SECONDS > 0, (
    '請確認生成模式、Prompt、STEPS 與 SECONDS 設定正確；SECONDS 必須為正數。'
)

'''
Require positive dimensions aligned to 32, without an artificial resolution limit.
寬高需為正數且符合 32 的倍數，不另外限制解析度或長寬比。
'''
assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 32 == 0 and HEIGHT % 32 == 0, (
    'WIDTH 與 HEIGHT 必須為正數且是 32 的倍數。'
)

'''
Round the requested duration up to an H3-compatible 17n+5 frame count.
依照 H3 的 17n+5 規則，將指定秒數向上取整為合法影格數。
'''
FRAMES = max(5, 17 * math.ceil((SECONDS * 24 - 5) / 17) + 5)

'''
Respect the underlying ComfyUI node's frame limit, not an arbitrary notebook duration.
遵守 ComfyUI 節點真正的影格數上限，不另外設定 Notebook 的秒數上限。
'''
assert FRAMES <= 3600, f'要求 {FRAMES} frames，超出 MiniMax H3 節點上限 3600；請縮短 SECONDS。'

'''
Generate a fresh 32-bit seed only when -1 is requested.
只有設定 -1 時才建立新的 32-bit 隨機種子。
'''
import secrets

'''
Resolve one seed per run without overwriting the notebook setting.
每次生成決定一個種子，但不覆寫 Notebook 的原始設定。
'''
RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)

'''
Require a supported seed before building the inference workflow.
在建立推論工作流程前，先檢查種子範圍。
'''
assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'

'''
Show the actual seed so a random run can be reproduced later.
顯示實際使用的種子，方便之後重現隨機生成結果。
'''
print(f'本次生成 seed：{RUN_SEED}')

'''
Build a unique ComfyUI output prefix from the user filename.
根據指定檔名建立不重複的 ComfyUI 輸出前綴。
'''
output_stem = Path(OUTPUT_NAME).stem

'''
Make each run independent even when settings are unchanged.
即使設定相同，也為每次生成保留獨立檔案。
'''
run_name = f'{output_stem}_{EXPECTED_VARIANT}_{QUANT}_{uuid.uuid4().hex[:8]}'

'''
Create an H3 workflow and detect the Qwen3-VL encoder from its checkpoint.
使用 H3 節點建立 Workflow，並從權重檔自動辨識 Qwen3-VL 編碼器。
'''
graph = {
    '1': {'class_type': 'ClipProjLoader', 'inputs': {

        'clip_name': 'qwen3vl_4b_fp8_scaled.safetensors', 'type': 'auto',
        'projection': 'mmh3-4b-ClipProj-v3.1-mlp.safetensors',
        'device': TEXT_GPU, 'mode': ACTIVE_TEXT_MODE}},
    '2': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': DIT_NAME}},
    '3': {'class_type': 'MiniMaxH3SigmaShift', 'inputs': {
        'model': ['2', 0], 'shift_video': 12.0, 'shift_audio': AUDIO_SHIFT}},
    '4': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'minimax_h3_video_vae_fp16.safetensors'}},
    '5': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'minimax_h3_audio_vae_fp32.safetensors'}},
    '6': {'class_type': 'MiniMaxH3ImageToVideo', 'inputs': {
        'clip': ['1', 0], 'vae': ['4', 0], 'prompt': PROMPT,
        'width': WIDTH, 'height': HEIGHT, 'length': FRAMES}},
    '7': {'class_type': 'BasicGuider', 'inputs': {
        'model': ['3', 0], 'conditioning': ['6', 0]}},
    '8': {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'res_multistep'}},
    '9': {'class_type': 'BasicScheduler', 'inputs': {
        'model': ['3', 0], 'scheduler': 'simple', 'steps': STEPS, 'denoise': 1.0}},
    '10': {'class_type': 'RandomNoise', 'inputs': {'noise_seed': RUN_SEED}},
    '11': {'class_type': 'SamplerCustomAdvanced', 'inputs': {
        'noise': ['10', 0], 'guider': ['7', 0],
        'sampler': ['8', 0], 'sigmas': ['9', 0], 'latent_image': ['6', 1]}},
    '12': {'class_type': 'VAEDecode', 'inputs': {'samples': ['11', 0], 'vae': ['4', 0]}},
    '13': {'class_type': 'VAEDecodeAudio', 'inputs': {'samples': ['11', 0], 'vae': ['5', 0]}},
    '14': {'class_type': 'CreateVideo', 'inputs': {
        'images': ['12', 0], 'audio': ['13', 0], 'fps': 24.0}},
    '15': {'class_type': 'SaveVideo', 'inputs': {
        'video': ['14', 0], 'filename_prefix': run_name,
        'format': 'auto'}},
}

'''
Use Ref2VA's separate conditioning node and attach video/audio VAEs.
R2V 改用獨立條件節點，並連接影片及音訊 VAE。
'''
if MODE == 'r2v':

    '''
    Check that the number of reference pictures meets the native node's limit.
    檢查參考圖片張數符合原生節點限制。
    '''
    assert isinstance(REF_IMAGES, (tuple, list)) and 1 <= len(REF_IMAGES) <= 9, 'R2V 需要 1～9 張參考圖片。'

    '''
    Reject unsupported reference scaling options early.
    提前攔截無效的參考圖尺寸選項。
    '''
    assert REF_IMAGE_SIZE in ('match', 'max'), 'R2V_REF_IMAGE_SIZE 只能是 match 或 max。'

    '''
    Extract the 1-based Picture numbers explicitly used in the prompt.
    讀取 Prompt 中實際使用的 Picture 編號（從 1 開始）。
    '''
    picture_numbers = {int(number) for number in re.findall(r'<Picture\s+(\d+)>', PROMPT, flags=re.IGNORECASE)}

    '''
    Refuse to submit a prompt referencing an absent image.
    若 Prompt 指向不存在的圖片，避免送出無效的參考條件。
    '''
    assert all(1 <= number <= len(REF_IMAGES) for number in picture_numbers), (
        f'R2V_PROMPT 引用了不存在的 <Picture N>：{sorted(picture_numbers)}；目前只有 {len(REF_IMAGES)} 張。'
    )

    '''
    Validate this enum against the live ComfyUI node when possible.
    盡可能使用正在執行的 ComfyUI 節點驗證此選項。
    '''
    check_choice('MiniMaxH3ReferenceToVideo', 'ref_image_size', REF_IMAGE_SIZE)

    '''
    Reuse the normal output dimensions and sampling settings.
    沿用與原本模式相同的解析度與採樣設定。
    '''
    graph['6'] = {'class_type': 'MiniMaxH3ReferenceToVideo', 'inputs': {
        'clip': ['1', 0], 'vae': ['4', 0], 'audio_vae': ['5', 0],
        'prompt': PROMPT, 'width': WIDTH, 'height': HEIGHT, 'length': FRAMES,
        'ref_image_size': REF_IMAGE_SIZE}}

    '''
    Register each picture in the order used by the prompt's 1-based Picture tags.
    依 Prompt 的 Picture 1、Picture 2 順序登記每張參考圖片。
    '''
    for image_index, image_path in enumerate(REF_IMAGES):

        '''
        Stop before generation if the specified reference does not exist.
        指定的參考圖片不存在時，先停止生成。
        '''
        source = Path(image_path)

        '''
        Check that a valid local file path was provided.
        確認參考圖片路徑有效且檔案存在。
        '''
        assert source.is_file(), f'R2V 找不到第 {image_index + 1} 張參考圖片：{source}'

        '''
        Accept normal image formats understood by ComfyUI's LoadImage node.
        限定 ComfyUI LoadImage 可辨識的常見圖片格式。
        '''
        assert source.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp', '.bmp'), f'不支援的參考圖片格式：{source.suffix}'

        '''
        Allocate distinct source node IDs for reference pictures.
        為每張參考圖片分配獨立的載入節點 ID。
        '''
        node_id = str(16 + image_index)

        '''
        Copy files into ComfyUI's private input directory with unique names.
        將參考圖片複製到 ComfyUI 私有輸入資料夾，並使用不重複檔名。
        '''
        destination = COMFY / 'input' / f'r2v_{image_index + 1}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'

        '''
        Create the image input directory if needed.
        若資料夾不存在則建立。
        '''
        destination.parent.mkdir(parents=True, exist_ok=True)

        '''
        Preserve the exact image bytes, without introducing unintended transforms.
        保留圖片原始內容，不額外修改圖片。
        '''
        shutil.copy2(source, destination)

        '''
        Connect a standard LoadImage node for this reference.
        為這張圖片建立標準 LoadImage 節點。
        '''
        graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}

        '''
        Autogrow socket indices are zero-based, unlike the Picture 1 prompt tag.
        Autogrow 輸入索引從 0 開始，與 Prompt 的 Picture 1 不同。
        '''
        graph['6']['inputs'][f'ref_images.ref_image_{image_index}'] = [node_id, 0]

'''
Attach a verified local image to an H3 image-conditioning socket.
將確認存在的本地圖片接到 H3 圖片條件輸入。
'''
for mode, image_path, node_id, input_name in [
    ('i2v', FIRST_IMAGE, '16', 'first_frame'),
    ('flf2v', FIRST_IMAGE, '16', 'first_frame'),
    ('flf2v', LAST_IMAGE, '17', 'last_frame'),
]:

    '''
    Only attach images required by the active mode.
    只接入目前模式需要的圖片。
    '''
    if MODE == mode:

        '''
        Require an existing image file for the selected mode.
        確認所選模式的圖片檔案確實存在。
        '''
        source = Path(image_path)

        '''
        Stop before generation if a reference file is missing.
        若圖片不存在，在生成前立即停止。
        '''
        assert source.is_file(), f'找不到參考圖片：{source}'

        '''
        Copy the input to the ComfyUI image directory.
        將圖片複製到 ComfyUI 輸入目錄。
        '''
        destination = COMFY / 'input' / f'{node_id}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'

        '''
        Create the target input folder when needed.
        視需要建立圖片輸入資料夾。
        '''
        destination.parent.mkdir(parents=True, exist_ok=True)

        '''
        Preserve the reference image without changing it.
        保留原始圖片內容，不進行修改。
        '''
        shutil.copy2(source, destination)

        '''
        Register the image loader in the API graph.
        將圖片載入節點加入 API Workflow。
        '''
        graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}

        '''
        Connect the loaded image to H3 conditioning.
        將已載入圖片連到 H3 條件輸入。
        '''
        graph['6']['inputs'][input_name] = [node_id, 0]

'''
Re-read relevant dropdowns because model lists or user parameters may change after startup.
在提交前重新確認下拉選單，避免啟動後變更模型或設定造成不相容。
'''
NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()

'''
Validate the user's GPU 1 choice against the actual encoder node.
依照當前編碼器節點的選項檢查 GPU 1 設定。
'''
check_choice('ClipProjLoader', 'device', TEXT_GPU)

'''
Validate the encoder transfer mode without assuming dropdown metadata is always present.
驗證文字編碼器載入模式，但不假設節點一定有固定選項清單。
'''
check_choice('ClipProjLoader', 'mode', ACTIVE_TEXT_MODE)

'''
Verify the H3 sampler against the specialized node or the core KSampler options.
優先用目前採樣器選單驗證；必要時參照核心 KSampler 選項。
'''
check_choice('KSamplerSelect', 'sampler_name', 'res_multistep', ('KSampler', 'sampler_name'))

'''
Verify the scheduler, falling back to the core sampler's shared scheduler enum.
驗證排程器；需要時使用核心採樣器提供的選項清單。
'''
check_choice('BasicScheduler', 'scheduler', 'simple', ('KSampler', 'scheduler'))

'''
Confirm optional first and last frame sockets exist for the selected image modes.
在使用圖片生成影片時，確認節點確實支援首尾影格輸入。
'''
if MODE in ('i2v', 'flf2v'):

    '''
    Read the available optional conditioning inputs from the running node definition.
    讀取執行中影格條件節點的選填輸入規格。
    '''
    frame_inputs = NODE_INFO['MiniMaxH3ImageToVideo'].get('input', {}).get('optional', {})

    '''
    Require the first frame, plus the last frame only in FLF2V mode.
    圖生影片必須支援第一格；首尾影格模式還需要最後一格。
    '''
    expected = ('first_frame', 'last_frame') if MODE == 'flf2v' else ('first_frame',)

    '''
    Reject an incompatible core node implementation before generation.
    如果核心節點不支援這些欄位，在生成之前先停止。
    '''
    assert all(field in frame_inputs for field in expected), f'MiniMaxH3ImageToVideo 缺少 {expected} 輸入。'

'''
Confirm that the running Ref2VA node supports its autogrow image family.
確認目前 R2V 節點支援可擴充的參考圖片輸入。
'''
if MODE == 'r2v':

    '''
    Read the current node's required and optional inputs.
    讀取目前節點的必填與選填輸入。
    '''
    ref_inputs = NODE_INFO['MiniMaxH3ReferenceToVideo'].get('input', {})

    '''
    Gather the names of autogrow parent inputs.
    取得可擴充輸入的母欄位名稱。
    '''
    ref_fields = set(ref_inputs.get('required', {})) | set(ref_inputs.get('optional', {}))

    '''
    Reject versions that do not expose reference-image autogrow inputs.
    如果節點版本缺少參考圖片輸入，就提前停止。
    '''
    assert 'ref_images' in ref_fields or any(name.startswith('ref_images.ref_image_') for name in ref_fields), (
        'MiniMaxH3ReferenceToVideo 缺少 ref_images 輸入，請更新 ComfyUI。'
    )

'''
Validate that all graph dependency links target existing nodes.
驗證所有 Workflow 連線的來源節點確實存在。
'''
for node_id, node in graph.items():

    '''
    Examine each declared input for possible node-reference links.
    檢查節點輸入中是否包含連到其它節點的參照。
    '''
    for value in node['inputs'].values():

        '''
        Only two-element lists are ComfyUI graph connections.
        只有雙元素清單才可能代表 ComfyUI 節點連線。
        '''
        if isinstance(value, list) and len(value) == 2:

            '''
            Stop immediately if a linked source node is missing.
            若連結到不存在的來源節點，立即顯示具體錯誤。
            '''
            assert str(value[0]) in graph, f'節點 {node_id} 連結到不存在的節點 {value[0]}。'

'''
Compare each workflow input against the actual node input definitions.
依照執行中 ComfyUI 的節點規格檢查每個 Workflow 參數。
'''
for node_id, node in graph.items():

    '''
    Read this node's current ComfyUI schema and accepted input names.
    讀取目前節點允許的輸入欄位名稱。
    '''
    schema = NODE_INFO[node['class_type']].get('input', {})

    '''
    Include required, optional and system-provided hidden inputs.
    將必填、選填與系統隱藏欄位全部納入檢查。
    '''
    accepted = set().union(*(set(schema.get(group, {})) for group in ('required', 'optional', 'hidden')))

    '''
    Find parameters removed or renamed by a later ComfyUI revision.
    找出可能被新版 ComfyUI 移除或改名的參數。
    '''
    unknown = set(node['inputs']) - accepted

    '''
    Allow dynamic Ref2VA image slots declared by the live autogrow parent.
    當節點宣告 ref_images 動態欄位時，允許自動展開的圖片插槽。
    '''
    if node['class_type'] == 'MiniMaxH3ReferenceToVideo' and 'ref_images' in accepted:

        '''
        Remove only valid zero-based reference slots from the unknown set.
        只略過合法的 0～8 參考圖片插槽，不忽略其它錯誤輸入。
        '''
        unknown -= {f'ref_images.ref_image_{i}' for i in range(9)}

    '''
    Fail early if a workflow input is no longer supported.
    當 Workflow 參數不再被節點支援時，立即停止並指出名稱。
    '''
    assert not unknown, f"節點 {node_id} ({node['class_type']}) 不支援的輸入：{sorted(unknown)}"

'''
Print the selected task and workload before expensive sampling.
在開始耗時生成前顯示本次任務與工作量。
'''
print(f'{MODE.upper()} | {WIDTH}×{HEIGHT} | {FRAMES} frames | {STEPS} steps | {EXPECTED_VARIANT}/{QUANT} | Audio Shift {AUDIO_SHIFT}', flush=True)

'''
Assign a unique WebSocket client identity to this generation.
為此次生成建立獨立的 WebSocket 客戶端識別碼。
'''
client_id = uuid.uuid4().hex

'''
Track WebSocket availability separately from workflow success.
個別追蹤 WebSocket 可用性，不影響實際生成流程。
'''
ws = None

'''
Connect before queueing to avoid missing early progress messages.
在送出任務前先連線，避免漏掉初期進度事件。
'''
try:

    '''
    Connect to the documented ComfyUI progress event stream.
    連接 ComfyUI 官方 WebSocket 進度事件串流。
    '''
    ws = websocket.create_connection(f'ws://127.0.0.1:{PORT}/ws?clientId={client_id}', timeout=5)

    '''
    Limit individual reads so failure and completion are checked regularly.
    限制單次等待時間，定期檢查錯誤與任務完成狀態。
    '''
    ws.settimeout(2)

except (OSError, websocket.WebSocketException):

    '''
    Continue inference even when live progress is unsupported.
    即使即時進度連線失敗，也不影響模型推論。
    '''

    '''
    Mark event updates as unavailable and retain an elapsed-time display.
    停用事件更新，改成單一進度列顯示經過時間。
    '''
    ws = None

    '''
    Explain why no step percentage will be shown.
    說明為什麼目前不會出現步數百分比。
    '''
    print('即時步數事件不可用；將顯示階段與經過時間。', flush=True)

'''
Send the workflow to the private local ComfyUI service.
將任務送交本機 ComfyUI 服務。
'''
response = requests.post(f'{BASE}/prompt', json={'prompt': graph, 'client_id': client_id}, timeout=120)

'''
Report invalid workflows before creating a misleading progress bar.
工作流程驗證失敗時先回報，不顯示誤導性的進度條。
'''
if not response.ok:

    '''
    Close an unused WebSocket before raising the server-side error.
    回報錯誤前先關閉不再使用的 WebSocket。
    '''
    if ws is not None:

        '''
        Release the client connection.
        釋放 WebSocket 連線。
        '''
        ws.close()

    '''
    Preserve ComfyUI's validation details.
    保留 ComfyUI 提供的參數驗證錯誤資訊。
    '''
    raise RuntimeError(response.text[:5000])

'''
Remember the prompt ID so progress messages belong to this exact task.
保留任務 ID，避免將其它生成工作的進度混入。
'''
prompt_id = response.json()['prompt_id']

'''
Start timing the actual queued job.
開始計算已提交生成工作的耗時。
'''
start = time.monotonic()

'''
Create a single notebook progress bar with real step events only.
建立單一 Notebook 進度條，僅使用真實採樣步數更新。
'''
pbar = tqdm(total=STEPS, desc='排隊／載入模型', unit='step', leave=True)

'''
Preserve a final history record for successful completion checking.
保留最終生成紀錄，用於確認任務完成。
'''
item = None

'''
Avoid repeatedly polling history when the WebSocket reports frequent events.
WebSocket 事件頻繁時避免過度查詢生成紀錄。
'''
last_history_check = 0.0

'''
Define truthful node-stage names for messages without step progress.
將節點 ID 對應到真實執行階段，沒有步數時仍可辨識狀態。
'''
phases = {'1': '文字編碼器', '2': '載入 GGUF', '6': '建立影片條件',
          '11': '採樣中', '12': '影片解碼', '13': '音訊解碼',
          '14': '合成影片', '15': '儲存影片'}

'''
Wait until ComfyUI reports a final history record for this task.
持續等待 ComfyUI 回傳此任務的最終執行紀錄。
'''
try:

    '''
    Keep processing until success, an explicit error, or timeout.
    持續處理直到成功、明確錯誤或逾時。
    '''
    while True:

        '''
        Measure real wall-clock time instead of estimating model progress.
        計算真實經過時間，不將時間誤當模型百分比。
        '''
        elapsed = time.monotonic() - start

        '''
        Bound long-running jobs to the current Kaggle session.
        限制長時間工作，避免單次任務無限等待。
        '''
        if elapsed > 7200:

            '''
            Include the prompt ID for later recovery.
            保留任務 ID 供後續查詢結果。
            '''
            raise TimeoutError(f'生成逾時；prompt_id={prompt_id}，可查詢 {BASE}/history/{prompt_id}')

        '''
        Receive a WebSocket message if this ComfyUI version supports it.
        如果目前版本支援 WebSocket，讀取即時進度事件。
        '''
        message = None

        '''
        Read events only while the client connection remains healthy.
        只有 WebSocket 正常連線時才讀取事件。
        '''
        if ws is not None:

            '''
            Keep timeout exceptions from stopping generation.
            WebSocket 讀取逾時不應中斷生成。
            '''
            try:

                '''
                Receive a progress event or stage transition.
                接收採樣進度或執行階段變更。
                '''
                message = ws.recv()

            except websocket.WebSocketTimeoutException:

                '''
                A quiet model-loading phase is normal and not a failure.
                模型載入期間沒有事件屬正常情況，不代表失敗。
                '''

                '''
                Continue checking job history after a read timeout.
                讀取逾時後繼續檢查任務紀錄。
                '''
                pass

            except (OSError, websocket.WebSocketException):

                '''
                Recover gracefully when the event channel becomes disconnected.
                進度通道中斷時回退到紀錄輪詢，不影響生成。
                '''

                '''
                Release the unusable client connection.
                關閉失效的 WebSocket 連線。
                '''
                ws.close()

                '''
                Disable further reads from the disconnected socket.
                停用已中斷的進度連線。
                '''
                ws = None

        '''
        Only JSON text frames contain progress data; preview images are binary.
        只有 JSON 文字事件含進度資料；預覽圖片為二進位資料。
        '''
        if isinstance(message, str):

            '''
            Parse a ComfyUI WebSocket event.
            解析 ComfyUI WebSocket 事件。
            '''
            event = json.loads(message)

            '''
            Read the event payload without assuming optional keys exist.
            讀取事件資料，並避免假設選填欄位一定存在。
            '''
            data = event.get('data') or {}

            '''
            Reject progress from another client or generation.
            忽略非本次任務的進度，避免步數混亂。
            '''
            if data.get('prompt_id') == prompt_id:

                '''
                Track which part of the workflow is executing.
                追蹤目前執行中的 workflow 節點。
                '''
                if event.get('type') == 'executing' and data.get('node'):

                    '''
                    Show the real stage rather than inventing a percentage.
                    顯示實際節點階段，不虛構完成百分比。
                    '''
                    pbar.set_description(phases.get(str(data['node']), '執行中'), refresh=False)

                '''
                Use official step updates when the sampler reports them.
                採樣器回報步數時，使用官方即時進度。
                '''
                if event.get('type') in ('progress', 'progress_state'):

                    '''
                    Classic events store step counts directly in the payload.
                    傳統進度事件直接在資料中提供步數。
                    '''
                    progress = data

                    '''
                    Extended events report per-node progress within a nodes object.
                    新版事件使用 nodes 字典提供各節點步數。
                    '''
                    if event.get('type') == 'progress_state':

                        '''
                        Read only this workflow's sampling node.
                        只讀取此 workflow 的採樣節點進度。
                        '''
                        progress = data.get('nodes', {}).get('11', {})

                    '''
                    Ignore progress emitted by loading or decoding nodes.
                    忽略非採樣節點的進度，以免誤顯示百分比。
                    '''
                    if str(progress.get('node', progress.get('node_id'))) == '11':

                        '''
                        Read the actual completed and total sampling steps.
                        取得 ComfyUI 回報的已完成與總採樣步數。
                        '''
                        value, maximum = progress.get('value'), progress.get('max')

                        '''
                        Validate numeric counters before modifying the display.
                        先確認進度欄位為有效數值，再更新畫面。
                        '''
                        if isinstance(value, (int, float)) and isinstance(maximum, (int, float)) and maximum > 0:

                            '''
                            Follow the sampler's actual total if it differs from the setting.
                            如果採樣器回報不同步數，優先使用實際總數。
                            '''
                            pbar.total = int(maximum)

                            '''
                            Update the bar to the real step count, never a time-based estimate.
                            依據真實步數更新進度，不使用耗時推估。
                            '''
                            pbar.n = max(0, min(int(value), pbar.total))

                            '''
                            Identify the phase accurately.
                            正確標示為採樣階段。
                            '''
                            pbar.set_description('採樣中', refresh=False)

        '''
        Check the persistent history regardless of WebSocket availability.
        不論 WebSocket 是否正常，都定期檢查 ComfyUI 任務紀錄。
        '''
        if elapsed - last_history_check >= 4:

            '''
            Record the most recent history query to avoid duplicate requests.
            更新上次查詢時間，避免頻繁重複送出請求。
            '''
            last_history_check = elapsed

            '''
            Show recent logs if the service died under memory pressure.
            如果服務因記憶體壓力中斷，顯示最近的錯誤 Log。
            '''
            if not ready():

                '''
                Preserve recent server logs for actionable diagnostics.
                保留最近服務紀錄供後續除錯。
                '''
                raise RuntimeError('ComfyUI 服務中斷：\n' + LOG.read_text(errors='replace')[-4500:])

            '''
            Read this prompt's final history record.
            讀取本次生成工作的最終執行紀錄。
            '''
            item = requests.get(f'{BASE}/history/{prompt_id}', timeout=60).json().get(prompt_id)

            '''
            Stop polling once the job has a final record.
            任務產生最終紀錄後停止輪詢。
            '''
            if item:

                '''
                Extract node-specific execution errors before claiming success.
                在宣告成功前確認各節點是否有執行錯誤。
                '''
                errors = [m for m in item.get('status', {}).get('messages', []) if m[0] == 'execution_error']

                '''
                Report the precise node failure when one is present.
                如果某個節點失敗，回報其完整錯誤資訊。
                '''
                if errors:

                    '''
                    Surface the error from the last failed node.
                    顯示最後一個失敗節點的錯誤。
                    '''
                    raise RuntimeError(str(errors[-1][1])[:5000])

                '''
                Check the server's final status before accepting a completed history item.
                讀取伺服器最後狀態，避免把中止或失敗的任務當作成功。
                '''
                assert item.get('status', {}).get('status_str') != 'error', (
                    f'ComfyUI 工作失敗：{item.get("status", {})}'
                )

                '''
                Complete only after ComfyUI history confirms success.
                只有 ComfyUI 紀錄確認完成才結束等待。
                '''
                break

        '''
        Update one stable progress row with elapsed time and no log flooding.
        在同一條進度列更新經過時間，避免持續洗版。
        '''
        pbar.set_postfix_str(f'{elapsed / 60:.1f} 分鐘', refresh=False)

        '''
        Refresh the existing row in-place.
        只刷新原進度列，不新增大量訊息行。
        '''
        pbar.refresh()

        '''
        Avoid tight polling if the WebSocket connection is unavailable.
        WebSocket 不可用時稍作等待，避免過密輪詢。
        '''
        if ws is None:

            '''
            Yield CPU time while generation runs in the server.
            讓伺服器推論期間不持續佔用 Notebook CPU。
            '''
            time.sleep(1)

    '''
    Mark the sampler display finished only after the full job succeeded.
    只有整個工作流程成功後才將進度條標示為完成。
    '''
    pbar.n = pbar.total

    '''
    Label the status as completed.
    將顯示狀態設為完成。
    '''
    pbar.set_description('完成', refresh=False)

    '''
    Refresh the final elapsed-time counter.
    更新任務完成時的總耗時。
    '''
    pbar.set_postfix_str(f'{(time.monotonic() - start) / 60:.1f} 分鐘', refresh=False)

    '''
    Show the final progress state.
    顯示最終完成狀態。
    '''
    pbar.refresh()

finally:

    '''
    Always free the live progress widgets and connection after this job.
    無論任務成功或失敗，都釋放進度列與 WebSocket 連線。
    '''

    '''
    Stop updating the notebook progress widget.
    停止更新 Notebook 的進度顯示。
    '''
    pbar.close()

    '''
    Close the WebSocket when it exists.
    若 WebSocket 仍存在，關閉它以釋放資源。
    '''
    if ws is not None:

        '''
        Release the local event subscription.
        釋放本地進度事件連線。
        '''
        ws.close()

'''
Locate the generated video in ComfyUI output directories.
從 ComfyUI 輸出資料夾找出本次生成影片。
'''
files = [p for p in (COMFY / 'output').rglob('*') if p.is_file() and p.name.startswith(run_name)]

'''
Keep only playable video formats.
只保留可播放的影片格式。
'''
files = [p for p in files if p.suffix.lower() in ('.mp4', '.webm', '.mov')]

'''
Refuse to claim success when no video was saved.
沒有生成影片時，不誤判為完成。
'''
assert files, f'工作流程完成但未找到影片：{item.get("outputs", {})}'

'''
Use the most recently modified matching video file.
選取符合本次輸出名稱的最新影片。
'''
source_video = max(files, key=lambda path: path.stat().st_mtime)

'''
Save the user-visible output in Kaggle working storage.
將成果儲存至 Kaggle working 目錄。
'''
OUTPUT_DIR = ROOT / 'minimax_h3_outputs'

'''
Ensure the output directory exists.
確保輸出資料夾存在。
'''
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

'''
Preserve the actual video container extension.
保留生成影片實際使用的副檔名。
'''
output_base = OUTPUT_DIR / f'{Path(OUTPUT_NAME).stem}_{EXPECTED_VARIANT}_{QUANT}{source_video.suffix}'

'''
Generate an unused path instead of overwriting prior outputs.
產生未使用的檔名，避免覆蓋先前影片。
'''
output = output_base

'''
Number duplicate filenames deterministically.
依序為重複檔名編號。
'''
for index in range(2, 1000):

    '''
    Stop once an available filename is found.
    找到可使用的檔名後停止。
    '''
    if not output.exists():

        '''
        Preserve the current unique output path.
        保留目前可用的輸出檔名。
        '''
        break

    '''
    Allocate the next numbered filename.
    配置下一個編號檔名。
    '''
    output = output_base.with_name(f'{output_base.stem}_{index:03d}{output_base.suffix}')

'''
Copy the completed video to the user's output directory.
將影片複製到使用者的輸出目錄。
'''
shutil.copy2(source_video, output)

'''
Report the result, elapsed time, and inference cache behavior.
顯示檔案、耗時與模型快取狀態。
'''
print(f'完成：{output}｜耗時 {(time.monotonic() - start) / 60:.1f} 分鐘')

'''
Save the latest successful result for the final preview/download cell.
將本次成功生成的影片提供給最後一格預覽與下載。
'''
LAST_OUTPUT = output

'''
Confirm that the video is ready without embedding it in the generation cell.
確認影片已輸出，生成 Cell 本身不顯示預覽。
'''
print('✓ 第 9 節完成：影片已生成；請執行第 10 節預覽與下載。')


## 10. 影片預覽、下載與 GPU 使用量


In [ ]:
'''
Display the most recently generated video and its download link.
顯示最近一次成功生成影片的預覽與下載連結。
'''
from IPython.display import Video, FileLink, display

'''
Require a real output from the preceding generation cell.
確認前一個生成 Cell 已成功輸出影片。
'''
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 9 節生成影片。'

'''
Embed the generated video preview only in this final cell.
只在最後一格嵌入影片預覽，避免生成 Cell 輸出太多內容。
'''
display(Video(str(LAST_OUTPUT), embed=True, width=480))

'''
Provide a clickable Kaggle file download link.
顯示 Kaggle 影片檔案下載連結。
'''
display(FileLink(str(LAST_OUTPUT)))

'''
Check both GPUs without unloading any cached model weights.
在不卸載模型快取的情況下查詢兩張 GPU 的顯示記憶體。
'''
status = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.used,memory.total',
     '--format=csv,noheader,nounits'], capture_output=True, text=True, check=True
)

'''
Show the actual readings without claiming they are exact model-memory allocations.
顯示實際使用量，不將其誤當作單一模型的精確權重用量。
'''
print('GPU 記憶體使用量（MiB）：')

'''
Display both devices in a readable form.
逐行顯示兩張 GPU 的使用狀態。
'''
print(status.stdout.strip())

'''
Compare GPU readings with advisory thresholds without claiming hard caps.
將兩張 GPU 的用量與警示值比較，不把它們誤認成精確硬上限。
'''
for line in status.stdout.splitlines():

    '''
    Split GPU usage fields reported by nvidia-smi.
    拆解 nvidia-smi 回傳的 GPU 使用量欄位。
    '''
    fields = [part.strip() for part in line.split(',')]

    '''
    Check either GPU and a well-formed memory reading.
    確認資料屬於兩張 GPU 之一，且包含有效用量數值。
    '''
    if len(fields) == 4 and fields[0] in ('0', '1') and fields[2].isdigit():

        '''
        Convert MiB readings to the same GiB unit as the user setting.
        將 MiB 轉換成與使用者設定相同的 GiB 單位。
        '''
        used_gib = int(fields[2]) / 1024

        '''
        Compare each device with its relevant configured target.
        將每張 GPU 的使用量與各自設定的目標進行比較。
        '''
        limit_gib = GPU0_MODEL_GIB if fields[0] == '0' else GPU1_MODEL_GIB

        '''
        Report an advisory warning if the whole device exceeds its target.
        實際整張 GPU 用量超過目標時顯示提醒，不將目標誤認為硬上限。
        '''
        if used_gib > limit_gib:

            '''
            Explain why the threshold cannot force CPU offload.
            說明警示門檻本身無法強制卸載至 CPU。
            '''
            print(f'提醒：GPU {fields[0]} 使用 {used_gib:.2f} GiB，超過目標／監控值 {limit_gib:.2f} GiB；非硬上限。')

'''
Confirm the notebook's preview and download step finished.
明確顯示預覽與下載步驟已完成。
'''
print(f'✓ 第 10 節完成：影片預覽與下載連結已顯示；{LAST_OUTPUT.name}')


## 11. 擷取最後影格（選用）


In [ ]:
'''
Count decoded video frames to identify the final frame.
計算影片影格數，找出最後一個影格。
'''
import json
from IPython.display import FileLink, display
from PIL import Image

'''
Ensure a generated MP4 exists before extracting its last frame.
確認前面已成功生成 MP4，再進行最後影格擷取。
'''
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 9 節生成影片，並確認第 10 節可正常預覽。'

'''
Confirm that FFprobe and FFmpeg are installed before decoding the video.
擷取影片影格前，先檢查 FFprobe 與 FFmpeg 是否已安裝。
'''
assert shutil.which('ffprobe') and shutil.which('ffmpeg'), '找不到 ffprobe 或 ffmpeg；請先安裝 FFmpeg。'

'''
Build a PNG path next to the generated video.
在生成影片旁建立最後影格 PNG 檔案路徑。
'''
LAST_FRAME_PATH = LAST_OUTPUT.with_name(LAST_OUTPUT.stem + '_last_frame.png')

'''
Read the frame count from the saved MP4.
從已儲存的 MP4 讀取影格總數。
'''
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v:0',
     '-show_entries', 'stream=nb_read_frames', '-of', 'json', str(LAST_OUTPUT)],
    capture_output=True, text=True, check=True,
)

'''
Convert the frame count to a zero-based index.
將影格總數轉成從零開始的最後影格索引。
'''
last_index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1

'''
Save the exact last frame as a PNG for the next I2V clip.
將最後一個影格儲存成 PNG，供下一段 I2V 使用。
'''
subprocess.run(
    ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(LAST_OUTPUT),
     '-vf', rf'select=eq(n\,{last_index})', '-frames:v', '1', str(LAST_FRAME_PATH)],
    check=True,
)

'''
Preview and link the extracted image.
顯示擷取圖片並提供下載連結。
'''
display(Image.open(LAST_FRAME_PATH))
display(FileLink(str(LAST_FRAME_PATH)))

'''
Show the input path to use for the next video.
顯示下一段影片可直接使用的圖片路徑。
'''
print(f"下一段：TASK = 'i2v'，FIRST_IMAGE = '{LAST_FRAME_PATH}'")
print(f'✓ 第 11 節完成：已擷取最後影格；{LAST_FRAME_PATH.name}')


## 12. ngrok 與遠端生成 API 啟動

In [ ]:
'''
Dependencies are available in the Python standard library.
只使用 Python 標準函式庫處理遠端服務與檔案。
'''
import zlib
import math
import re

import queue
import copy

'''
Use secure randomness when the API requests seed=-1.
API 指定 seed=-1 時，以安全的隨機來源產生實際種子。
'''
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

'''
Expose only the inference bridge, never the Jupyter or ComfyUI management interfaces.
ngrok 僅轉送推論橋接 API，不公開 Jupyter 或 ComfyUI 管理介面。
'''
REMOTE_PORT = 8765
REMOTE_SERVICE_ID = '06_MiniMax_H3_影片生成_Kaggle'
REMOTE_KIND = 'video'
REMOTE_FIELDS = ['task', 'prompt', 'seed', 'width', 'height', 'seconds', 'steps', 'ref_image_size', 'audio_shift']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\xbd}kW\x14W\xda\xe8\xf7\xfc\x8a:3\xeb\xacn&@\x00M2\xf1\x0cs\x96\xa3h8\x93\xa8\xafh\xe6}\x8f\xaf\xabW\xd1]\r5\xf6-U\xdd \xc3\xcbZ\xa0" "&\x11\x0cB\xbc_\x18\x8d\xa0\x89Q\x04\xc4\x0f\xef?9CUw\x7f\xf2/\x9c\xe7\xb2\xf7\xae]\xd5\x05I\xe6\\\x9c\xb52tuu\xd5\xb3\x9f\xfd\xdco\xfb\xb7FO\xa57o\x97\r\xd3\xc8\x15\xd3f\xce8P\xccg\x87Nv\x1b}\x8eY\xea7\xccB\xc68cY%#_\xccX9\xd7H\x9b\xe9~+cd\x8b\x8eQ\xee\xb7\x8c\x82u\xb6l8\xd6\x97\x15\xcb-\xb7\xbe\xf7[\xa3>:\xe6M\xac\xfbK\xdf{K\xcf\xd4\x93\xbcW\x0f\xb6\xdf,\xf9?\x8dU\x97\xa7\xdfm^\xde~\xfb]un\xc1_\xbe\xe3\xdd\x9c\xf6\xde>\xf1f\xe7\xb7\xb7\x16\xb7\xd7\xa6\xfd\xef\xefT\xaf\xdd\xf2\'\xaf\xfec\xf4\xdc{v\xbeTt\xca\xc6_\xddb\xe1=x\xee\x81bi\xc8\xb0\x0b\xa5J\xd9\xb0\xf3f\x9f\xe5\xc2\x87r\x91@(9\xf6\x80Y\xb6\xd4\xdb\xf8\xael1\x97\xb1\x1c\x04\xc9{6\xe1\xcd\x9e\xaf\x8d\x9e\xf7\x96\xe6\xabS\x13\xb5\xfb\x13\xb5{o\xbc\xc9\x00\xbc\xea\x8d\x0b\xb5\xcd5o\xfcA\xed\xc7\t\x7f~\xc1\xbb\xbf\xa5\x01\xe0\xf6W\xcav\x0eA8l\x15,\x07_T)\xd8\xb0^#k\xe7\xac\x82\x99\x07P\x18\x19\xb6k8\x95\x02\xbe\xb1z\x0e1\xa0\x96\xe3m\xacW\x9fLWg\x96\xe1\xbf\xfe\xe3k\xde\xd5\x19\xed\xf9\x95\x8a\x9d\xc1\xa7\x1f3\x1d\x17\xd0Y\xc9\xf7Z\x0e\xe0\xf7\x98\x9d.W\x1c\xcb(\x9b}\xb8T\xe3x\xc7\x17\xb0\xd0b\xbeTv\xf1\r\xb5G\xf7\xfc\x9bW\xe9\xea1\xbaj\xd4\xee\xdd\x81u\xa8\xdf\xf1Z\xab\xaf\x96k\x0b\xb7\xb4\xb79\x16#\xb3Pv\x8a9\xe3\xd3=\t\xd7(\x98e{\x00V\xe3\xc0R\x0c\xb7d\xa6\xedB\x1f\xbe\xc2\xbf\xf2\xc8\x9b|\t\xf7 ~\xbc+\xb7`-\xde\x9b\xe7\xfe\xedM\x7fn\xad\xf6p\xd6\x9b\x9a\xd0\x9e\x9b7\xcb\xfd\xf8\xe4\xe3V\xda\xc2\xa79\x16\x10\x92k\xe6K9xZ\x8b[\x06\xfa\xb1\x06\xacB\x19p\x05\xe0\xd2\xb6I\xec\xff\xc5\xea\xed)\xa6\xcfXec\xff\xb1n\xda\xaf\xad\x9b\xf1_\x02H\x0f\xfck/\xabK\xb7\xbc\xd5\x87\xfe\x95;\xfe\xf2=\xff\xe9\x03\x00g{}z{\xe3\xa5\x06\xce\xa0\xd5\xeb\xd2\xcf\x10\xa6\x9e\xfe\xe2 \x10\xb7\x0b\x90\xe4,#\x87\xe0\x01&\xfb\x1c\xcbu\x8d^\xd3\x01\xec\x02|f\xc6(f\x01\xec\x92\x05;\x9cA\x9a*\x94\xe1\xde\x82E\xe8\xde~\xf3\xb6zm\xd9\x9b_\xd9^\x1b\xad\x8f\xfe\xe0\xad?\xf4\xef~\x87t\xbbq\xaf>1\x83\x14\x05\xf43\xb1\x0e\x88\xaa>\x9d\xf2\xb6\xc6k\xcb\x97\xfc\xb1U\x04\x88\x97\xfbe&\xdfjV\x80^\x05|x\x81@\xb3rV\xbaL\xe8\xc8X\xae\x8d\x1b_6\xdd3\xc6\xa0]\xee/\x02\x11\x17\x07,g\xd0\xb1\xcb\x00:\x80\x99\x01\xe8\xe0?\x00\x98F\t\xf5\xb15\xff\x9b\t\x9d\xdc\x00\x8a\xed\x8d\roz\x01Ymm\xa6\xf6\xf0\xa2\xb7\xfa\xc4\x1b\x7f\xe9\xad\x9c\xe7\xebD\'\xf4\x04\x84\xf0\xc4\xfe\x9e?\x1b\x9dF_\xae\xd8k\xe6\xdcdSk\x9fUN&\xf0j\xa2\xd9H\x94;\x06\x12M\x86\x01\xbb\xb2\xb4\\\xfd\xfe{\xa3\xc3\xa8\xae\x8e\xf2;\xdfm\xde\xa0\xaf\xe1.\x9b\xff/\x9b\xcb\xf2\x1f\x0e\xfc\x1f\x93\x03\xe0\x15W\xe7\xd2B\xc5\xf2\x80\xe8J\x8e5`\x17+nn\xc8\xb0\xceZ\xe9\n~\x95.\x16\xb2v_\x05X\xcc.\x16\x8c\xb4\x95\xcb\x11\xa5\xaf\x8c\x02\xa2\xfd)|)\x83\xef\xbd\xfa\xc1\xbb\xf5\xaav\xe7r}\xec\n\xb2\xee\xf2So\xe5\x86q\x00\xee\xc7\xe5\x98\xaek\x01\x86iU\xc071\xcbJ\x1d8z\xe4P\xf7\xe1\x1e\x00tx\xa4\xa9\xd9\xc8&jO\xa6\xbd\xf1I~\xa81\x8c\xf7\xb4VJ%\xcbI6\x8d\x10\xf1\xcf\x9eG\x9a\x0f\xbdG,\xcf-\xe6\x80\x9c\xe2V\xa8o\x97k\x95q\x0bi\xbf`1\xde\xd6u}=H5$\x1b\xf8=\xb8\x08F\x05\xec\x8a\x0e\xef)\xfcp\x1a\xdf\xfbg\x94\xc7\xc5\x02\xbc\x14%\xb7\xeb"\xbe\xb29\xa0r\xb7\xdfD\x1a2\xd3N\x11\x88\xdb\xcc\xe5@4U\x1c\x02\x89_\xbe\xb8X]^a2A\xf9\xbb9\xeb\x8d?G\xca\xbe\n\x942\xba\xbd\xf1F\x97\xd4\xfe\xec\xd5\xed\xf5\xfb\xf5s\xb3\xb5M"\xe5\xcf\x8f\x1e\xec\x12 !\x10\']K\x88_\x92?n\xb9\x88\xaf\x06\x947`C\xed%\xb3Quq\xc5\x9b\x9a\xe1\xb5kHU\x12LQ\xe6\xb1\xe3G??v\x02^\xc9\xe88\x95\xe0W%N\xeb\xafWoJ\x9b\x85\x01\xd3\r\xe3\xfa\xc1ux_\x04\xd7\xcc\xaa\xde\xb3uo\x95p\xfd\x97\xee\x83\'>m6>\xed\xea>\xfc\xa9\xfe\xb2A;S\xeeO\x9cnV\x17\xfa-\xbb\xaf?\xf2v\\^\x8b[\xb2\xd2v\xd6N\x1b\x19I\xbc\xa84\xa5\xf4\x83\xdfW\n\xe5\x1d\xc1\xa9>\xfa\x1a\x89krB\x17h\x08WO\x17\xec\xfb\xc1\x9ef\xa3\xe7D\xd7\xb1\x1e\r0\xd7\x82\xbf2\xae\x0e\x1a\nX\x97!\xdb_*\x01S5\xc2\xe6ZH\x18\x00\x17\xc8\x15\xd2\x8fB{\xed\x0c\xd8\xf2\x8a\xf7\xf4*\x00\xc6\x08\x0b\x14WOW\xd7\xc1f\xe3\xe8\xc9\x13\xc7N\x9eH\x1d\xd9\xffyW\x086+\xa3\x03\xc6/K\xe1\x8b\x18<\xc52!\xe0\x1c+\x0bZ\xaf\x90\xb6\xa4r\x97\x020o\x9fE\x14\xa2\xed\xc1\x82x\xeb;\xff\x9b\x1b:\xa4B4?\x9b\xf1\'\xa6t5\x0f\xc2\xaf>\xf6\xd6\x1b\x9f\x01\x11\x08\xe4\xcd\xd4\xee\xbfz\x057\xe3"\x0eu\x1f\xef9\x91\xea\xfe|\xff\xe1\xaef\xe3\xb3\xfd\xf2om)Y\xdbq\xcb)\x82G_Q\xce\x0c\xae\x86\xf1\x1d\x95\x00f%c\x17A\xed\xf4\xe5A\xf9\xf4\xdb\xd9\xb2Z\x95\x95a\x91\x8e\xbf\xeac\xab\x02\xa9\x86\x0c\xae\x80O\xfc\xa7\xf7\xd5f\xd4o\xfd\x08:\xc5\xe8\xa1\x87\xf5\xe0\xc3X\xb8{o\xc7\xb7\xdf\xae\xf8\xd7^3O\x1b\x7f):gP\x12\xe0\x12\xf7\x9f<\xd8}4\xd5\xf3i\xf7!\x9d\xac\t\xaa\x14\xc1\xf3s\x0b8|\xf8\xe4!\xe3\xcb\x8aY(\xdb\x7fc\x08\xd1\xd0!;\x10\x1f\xe7\xda@v\x85\xf4\x90\x01\x06a\xfa\x8c\x1b\x03\xb8\x7fy\x02\xb8\x1b\x99\x9a\x1eU\x9f\x98\xf5.\xcf\x83\x05\x01;\x81\xe0o<\xa8\xde]\x07c\x90-A\xff\xf1]\xff\xd6\x03\x10C\xb5\x89\x17\x08\xfe\xbf\x9c\xdc\x7fD\x07\x9c\x00\x91D\xf4WT\x9b\x95\x82\x0bb\x1a\xf4)\x80\x1c\x02\x93\xad\xb2^\x0b\xc0\xb5\x0c\xb3\\\xb6@j \xbe\x03\\\x93\x95\xc6\xc6\xd9\xd4\x0c\xc8~\xff\xda5\x7f\xf21 \xd4\xbf\xb6\xea\xcf\xbe@\x8ck\xc0\x02\xe1W\x97\x9fk\xda\x85a\x03YG\x7f\xa4z\xba\xffgWO\xeap\xf7\x9fP\x99\xe8\x0f\xe1\xefAQ\x0e\xd3\x1f#\xa45>+\x82RT\x14\xdfR,\x00\xfaK&\x9a_e\xcb\tH\xdf\xccfa\x91\x086\x99f-\xbd\xa6+\xb6\xc6\r\xf4"Y\x80\xde\xb3Id\x01\xd2\x1e\x82,\xde<\xaf\xdfz\x0bF\x1b\xd8\x05\xb8\x96\x87\xf3\xde\xb3-a\xbd\x11#\xe0Z\x8ew\x1db\xa2\x0f\x84\x0b+I\x80\x8d)\xdc\x05\x15y\xeatSH\x93\x07\xac\xea\xda\x7f\x83\xff\x94\xd1$\xee\x1b2h\x19\x81W0\xc8\x16+\x98\x85\xac\xf4\x1e\xfbKStI\xf0\xab\xe2\xd45\xa0\xde\xad\xea\xd3\xf9\xea\xdc\x83\x10P\x84\xd4\x9d\x00K\xe1\xbb\xd1\xd0\x00\xbb3\xdd\x9fh\x8a\xd8R\x00\x1e\x10\xa9\x03\xb0\xbav\x86HT\x81)!LW\x1c\x07U3\xe230\xa3X51\x8ap\xfb\xba\xc4\x83j\x9bS\xe0\x1e\x04P\xee?p\xa2\xfb\x8b\xae\xd4\x89\xae\x7f=\x91\x12jQ\x87\xb3\x0cNQ\n\xf9\x04@T\xf7\x10\x90\x9f#\xbc\x8c\xa2J\xderP\xf4\x99\x85>\xb4\xfe\xd29R\xdd\xbd\xcc\x8a&X\xf1`A\x7fn\x17\xec\xcf\xcd\xb3\x9f\xee!\xc6\'\xbe7\n\xf0\\\xb2\xcfW\xafU\xc7\x1f\xc5\xde\x82&\xda\xc6Mo\xe5\x9e\xf7\xd5%X\x06\x90\x857\xbaY]\xbd\xe0-\xcd\x80\xf5\nF\x0e\xf3\x9aF\xd0m\xadm\xed\xc6\x1f:\r]f\xc0\xc7\xf66\xf8\x02\xd0\xac_\x06\x91S\xbf=\xb9\xbd1\xed\xcfo\xf2\xef@G\xf0\x9d\xc6\xf6\xeb\xe9\xfa\xfc7\xd2<\xea\x02\xfe\x04\x06Di\x07+;|\xec\xa4\xd1f\xe4\xad|\xd1\x192z+\x19\xc0\x94k\xf4\x9b\xa0\x0bz-\x0bT&\x88!\x1b\t\x9c\x9d:\xf0\xa4\nH\xfb\x00\xdd\x80\x9d\xa6\x05\x83\xb0\xa8=\x9e\xa9=~\xeb\xcf\xbd`\xd3O<\xb3\xb6\xfc\xad\x7f\xf1e\xfd\xc9\xb5\xfa\xed\xab\xd5\x95\xebh\x1f\x92B\x03\x1fOl\xe8\xd2\x0cH#m\xb9\x11\xab\xf0\xd3=)\xb1\xa3lj\x81\xc5\xdb\xd9\t\x9eO\xea`WO\xf7\xf1\xae\x83\xe22 \x82_\xa8\xdeS[\xb9\xe4/\xbe\x00\x9e\x03\x0b\x12\x1c\x01\x7f\xfe\x19\x1b\x91h)\xef\xc5m\xf0\xe6ny\xd3s\xd2\x9d\x91\x889\xe6\x903\xa4\xcb~0\x11\xec\x1cSf\x7f\xd1\x05|\x84\x04\xda \x186\x99\xe2`!\x07\xa2C\xa8q\xc7\xeaC\xf9\x0b4C\x0c\x06\xf6\xc3\xecbHfm]\x06\xb8x\xb3\xd0\xb8]\xb9\x0c\xc2\x0e\x9cl e\xdc/\xe9`\xb1\xec\xad-ox\x17/!\x82`\xd7\xce\x96\x90\x8fL`\xf9l\xc7\x17\xfb\x814\x0bE\x1b\xb0\xc6\x1c>\xd8o\x15\xc2r\xa0\x858\x92\r=\xdbUJ$`{`-)\xa9\xfc\x85s>\x98~$\x02\xe4\xf3\xbd+\x1b\xde\xc2c\x86\x03!\xe8\xfa\xd7c]\x07N\x00\xd2\xbf\xd8\x7f\xbc\x9bU\x002~\xc7\x80\x99*\x01U\x80}a\xd8Y\x83\xb9\xae\x93]\r\xc3\xca\x81=\x06\xee\x87v\x13\x0b\xad/+\xe0T\x19\xbd\xc523\x1d\xeb\xae\xac\x99\xb7a)\x88E\xbc\x18\xc24\x90_^\xf1\xa8\x14\x0eh?,\x9cc\xfac@\xc18\xaa\xdf\xb9\xe9\xcd^\x8a\xe8469\xea\x97Wk\x8f\xef\xfb?\xd02\x0f}\x86\xab\xf4\xbf\x87[gv%A\\\xd2g\xa9\x03\x9f\x1e\xed>\xd0\xc5\x14\x98\x8c\xe2\xa2\x99\x15J\x13\xc1\xbe#\x11\xd3\x83~\xe6\t\xcdF\xf2=\x03\xfee\x13\xcc",\xff\x0c\xa1\xa7P\x8b\xbc\xb9\xe8=\xbb\xe1/=\xd6)\xc7\x9f\x9cg^\x02\x0e\x00\x19\x02\xcb\xd1]\'\xe5\x1f\x02\x81)\x0e \x9ao\xd2\xc4\x01"vP\x98)\xca\n\x05_e\xd0\x1cr\x19\xf7\xa0\xbbu\x11\xad\xefO \x07\x94\xa9c\xb0e\x8a\xef\xd4V\x12\xd1\xe0\x81Y!\xd0\x7f\xb0[\xda\xae\x9d\x80\x83<\x08\xd1\xbcy6\xd5\xbf\'5\x1c\xc5\xd8H\x8b@Jk__%\x8bJG\xda+\xe2\xb5\xc2H#\x8ax\xb79\xcb\xb6\x01(a~\xa7\x12\x0e\xb5W_)\x04IA\xb0\x7f\xa0h\x83\xa7\xd0_\x1cDY\x07fb\xd9\xcc!k\xb9\x95\\\x99#%f\xc1\xb0L\x07$\xa3\x03\xce^:m\xb9n\xb6\x92C\xe9\x08\xf6\x01p?\xfc&k\xda9\xc0++1\xa4\xbeP\xe8\xe9\xfes\x7f\xee:\xcb\x02\x7fi\x14\xfe\x00\xa8\x00%\xdb\x1b H\xef\xd7\xef\xacV\xef\xaf\xd7&/\x81iP\xe5x\x0e\x99\xc2l\xdb\x03\xe3\x1d\x01\'S\x04\x8c\xc0\x1c\xe6\xb8\x8du\x16\xe4\x0e\x82\xcb\xd1B!\xa0\x89\xf5A\x84\xe5\xa4\xe4\x0ev\n\xe9\x03\x8c,\x92\x84\xa8\x8d(6\xe8_Y\xae=\x11\xc4\x04\xc0\xf8O\xefykk\xf5\xb1i\xf0>\xb5]r\xc0\xea\x18J\x02\xb1&\x82P\xe2\x0f\xdbkO\xfd\xf9W\x8cWxv\xac\xdc\x95\x08\xfe\xc2\xcc\xd9\x19\x8c\xd9ELm\x14\x04\xcd\xc2um\x0e\\5\xf2\xa3\x9a\x89\xbbJE\xd7\xa6\xc0\x98t\xeaH\xe7\x92\xde\xd4\xfd\xe7\x7f\x8c\x8eI\x87uL\xf7\xe0@4\xb0g\xe7\x7f\xbb\xea]}\x08b\xd0\x9f\x9b\xd4VF\xf2\x0b\x0c\xc8\xe4n!\x14\xe6sv\x83[\xc1\x84\xb1KI\xbe\xc4.\xe1\x1fA\x15\xe1\'\x0c\xbe\xb5\xdan\x16\x88\xb8l%\x85\xdf(n\xe4\x0fx\xab\xe4x\x8cv\xf0\xc6\xec\xb4\x0e~:j\t\xf9sv\xd3a\x93\xe0\x87\xef6\x17\xe5e\xd6.\x18\xed|z\x8f\xbdV\xc1\xeaR\xf0\x06H\xb4\xf3V\x01\xe3\x14\x18\x95\xb0\xfb\n\xac\xe8\xf7t4\x07\xe6.P\xb4SFo\xd0\x06\xaar\xd0M\xac\xd0V\xe5\xec\xbc\xcd\xee\xf3\xea\xf7\xf5\'\xdf\xd6\x97F\xd5\x0b\xb7\xd7\xaeU\xbf\x7f\xe8]\x9d\x84\'Q\xa0ft&0\x81g\x1fz\xf7\xe7\xeb\x0bW\xbd\xc9\x97\x1c-\xc5\xa8\xdd\xe4|}\xee\x15<\x08\x8c\'m3(\x04\xa0\xd0)"\x01\xf2#\x7f\xf9_\xf1\x15 +B\xb7\xa8\x8b\n\xb7|3\xa2N\xdc\x12A\x11@\x0c\x04\x11\x02W\xc3Z\xb1R\x90F6\x05\xd4\x01K*\xa4P)!\xc6\x00K\x9f\xeeiI\xc3F\xc1\xd5^\x90\x15\xed\x1f\x17\xde\xffP\x04oU\xa0\x01\xbcd4\r9t+n\xe1\x90-\x9a\x04\xcf&\x84_F\xf4\xe9]\xfdj{\xed\x12\x86\xd8\xe6^\x00\x98\x80L\xff\xc79\x15\xe8%W\xf98\xc8I\xf4\x11@B&?l\x86\xe7\x19\xbfc\x9aK[v.)\t\x0e.v\xec5Z\x8c\x0f\x9b\x8c\x0f\xe0\x9e&\xe3}\xf8S\xb8\xfc%i\x99\xc3\x02-\'7\x84\xbc&9\x1a\xed\xd9\x84+\x96@\x9b\xdd\x0c\xd7\x04A\xf4\xda`\xb8\x83\xbd\x08\x17\xac\xdeb\xf1L\x88\x1b\xebc?y+\x93A\x18\x9f\xac\xde\xea\xd2-\xa4S\xc0\xae\\\x03,\x0f\xc8@\xa7\n\x11u:"\x1f\xaa\xc20|\xabF\x19b\xed`\n\xef\xf9\xa8\xad\x8db\x85\x0f\xc7\xfc\xe7\xe7\x8ca\xfef\x84\xe1vQ\x1a\xbd\x1c\xf7&\xd6\xa5=N\xc8\'\x80\xf8\x99\xf4{`\x1dd\xbe\xb5\x95\xea\xad\xa7\x92\xb7\xa4\xa4R\xd9\x05\x90\xe8@\xfd\xfd@#-\xb0z\x8e\xdb\x04\xa6WK;JZE\x1f\x81\x95%\xd6\x04\xdf\xb3\x91\xc5I\x07\x90\x8cH\x02\xe2Y\xf5\x1b\xcb\xfe\xdfoq`GOqXi\x07\x8cq=<C\xc1E|s\t\xdd(\xd06q\x11irc$\x0eE\xdc\x8d=\x93Y\xa5\x80\xfc\xe7\xeb\x00\x16(\x1dot\x9a_\xcc\xb6\x85\nL\x87v\x01<V\xef\xd14/\x85\xbc\xc1\x93GR\x18n\x02\xe2\x13@\xb6\x82\xbf\x94\xe9\xb5@\xef\';~\xf7\xbb=\x1dMh\x0c\xf2-\x9d\xb8x\xb2\x04\xedB9\x89\xd7B\xb2\x08\xb4\xab\x8a\x16\xd0\xd2D|\xa0\xb7b\xe72r9vAz\xb7\xd2D!\x0c/-\x0bt\x92\xd6\xd2\x83\xa4\xa0\xd9\x90\xa7\xc6\'\x85j\xa0%\xb2\xab\xa5\xbbWHAj1\x7f0\x08t\x10\xf2\xf41\xecP\x19\xb5\x89\x1f\x8d\xbd\x1d\x9f\xec\xfd\xe4\xa3\x8f;>\xf9\x10\xf5\xf6$\x12,\xf0&\xacOR\x0b\xe544G\x91\x16\xe4\x82|@\x7f2\x03v\x03\xeeY\x1aX\xa8\x17\x85\t\xe8\xb9L%\rw\xe4\x80\xc0(\'\xc6\x9a\xdf[}X\xbf\xb1(\xa2\xb22\xe2\x87o\x9c\x7f\xbd\xbd\xf5\x16\x9c9\xb0\x17@\xbdVg\xb7\x04\xe5p~\xe1\xa7\xab\xfeM\xd2\xd2\x94%If\x13\xba\xc5A\xa0`\xc4C.w\x84\xdd\xf3?!\x9a\x01>\x91<\x93L+\xa2\x91%\xb0\xf0\xed\xb3Ar\xa8\x82\x0e\x87\x1e\xa2\xf4o\xbf\xc6\xc0\x1f\t.\xb6\xbcxG\x80\x8e8\x03\x83d.\x1f*B\xbcS3\xd5\x97d\xf3\x89($0L\x1e(\xe9\x18\x88\xae\xa4\x16\xbelj\xc5/\xd8;?\x03\xf6\x8d\t\xd6?\xa2O\x8f\xe0\xa3\xbf\xc6\x0c(\xe3\xcb \x9cP\x9c\xa5\xfb\xd1\x7fg>\x9c\xf9\x11P\xc9\xc4[]\x04\xc1\x8aF\xd7\xf6\xeb[(\xfc5\x8e\xe0\xec\xa7\xca\x06\xfaw\xc8,\x80\x17R\x94\xd4@stX\x03x$\xc6"M\t\x8b45\x8c\xd9\xc3V\xfc\xcf^p\x03\xfa\xad\xb3\xa7\xf6\xfd\xfe4G\x99\x0e8\x16K\x93\x0c\x10\x7f\x9a\xf3g\xca\xec\xae`2\xcc(f\x85\xb6\x05Y\x85R\x18\xf9\xc6r\xb2fZOzi\x82lc\xba\xfe\xdd\xdd\xea\xe2\x0b\xff\xca\x03F>=T\x0f9r\xca\xb8\xd3\x18fm\xd8\x9e\xd8g\x0c\'\xd29\xe0\x82Ty\xa8d\xc1\xc7\xc4\x81\x9c]\x02K\xe3\xaf\x18\x05\xb3\x1c\xb2|0_\xeb\xe2\xad\xef\xd1\xcf\xf0\xdfo\x8d\x83VY*\x8d\x7f\x19\xb4\n{Z\xbe\xf8\x0cP\x9e\xee\xb7\xf12\xfa\x11\x8aV(\xfcX*b\xc2\x0esl\xa0\x99l0LsC\xad\xda\xd3\x04\xd4[7\xd9\x0f\x03\xbc\xd7&\x1e\x83=Z\xdbZ\xae=\xfd6x\x83\x7f\xf3\xa5\xff\x88\xdcP\xf9[\x00\xdf.q\x00\x1b\xa0\xff\x12o\x1c\xc8\xa5\xf6\xf6\xa6\xb2\xa5\xdf\xa7\\x\x11l\xbekf\x01\xda\x82[t\\\xca\x99\x89\xa5"4\x89\xe6\xe0I\xc0\x86\x18\xb7\x04\xed\x85\xdf\xe6\xf3\xfd{Z\xf6\xf6\xb6H\x84\xb4\x0c\xecimo\xc9\xe7J\xe1\xc7\x05?\xcfXhh\xc3O)\xa4t\xf8\xd8I\x8c\x7f\xa1\xf2\xdcgD\xe3Q##\xfc\xbbDG\xcc\x0e\x9c,Xe\xc6>Fe\xc3;\x90\x00\xdf\xb9,W+}$\xf5\xb4=1O\x8b\x89?E6U-\x80\xbco\xb8t\n\xc0j6\xdaN\xc3m\x14\x87N\r\xd8\x19\xab\x08_\xb4wP\xb4\x89/R\xa0\x1a\xd7\x16\xc4\x9e\x14 {c\x00\xf9b\x7fW\x1cM%\x06LKm\x9f\xe6\xe7\xd1;S\xf8e\xb6\xd4\xfeQ\x08\xe7\xea5\x1f\xfe\x9f\xbf\x86\xc3\xed\xfc\x9a=\x1d\xf1\xaf\xf9h7\xb4vct\xe5D\xf1\x0bB\xd1\x0e\x88E\x12%\xbc\xb6K\xbc\xc2\x0b\xe9\xc2^yA\xa4\xce\xf6\toB#+\xces\xed3D\x1eL\xa6\xb9\xf6\t\x0b\x16\xae\x80\x14\xee\xa3[\x84\xcd#\xe1\xfe8\x06\xee?\x99\xae\x9d>\x0c\x02\xa9\x81\xb7c\xc8`\x8f\x84\x0e\x13Z62\x06\x88%\xfa\xe6#\xfaF\xbd\xe9\xf71o\xfas\x0fzm\x96\xc3q\xdf\xc8n\xb8\xfc\x9d\xda\x11\xb0\xa8Ryp\xaa1dV\n0\xff\xc9N+\xe8\xc1"\x97J\xeeW-\xc2U\xbf\x81\xa7\xb86\x02\x80\xbf\xe6\xdc\xdc>\xf6\xd9\x9a\x91\x8d)\x9e\x86\xe4\xde\xda\xa6 io\x8b\x01\xe58\xe9\xf2#t{x}\xf4\x88\x14e\xdb\xf6)\xc3"xX\x9c\xe0\x15\xe8:PqAF\xee\xcf\x0c\x98`\xe5dvZ\x9e\x04\xf1\x14\x02&\xd6\xd7\xc7\xbb\x8a\x17?\xe6k\xc1\xfd\x02\xdf\xf4\xe5\xef\x15BP"\xb8t\xed\x13y\r-\x90\x82L\xa0\xc9\x9dn\x0fv\xba\xbd#\x9e\xe7\x0eZi\x8e\xa77\xee2\xbf\xa1}\x07\xda\x0f\x9e\xbcg\xb7\'\xef\'i\xf3k\x1e\xffa\xe4\xf1q2\x89\x95\xf0\xae\xbc+\x12-\xf4\x0e%\x16\xa5\xec;\x85P\x8bkY\xa2\xa2\x8e\xbd:\xd1|\x18\xbb\xcf\x03\xbb\xbfQJ\xdbS\x08\xb3|\xb80\xb5Rl\x88\xc1\xb7\xd2 \xd16\x19\xcceP\xaeJ\xb3!\x18#2\x19\xce\xc1\xdc\x04\xc6\x7f1\x97UFo4\xe0i6/\xd0s6\xcbe\xb4\xaf\x08\x86\x0f8C\n{@\xf6\x06\x05\x881\x93\xb9,\x8c\xa3\xbb\xdfmo\xbcd\xdb\x83\xbc\xb7\x87\xf5\xd1{\xa8\xc5)X\x85\xb1WN\x8a\xc2\xef\xc9\x99\x89\x04\x85\xf7\xbd\xc7\xaa\xff\x00\x9a\x07`(\x98e\x99w\xe9\xc5`vV\xcbd\x95\xb8h\xca5\xf2\x16&%\xe86\xae\x8a\x12\xce\xa9\x88A\xf0\x03\xd9\xd2\xd7s\xcd\xde\xe6O\xe0<r(\x82+\xa5\x18j\x8e>HkB\xf8\x02\xb6\x8b\x05G\xc8}\xc9 \x03\xd7l$\xcb\x15 \xb6fx\x95[n\xe2\xe0\red`_\xb4\xfb\x9a\xf0\xd2\'\xb0a\x88\xac\xfa\xd2(x\xa3F\xfb\xbb77?1\x00\x06\x1d$v\x14\x18\xe0\x98t\xa9\x96\xc5\x03\x13\x86\x8c\xc1R\x99\xc23\x18m\x1cRK\x9d\xc5\xfc(\'G\xab\x17\xee\xf8s\x93\\\x0e\xc3\xaf\xe1\x1a\x8a\xfa\xd8Z\xfd\xf6xd\x91\x914\x1e\x05\xba8WGI\xbb\xb3\x89&^C*r#\xf8\xb2\xb5\xf3o0D\xc2Qx\xf0}0\xec\xa0\xaf\xa6\xebl\xd91\x85e\xd8.\xb2\xa2\xb2\xea\x8d7\x17\x16q\xb6\x94\x03\xd3\xb6\x0c^s\xc5\r\xaaRX\t\xca\xc5\x89\x0c\xaa\xa8\x9f\xdb^{\x1aq\x87\xd4S\xb9\x8a\xee\xdd\xe6$V\xa6\xb5\x1b\xf5\xf9ipR\xdfmN\xc95\x0b\xf2I\xc9\xb7\x83\xfd\x8b~\x11\x7fl\xa2\x94\xa3\xa0:\x80\xc3\xb1Z\xb3\xe0Y\x80\x81\x9at\x12\x7f\x10\xaf\xf8w\xf7\xfd\xe4\xbfg\xdeo\xfa#\xa0G(h#\x9b3\xfb\xdcN\xb8\xbd\xfb\xf0\x91\xa3\xc7\xbb\x0e\xec\xef\xe9j\x1a\t\xb64[\xc1\x12\x94"\x97\x00a\xbaF\xd4\xe1\xc8\x9d\xa5\x80r\xc10{]\xf4_H\xca\xa8\x85O?\x90\xabF\x87\n\x03@3\xde\xd3o\xb1\xc2\x0bv7\\=\xc1\xf5\x9c\x91\xbdg\xce\x8cl9\xae\x88(V\xac5\x86v\xc3\x98\x88\xa0M\xa5$8-\x81\xc4!\n\x80\xbc\xcd9\xd8\x91\xed\xf5\x8b:\x9c\x86\xc4\x9dq\xe4\x8f\xe8n\xbaD\xd8\xc9\xe8CG\xdem.r^@\xe4\xa0\x87#@\x8d \xe7\x10}\xe1{\x9b$\x82\xb5\xc0\xb1\r\xf4\x04\x8f3\xcc>\x139\x97(\x89\xaa\x08\xf5\x08\x16;\x87\xa5\xa2\xebbXNb\xba\xbax\xc7\x9b]\x05\x8a\x96\x85\x11\xf70\x94\xc0\xa1j\xcd_\x15\xd2\xe2\xef\xd7kO\xa7\xfc\xa7\xf7\xc3\x1cE\xeeM*\xdd_\x04\xe3?\x19\xd8\x85\xc7%\x07k\xb6aC~<\xcc]M\x01\xf5TD\xfdR\x01EzNz\xe1z\x94V\xaf`\nj\xaa\x84X\xa0\xccVmr\x82+\x0cD\xbe\x9c\\^,\xb0\x92\xd1VU\xd4\x14\x04t\xf0\xf7\xe4$\xa2\xe2?\x8d\x9c\xb2\x93\xdd\x1b\xb7\xbe\x7f\xd2\xf6U6\xb8\xa6\xb7\xc3nY\xc84\xfe\'Lb\xedi\x91=\xd8\x17\xd9\x83\x11\x8d\x859\x81\xcb\x01\x07A\xb7RR\x15\x1d,; \xd9%*\x02\x04\x94n\x83\xcc\xc3\na\xb91\xdb[\xdfI\xce\xd6\x05X;F\xf7\xc5\xdf\x1dF\xfd\xf6Eo}\xb6\xba\xb0\x81y\xf3\xd5\xd9F\xc5\xc1\x89A\xe0U^\x07\xc6A\xce6\x8b\x0fX\xd1\x81@ZT\xbf\x00\x0c\xa2\xf3\xd2>\xcd\xe3\xee)\x17K2\xc4\xa6%^@KS\xa9\x11W~\x854Q\xa6\x08\x1a\x18#\xbf\x94a\xd2\xbdwUG\xa4\x03\xaa\x84\x81\xbfp\x8eCp\xde\xd8\x92\xff\xf4nPB.\x1f\xe0\x16+N\xda\x92\x91\x9f`\x1dMz\xbcA3\x13Lc\x00\xb9_$\xb8\xd04\xa2B\x16\xca\xc2\xc36\xa0\xed\x92\t\x05\x178\xcd\xa5\xd7\x98\xbfZ\xf5\xb6\xbe\xe2\xa4\x0f\x86\xfc)\xcc\xc3\xd0\xea\x80\t\xc9\xc9\xf0\xb5\xdan\n\xdf\x95\xa4\x92T2\x86\xa6\xb6p\x91\x93\xcf0\xad5\xacm\x86\xf1\xbe\xd1>\x12\xd5\xf8,\x04\xf1I#\t}a\xfb\xd3i\xabT\x96\x9c\xce\xc9z\xb6\xe4\\\x0e\xc7\x83?P$2\x13\xe2\x08h\x0c=\\r<E\xb5I\xf044i\xb0`SH\xae\xe0F\x94s\x14L\xc1mZ[\xab=\x9cf\xb08\xff\xba\xf3\xaa\xddJ\x16\x0c\xce\xd6\\q\x10\xebm\xd9Rh-\x81\x1b\x08\\\xd6\xfa\xd7\x92\xfc\x7f\x8b\xff\x18\xb4zK\xf4Go\xbe\x94h\x8a\x96[\xe9\xe8\x90)y\x89\x14\xf1\xa2\x08nr\xb8\xc7e\xccOaN3-\xa1b\x91\xde}\x90K\xfb\x1b\r\xc5\xd0\xeeSx/\xb2\x19\xde\xe4\xc5\xfa\xf8\x0c\xdb\xb1TsJ\xf5C$\xe6\xe1\xb1:6\xf0M) \xb6N\xacQJ\xb6\x7f\x04{\xab\xedt\x98D\xb1\x07\x02ID4?\x04\xfb%[ \xb8\xf5\x81#~Xf\x83Q{\x19q\xa5\xa28\x1d\xf0\x9f\xef\x8bx4\x86\x89\x85pk\x04[\xe2\xac\xcfT\xf05(\x0e\x95\x8f\xcfX\x88P\xe6\xf9N\xe3\xc0\xd1\xcf\x0f\xfd\x9b\xf1\x81\x90\xe0\t\xf8+\x8b6z\xaa\x81\xa8w\x88h\x0e\xc7\x12Kx/E\xc0\x93\xa2\xf8D\x91Qd\x80\xe8)\x80\xc3\x1c\xe6]\xb0\x85\xd4\xd2\x94D\xf1`\x03)\xc0\xb9\xc3\x8aZ\xc1\xc5\x01\xab\xaa5\x7f\x06\x9e\x9e\xe4\x0fn\xe7\t\xa7\x02\xf6;\t\xafT\xf1\x0c}\x0cm\xdf1 \x1c\xcb\x11\xe5\xe2\xd6Y\xb4a\x19\xd0\xde\xa1\xb2\xe5\x06\x99P\x1b\x9b32\x15\xb2\xe0`\xef\ne\x8cAg\x0c\xb0z\x0b.2nh\x139\x9a,\x88\x8e\xb2\'\xde\xf8#o\xe55\xa7\xbb\xeawf\xbc\xfb\xf3\\!\x1a\x16\xef$\x17\xa9\xb5\xa55\rd\xd5\x91d\x0c7\xeb\xcb\x8c\x10_\xa1\xc0uA\xe8\xbcdL\'\x13\x91\x12Z\x17\x8c\xe4\x97\x08\x9b\xd4G\x17\x90M\x18V\xce\xa7,/\xf8\xeb\xf3\xda\x83\x98Et \xd9Z\x10\\\x12g1\xa8\x1fG\xbcw\x19b\xd0\xf7\ry@j`!\x01\xc0\x93\xeds\xb0t\x9e\xdbL\x80\x14\xc1\xce\xe2\xe8\xfe\xdf,\xa7\xc8\xca\xb6\x196"g\x9f\xe1\xadSZU\x9a\xdc\xa0\x81[\xe3\x1e\xc9\x9cS}q\x17\x0cXt\x1d\xda\x94\xebp\x193\xc4q\x8a\xda\xe0j\xe5F\x04\xa0\xb9tJ.\xef\xf4\xa9\xacV\xa4\xd9\x1a\x98\x1c:?\x8d\x90}%1\x876\x0fU\x9c\xb0\xff\r*\xcerX\x01\xb3\x96\x13E\\"\xc3\xcc\x1f[B.<#H4T\t\x95\'\x8dq\xae\xe9\x10\x82\x17<t\x10"\xf0\x10\xf1\x99\x9c\x05\xc6\x05u\xc4\x88\x1ab\xdd\xa2h6\x14\x98\xb4DN\x85\x80.8EhH\x8a\xea\x88P\xd1v\xa2\xfd#*\x98\xa02m\xca\xc2\x82B\x10\xb7\xcb*\x8a_\xfd\x83\xa0\x12\x1c\xef\xc70\x17\x17|\xab\xdbO\xcb\xb0\xc2QL\xc9\x8ah\x86(Zw8\xd3\xa8\x17q\xa2\x9b\x90W*\x94s\xb5\x80\x1f\xa0\n\xad\xd2\x94\xfdx\xe5{\xc9\xcd\xd7\x02\x1a\xf8\x08\xdd\xb8R9\xcdBP\x8e#4;Z+\xb2\xc8U\x15t\xe7#Z\\\x94\xb2Q\x19\xbd\xaav\x15\xdbE\xb6\n\xdc\x00\xdep\xa3\xc5\xf2\x8bL\xa9\x9dM?S\xd3\xa5\x04)H\x0bj`)\xf4E\x84r\xc4\xc6C\x03\x0f(MVlc\xb3\xcc\xcc\x8fl\xef\xfdB\x83J\x19S\xbf\xc4f"M\xcb\xf9`\xd4!\xa2&U5\x18\x12\xa6\x95b\x89\xea\xd4\x1d\xb4\xa9\x90\x06\x8b+\xf5\xcb\x17~\x9d\xa6\x1c\x16\xcc\xf1\x7fI;\x96M\xa7\x8f$]\xd0&\xc9\xdek\x8cv|8\xcf\xc4\xc9\xe2Z,\xad\xb1W\xf2\xff\x8d\x8e\x8ct\x85(\x05I9W"\xf9r\x8c*d%\xc8JFW\x85T\xf8\xcc\xaa\xf0\x9fV\x82\xcag\x0b\xac\x0c*\xc9u\xa4\xdf\x86\tQ\xd1\xc6\x11G\x14\x9a\x19\xe8]\xba\r\x7f7dP\xff?\xa9=\xa9\xcd)\x96\xc1E\xc5J\x03\x80\xe4\xd6\x05\x7ft%\xaf~\xe0e\xf0\x92\xea\xa3\xf7\x84\xb4\x8f\xca\xf9]\xf5W \xe5\xe3\x94\xd4q\xab\x05+\xff\x80\x00r\xd6\x80Y\x00#\xce)\x96\xb0\xfe\x19\x9b:\xd2&\x06/\xb8\x92\x17\x83\xa5.8\xcb\\/\xa05Q\xe4\xcd!&\x13K\xd4I\x82\xd1\xe2\x94+%Q\xcd\xc1\xadmX,J\xc5\x83,\x0e\xb1\x03y\x9a\xaa\x95\xe7WT\xfc\x8b\xcb\x16\xbd\xad\xcb\\\xea-J?\'E\tQ}\xf46\xd5\xc6\xce`\xf0c\x85\xe8\xea\x08\x88\xecT\xf7\x91CGa]\xa2L\xc7\xa5\x02]\xe0\xe3?\xed\xef\xe9\x1a\xf9\xa0\xd8\x8b\xe1XP\xd2Y\x0cj\x94\xed\xbc\x05D\xdd\xf9I[S+6;\'\x9b\x1a\xea\x16qq\xd8\x8es\xec$\x18\x08\x1c\x08\n\x05\xa3D\r\x88\xecz\x90\r\x02\\\x05V\x9d{\x89\x02\xf3\xd5r\xf5\xee\xa6\xb7\xb0,\x8a\xa4n\\\xe0 \x13\xc7\xb2\xc5\xa3\x830M8\xe6\xd4\x98\xb2\x17\xa9\xe8f\x95\x8an\x04ZB\xc3f\xab\xc5:?\xa8\xf6s\xddJ\x1eYX\xee\xac\x01\x1bg\xc2\x8fMT\t\xa2\x18\xb8\x84\xc2\x80+\xdaDll~\xc2{z]\xad\x85\xe9\x90\xf5\x97\xaa*\xf2\xc6&\xb0ZF\x94^\x8db\xb9\xc8\xd2\x94\xb7\x88UHb\xc9k\xe3\xb0\xc1\xbfd\x95\xa2\x83#\x9a\\\xe7\xb5\xa2\xf9\xc4:\x02h_\xa4\xbfB\xbbB!\x0e\xc0\xc8\xdf\x80\xb7\xc8>\x96\x9d\'\xa8\x16e\xf2R\x86\xdd\x89./<\xf6\xc6\'U\x83$\x87\xce`\x99L\x91\x8c\x81w\x9b\x8b\xde\xdbq\xacA[8\x07\x8a\x0c\xf6\xd7\xbf\xbd\xe6\xbd=\x1f<0\x08\x1e\x86\x97\xd7\x98.\r\xe5H\x9b\xa39\xd2f#\xf8M\xc3\xddMQ\x1c\xa8\xdc\'\xa8Z3GA\xc3^\x13\xe3)\xc5`\xcd2G\xe8\xca6U\xf5#\x8a&i\xdb|\xe5k\xac\xacZ\x00\xad\xbf\xc8\xea\x07V+"\xa8\xb4Z\x85\x1a\xe4\xe3\xad\xc5\x80\x9aw\xd8\xda\xc6\x94\xae\x1b\xfa s\xb5\x91\x15\xab{x\xb9\xb2\x08\x9a\xb7\x8c"C`IR\xa4\x14mD\xd9AOf\xb2\xcb:\xae\xd1\x12c)\xab\xfa\x19A\x16\x89\x86(\xee\xd7g\x03\x87\x12\\\x1c\xd6\x12\x95\xba\xcc\xb5d\x94q\xb4Co\xe9\nd\xae4\x19)\x8a\x12\xaa(n\xda\xa7\xc2\x8e\xa2\x91\xcb\x1c0\xed\x9c\x895\xa4jA!{\x9feuPf#;r\x88\x963\x16U\x1bS!\xa6\x9eM\xe1\xd86\xe6R\xb8\xfaRK\xe1\xf1.yw\x9e\x08\xfb\xe1\xe1,\xdc\xa0\xa2\x8d\x88\xbb\x94xe\xa7\xa1\x84\xe8\xa9\x1d\n!Ns\xcf\x03\x1bI\xd4\xc7\xcd\x17\xe4R\xf8Z\x10jes\x19\x97\xc1\x9bF\xefk6J\xb9\n\xa7\xfb\xb4\r\xa4ZK@\xe0\xa1\xcf\x0eu|\x116\xdea\x8fh\x08\x02\x9a\x16T\xb4\'"O\xdf\x7f\xcfU\xf5H\xaf\r\xadv\xf5s\x17\x04\x11k\xf5\xf7r\xdd\x165\xdbX\x18\xfeI\x86<\x93\x88\xe3\x11jy\xe1\x1d\xe5R\xc7\xc8\xaf\x9a"9?\x13\xcb\xd8\xb4rabD\xae\xf5B:\xcf\x83xe\xcb\xb3\xc1\\W\x8b~x\xce\xbf\xb9\xc4|\'\xa5\xaa\x88\xb8\x81?\xbf\xbd\xbe\xe8\x7f\x7fa\xfb\xcd\x8cn\xa0c\xe1 uU\x86\rt-O\x04\x8eg\x8e\xb2r\xa1}GfQ\xdfH\xcc\x90\xf1\x1eK\x04Fus\xdd{\xf6\x951,o\x1d1\x14\'$\xc2]\x0b"\xf7+IXt\x1e\x11\x1eDN\xd4\x05c\xd2\xa5\xfa1r\xdd\x853E\x8dBA\'\x03\x0bfJT3*\x18\x0f\xde\xec\xaa\xff\xcd\x0bo|<\x12\x82l\xe4\xcbhnZ1\xa3\xecu\x11\xf9f\xe5IR\xc3\xb6dOFS\x98\xe1\x18$\xc5a@\x96\xc8a\x93\x13:\xab\xc9\r\xa08\xc1n,\xd6\x90sid3I_\x87\xc1\xe9\x13v0w\xdb\x16\xb3\x01\xf6\xd8\xda\x8f\xc0\xcb\x83\x0f\x14\xae\x84;t\xe3\x82\xbf:\xc3$\x14t\xd9J`\x89\x18\\*\xfe-\'\x03\xe8e3(\xe3\x88\xe1j2\xfe#\xf6\xae\xb0@\x88\xb2\xc7\x00\xd8\x8a\x94\xee"\x12\xc9\x14E\xfa\xa1Tt\x1b\x1b\xdc\x02\xda\x08/\x8b\xf8Cl\xc0\xd4$6\xdc\x12Y6\x12\x02\xd5\xde?\xe7\x94z,g\xe8\x8d\xb7\x9c*V(\x00\xce0\x0bCI\xaa\x86%S\xd6EK*\x19\x1f\x05J\x88L\xab\x88\xa1\x04\x8f\t\xa5Xw\xdev\xc9X\xc1\xc3\r\xb5\x82\xda\x93i4\x83\xe7\x9f\x85z\x1aE\xea4d\x02\x9a\xc4\xeab\x8a\x90\xac\xa4M\x0f\xe1P\x953\xaetBU\xf4\xa2 5\xa20\x01\xa6F1Y\xabz\xbc\xc0\xcd\xa8\xbe\xc2\x0c\xd0\xf6\xd6E\x7f\xfd\xaa\xae\x0f\x83 \x05-Zz\x12\xa2\xa0Uxcv\xd9\xca\xbbI\xa5\x04\xbb\xce\x02g\x17D\xad\xaf\xea\xbd\x95\x1e\xb1\xa3\xf2\xb8\xf4\x94\x96\xc0\x0f%\xd8\xc3u \x0c\n\xe3\x07;\x93\xa8\xdd\xc7\xbb<\xee]}\xc2\xbe\x11O^\t8\x94\xac6=\xcb6`\xe6*\x04*\xbe,\xf0\x91Z\xe9\xba\x02Z\x8b:\x95\x07\x8b-\x16\x0bo\xe1\x01a\xb42<\xb6)\xcd\x0e\x1e\xdb\x96\x9a\x03\xc7\x1d\x99\x8b\x0b\xde\xf8\xf9\xea\x8b\xdbl0a\xb7\x00g\xa97\xee\xd5\xee,G\xd3\xd2\x84z\xdd\xa3\x03i\xa6\x15\xb1\x10\x98\xa2n\x85\x8d!\xab\xc0\x17\xa9\x03\xb1#\x00_\x8b\x0e\xd9\xf9\xbc\x95\xb1\x81PrC\x1c\x17B\xccR\t{\x90~\x89\x8b\r\xa9\xf8\x10B\xf5\xd3U@\xaf^\x13\xa0\x13\x07\xdaN\x14%\x12\xa5\xee\xe3\xaf\xb0A\x98\x1a3\xf5\xb5\xe8Q\xa3\xb2\xc3p\x9fj;\xdd\xa4H\x07\xd5\x8fH\xdc\xa88\x8c\x11\xfbvy\x97|\xc6H\xa0\x87\xf2(\x11\x99\xdaT\r6S[\x8c\x1f\'(\x97\x92\x17\xca\xccr\x03\x8fN\xd9X\xfa\x00+A\x86dT1abRjt:`\xa1`\xba\xcc/\xe7\x13\xa1\x9flW*&\xa9\xa7\xe4\x9b\xd1F\xce\x9b\\\xd9EyF\xc5FZ\xba)NWy\xe3c\xb5\xe5\xe7j\xf2V\x9c\x06\x10\xcf\xd6U\x15s\x88\x16\x089\xbd\xb3\x86\xea.\xa4s\x95\x8c\xa5tis\xa0H\xa9\xb8a\x08\xab\xeb[d>\xd7\xe8\xb73 \x9f\xa2\xc2\xfd\xd9\x04k\xd4\x7f\x8c\x8e\t\x8d:9Q\xfdq\xa3\xfa\xd3\xf3\xfa\x8d\xe7\xb5\xeb\xb3\x02\xee\xf1\xe5\xfa\xf9\xe5\xea\x8b\x19\\\x89j\xad\'\xd2\x92Ha\x1d\xd6\xd4Z)\x00\x08\xc9\xdf%\xf1\x13/\x90\x16\x00\x8a\xa5R\x12\x8a\x0c\xb7\x87>\xb3\x19\xaf\xe99]\x9b%\x18`\xf0M\xd4\x8a\x0f\xd9\xd8\xb7\x18\x04A\x1c+_\x1c\xc0\xee!La\xe2~Pd\xda\xe4.\x10\xb5\x858\x89\xc9\xd5\x0c>\x7fj\x0b\x9b\'H \xd4\xee>\xc1\xf0\xc8\x94\xd6d\xf5h\xa3\xbep\x1f\xfb\x80\xaf\xbd\xc6\xed\x92\xd3\x91\xe4\x82+\x853\x05t\xe7y\xbda\x89\xd6d\xb4(\x84H\x981\x03;hd\x86\x00<;-\r3V\xb8n\xae\x08\xa2-:\x19\x81*o"\x96\x86*\xb7\x99{\xa9\x8f>\xd0\x15\x987=\xe7\x8fO\xf3v\xc9\x82\x01\xa4@n\x03\xf0\x9e\xcf\xd5\xe7\xa7\x83\x18\xf8\xec\xd7\xfe\xa37Z\x1c\xbe\x91\xee\xc8\x94\xdb\xd9p"\x12\x8b\xeas\xb9\xf6p$\x1f\xf7\x88\x9d\x0e\xae:\x08\xb2Nz\x95\x1e\xe1"h\x8f\x11Xv\xadrD\xbaW\xe7\x1e\xd4\xc7\xaep\x17\x1ff\x96\xda\xde\xbd\xb9\xf9{#\x94\x18\xa7\xc5\xc9\xf11o\xe0\x07\xac\xa1X66\xc6\xef\xe4\xbbZ:\x8d\xe1\x1d\xd3N#\t.\x18!{\x03\xc3n\xc9O\x82b\xb5C\xe0hr]!\xcb\xfa\x88\x08$\xe9b\xe4\x8a\xf0+\'\xe8\xd1\xd2\xf64*\xc3\x10\xf0\x8b3@\x9b\xba\x11.\xdce-1\xb0\xbd\xf6\x10\x0bG\x80\x96CbEH{\xb4\xf4\xc4\xd2@\xc4\xff\xa6A\xc4\'\x87\x1b7}\xa4\xc9\xd0\xcb\r\xa4Q\xa4\x8a\xcf\xc4\xf3\x9aF~\xc3S\x1e\xb0%\xa5q\x1a\x16\x92\x06\xa2\x00\xe3\xae\xd2\xefB\x17\xa6\xe0"i\xcb\xa2+\x11\x1d\xe0\x8cam\xf4:\x0e \x90\xe9\x0fVi\xdcu%\xe6hMNpWZ}bV\xef\xcc\x1aF\xb7#\x18f\xf6\x1f\xc60\xd54\x8d\xfc\xe7\xf5a\xaee\xa2k\xe1^F\xbcB\xc5\xea#\xdc\x0f\x8d\x9f\x1bZ\x90>\x10-H\xf0%\x95N\xf3P"cX\xef\xd6H`\x11c\xc5\xed\x97a~\xe0vP\xe7}\x85\xa0\x0b,\x18\xef\x97\xce\xd9\xe42`\xb7\x95]\x1e\xe2\xc0\x11PFdd\x0f\xf6\xd1\xde\x8f\x1d\xac\x88\xc4\x1e<\xce[\xb9\xebO\xbe\xac>Y\xad\x81v\x9e|P\xbdK\x1e7\xbf\x84\xab-\xa2\x99\x14\x04\xef\x84\x831\xab\xe0)"Db\xe7\x10"Y\xd8\x8ccn\x90\x0f\x15\x15\x8b\xbe|\xda\xb0\xd1ix[\xed\xed\x9b\xda\xeb\xfb:4\xb3\xab\x18\xbb\x1a}\x14\x9a\xceC\xd5\xa6\xa2K\x92z\t\x11\xc4A7\xdcvOQzA$\x80\xb1\x8aE\x9d\x8aE\x00\r\'\x07\x08\xfbH0\x97\x9aj\x08\x9b\xe8\x12\x8b\n\x1a\xa2\x1aN1<\x8f\x1cs\xb6\xe9T\x90\xdb\xdf\x9c\xf5\xafLy\x93\xdf\xf9K\xb7x\xbca0Q\xb1\xec\x0c\xa9Jj\x994\xe0\xa0^\xa6\x98\xae\xa0\r\nt-\xb5\x83\x82\x80\xe7\x9b\x80Ee\x99y\xc9\xc9\\\xbd\x1d\x0c\x05]\xf9\xd6\x9f\x7f\xadaI\x7f\xf3\xf6\xda\x0f\x80\x13\xc9\xb2\x84\x145\xd3\xb15M\xb9\xacT`\xe2\x02\xa1\x0f\xba\xfb>\xf8\xa0\xbd\xe3\xe3\xd66\xf8_\xfb\xbe\xe1cG\x8f\x03\x91\x0e\xba\xff\x9d\xf7\xbc;\xd39\xacv\x7fD\x0b\xbb\x7f\xa8\xb4\xe7gX\xe3M\xf9\x7f0\x08*\xd4\x83n\x82\xe7\xe5\x16\xe5\x84\x05b\\\x8c\xa5\xe4,\x1e\xf2\xe6\x88\xde3\x92\xd4}\x95\x9c^9\xcd\xa5\xdfh\\\x03\xb1\xd2LH`\xe0\xfa\xfc7\xa8x0$}\x8b\xed\x04!t\xe5\x10\t\x9e\xb5Q\x9d\x1e\x05]\x15,\xbe\x95\xea/\t\xded\x87\x0cE\x82\xd7T\xd1\xfbU\x83\x0e\xc5\xf0\x84K\xdb\xd5\x0b\xbf\x83>E\xf8/BD8gr\xe0Y\x11\xdc\xb3\xa8\xa8T$<\xa8\xef\x15!\xb2\xceR5Y\xf2hO\x97\xe3\x14\x9dfm[\xd4Nv\xd1=\x98;\x93\x94\xf3\xb9\xe9\x9c\x114Q)\xa1\x7f\x08.\x0b\xc2\x15\xc4 y\xa4M\xd9D5Y0\xac\x9cY\x02\x05\xd8\x82\xab\xc6\xda\xacR\xceT\x98\x05\xe1N\xd5\xc0H&\xec\x8ab\xc7\xea\xb5\xd7(\x0f\xb4\x01\x9d\xde\xe4u\x16\x93\xd5\x97\xdf\x80Fd\xec\x87\x08\x8a\xb9L:\x84\xf0\n\x1b\x11\x88\xbd\xe6$\xf7\xb0\xf59\x8d\xf11\xca<\x82\'\xdbk\xd1\xa4\x8e \xe2\xf9\xf8\xb1\xff\xed\x15\x10I\xdb\x1b\xa3\xf5\xf5M1\xd6\x10\xf4\xc3\xd2y,\x99\x9e\xdd\xe2I\x10\xd5\x85-o\xf2\xa2\x188@\xe5\xe2$\x9b\x13\xbc\x01\xfa\x00S\xeaOGI\x81Q\xfeg\x13\x0c~\xfd\xc6\x15\x7f\xe5\'4;\xc3\xebh\x14\xae=\x96\x98\x1e\xa0$Sd@nx\xbc\xaf6Y\t^\xc6\xf4\x07\x92b{\xfd>*\x96\xbf\xdf\nF\x04\xa9\xe9I@P%pE,=\xb5\x05\xae\xb2\x96\xdb\x12U\xaa\xcd4\xb7\xb7s8\xa8\xad\x15\xbeTB1!\\\x0b\x182\xe0\xc7\xf6\x8e6\xd1\xb5M\xad\xe9v\x81\x8d"\xb9$5\xd5\x8dd\x00\x0fM\x01!\x98\x03V\xc5\x0f\xfa\\WZ\x98\xd6\xa9\xcd\x81\x05&r\xcc\x9e\x8cOz\x8b7\xbd\xdb\xcfe\xa5\x14\x8dBy|\xdf{v\x05\x045\x06\xab\xe5\x9cW\x11\xc0C\x83A"\xa0\xb5xF\t\xc5\x1c\x06\x8bL\x9c\xe0D\xe5\xba\x81<\x13\x90:\xa6\xed\xca\xf6r\xcam;-8\x95\xcc\xb0\x90\x7f\x14Q\x13(,\r\x84\x88\xbe\xbeT\x9f\x9fbS\'hIPO\xd7\x8c\xd3A\x97M\xa82Qt\xd8\xb8\x04\xc4\x88Jo\xa1`\x03\xa1\x19\xaa\xea\x9c\x98\xf6\xafm\x85dq\xd4\xe9\x071\x94\xc6\xa5&\x95\xc0T\xb9\xfa\xa0\x0eq\x80\xc3?(\x1d3\xc8\xcc\xb9\xa02\x99\x92\xf3\x01M\xc9\xc4\r[t\xbc7B\x18\xfe8Q[\xbe\xa4\xc2\x80\x80=\xcb8\x8e\xe3\xb1\xf2\x16\x89\x9c\xa4\xda\x04\x1c\xb3vj\xdf\x87mmmrF]\xde\xa2\xe6\x02m,i\xf7A\x94\xe0\r\x9a\xd1\xc0\x11\x02\xacI\xb9\xc0\x9f\xca\xf1\xe4\xbc\'\x06\x96Y\x02\x9e\x10$\x83\xc1\x1d\xe4h\x0e\x9b\x1fD]\x01\xa9\xbcz%Lg~7\xdb\x19\nZN\xee\xca!\xa6\xc8\x01\x94\xeb\xee\xc1h\x1e\x92\xbf\xa4\x11\x11\x05 e\x9f1\xfeZ\xec\xa5\xa8\x18\x9b\x81\xcb\x93<w\x8c\xf3\xd7\x11(\xd8H\xc4\xf7S\x84\x10\xde\x8d8k\xcd\x17\x814\x8a\x05;\xcd\x99e\xd5\x17.\xe6!\xab\x01\x0e\xa1\x99\xc8T8\xcaC\x9c\xb5\xd9\xcd\xe8\xa5\x10[q5\x08I\xdb`~\x83b\x18\xd4q\xe7\xc7\x05\xd16\x0ckf\x99Mh\xc27u\xd20\xe4d\xb9X6s\x9d\xa2G2c\xb9\xe9\xce\x84\x7f\xe5\xeb\xfa\x8dK\xef6g\x83<\xef\xcd\xe9\x04\x96\xe2\xd9\xe5\xce\x84HR\x02\x81\x0fX\x81\x0cT4i\x1aY\x1b]}\xd8\\*\xfet\xact\xd1\xe1\xb1\xe5\xda\x1c%M\x9b\x93&\x17v\xb7\x18P\xbe4Z\xfdI\x18\xdd\xd5\x17\xa3\xf5\xcb\x17\xd0\xc1\x00;n~S\x14\x0ch:\x9b\x84\x04\xcf\x17\x90\xd6\x1b\x0fw\x92#\xa5\xd1>+rfT\xc2\xa4f\xa9\x05|\xe7X\x9c\x8c\xc8\x92|\x15\x83\x07x\xd0sp\x13k\x8b\xfa\xed\x8d\xeak\xcc\x8c\x8aF\x1d\xf0\xfa\x00\xfb\xb7\x1e\xd4\xfe~W\x87\x19\x01\xa3L\x92xk\x8a\x16\n`\x82\xa1\x84P\x1e\xc4\x10\x13\x88\x08\xa7R\xeeG\x94P\xc4\xd3%\xad\x17\x0c7Wl#\xf3\xf7\xac \x05\xc5\xc8\x9a@Y\xdel\x88i\xac\x93\xcfx\xf7\xc5\xb4fRd\xa8\xab\x7f\xf8\x1a+\xe3\x99\x18\x16\xcemo\xcc\x04\xb5\xe3\xca\xf2)\xf5\x83\xe0\xa2V*\x9a"\x90\x88\xa4\xfe1\x00\x82\x9d\xa2\t\xa6\rCv\xb2c+uB\x90\'gQ)\x03\xa9\xb5\xd1\xcb\xf6\xd8\x04\xd3\xe4\xf6\xdaS\xfc\x1d\xb5\x9d&\xf8\'\xb5G\xf7\xe0%t\x15[F\x13\xdcf(\xaf\x86\x1e\xb4\x97~\x05\xce\xb6L\xda\xd2\xaf>\xa4\xab\x17~\xc0\xc8 _\xa56\xc9\xbf\x98vY\xcc\xf9\x0bb/\xbc\xdd\xbbP+\x89\')\x98\xfc\xcbc\xd5\xf5\xcblO\x06\x864\xe8\x8es?\xea\xd3b\x99rE\x9f\x91\xa2\x02\xcd\x92\xa7\x01\xd2\xb0}\xc8\x07\\f\x8cP\t\xc6h\xe6\xb2B\xee\xa0c5\xd5L\xc5\x0b\xac\xa2U\x9c\x88`\xa9-\xccU\xaf^\xac.\xbe\x80\xddF,\\\xba\x85\x83\xb0\xc0,\xba\xbb.\xa6\xe0M\xce\xd7G\xb7\x84`"E\xd2\x8f\x85\x7f\xc8\xb1\xba\xaa\xfa\x1c\x14\x15\xda\xd8$s\x06\xcd\\\xae\x05\xb4\r\x16\x10\xd8\x94\xc4P\x13\xda\xb1\xaa)\xcf\x9a\x9fK\x80\x02"\xd4\xe2\xc3$\'\x99\xf8t\xb3I\xb8_\xcf&\xf8#@W\x9d{)\x06\xfaE\xad4\xca\xce\xb2\x19\x1a#I\x8d\x16\xae(\xd2k\xaa\xfeD\xe3\x9bP\xad\xb4\xc8D#HoW\xd5@\x88\xd0\xe9\x9f\xcd>\x94\xbb\xae\xe5\xbaQ-L>C}\xee\x15\x83\xc7\x82=P>\xe4K\x88=\xbep\x07nf:\x88D\xe6%\xcc\x7f4>\xeehk\x8b\x06\xdfeP4\xac!\x91\xce8(\x88\x84\x07\x16\xcaP4\xe2\x1e\xd1\x88\x06(oo\xeb2\xec\xbf\x108j$\x8c\xfe3\xd6\xdd\'\x98lXwg\x13,\x9b\x98$\xc0\xceU\xda\xb0sX\xfd\x89\xe3\x071OH\x8f6\x84e)X\xe3\x03\xed\xb6D\xa46\x8f\x8f\x1e05I)\xe4\x16w\x1d\x01\x1fI\x96\x11y?=\xf9\x1b\x8a\xa3qN\x8f\xc3\xd5\x94\xd3\xe3\x98O\xf0d\xcc\x84q\xc1\x83\xe6F\x05N\xb3|\x94|\xbf\xeelh\x91uM\xad\n\xb6\x88\xb5\xd40\x90\x8bY\x02\xa3\x1f\x98\xa3\xdc?\xd4\x98\xd0\xd1\x96,&\xe9\x91\xfd\xc6\x93\xa0\x18\xd0F\xe0v1\x1d5)!\xb8\xde\xb0\xa4S\'\xc2\x90\xb0\x1b\xa5Rt\xa8r\xf8\xf7\x01P\x0c\x02o:zH\x13S<\xc4\xaf\xb1W\x0b\xff)A\x15~\\\xb0\xc1\x910C\x91\xca\xfb\xb0x\x11k\xcd\xecFP\x84\xc0\xa2\xd3#X\xec\x0b[er^\xd7M\\\xe2\x17\x85&\xbc\x8f\x83\x18\xf7L\x0f$5\xd2\x13\x81l\xb0\xd8l$9\x94J-\x18\xdb#\x87\x04\xd5\x18\xe3\x98\xda\xaf\xd0\xcd\xa5\xe1f\xc1\xd8\xc6\x08\x90<\x8f\x95-\x9e\xa5[ \x06X]\xf2\xf6y\xcf\xbf\xe7\xfd\xf5\xcf\x8f\xfb?\x8e\xb1<\xe3\xac\x1d;6Q\xe0\x85\xbb\x1e\xe3\xa4K\xb6\x94\xdb\x1a\x87q\x15f\x90\xc6\x11\n4\xa5\xa4\xe4\x00J*\xd5\x0c)\x87H\xb6N\xdb|\x14\x1a\xaf7Qn\xf0\xb4o\x96fJEE\x7f\\2]7\x8a\xe8\xe3,\xa2\xd0\xa3L[`\xb1\xe8Ci\x99\x1e\xb0\xf0\xb3`\xa1\xc3\x0e\xf6\x1d\x18\x11\x19\xdb\x15\xcc\xa4W7\xcb\xa0\x14\x07@n\xd4\xc7\xbea\x9aD(\x17o\xd6GG\xd1~!\xc8j\x9b\x8fA\x10\xe9\xb1\xbbx\xba\xfdgb#\xd1\xb5\x05\x0e\x1b\xfa\x93\x14\x1a\xd9\xc5s\xd3\xd6A\xee"\x11\xc1d8\x14\xda\xe8\xca\xed\xe4\xd2\x85\x9fx\xd0\xe6\xd7g+\x0e\x95zp8L\xa5 t\xa4\xaa\xde\x90\xc6\xa7p\xb4FM\xedT\xce\xd2nP\xc5\x88K\xca{\xff\x8f\x9e\xa3G\x0ct\xfad\xa0\x1a\x00\xa0\x88\x91\x92\x07X@\xfa\xdf\xf8p\x11kP\xb6e`\x9c\xae\x17\xcc+\'Np\xd23\xd9\xb8\x14,\x86y{\x84\x90\x0b\xdc\xb1\xa4\xeb\xf6\xd5\xda\xc37\xa2@\xef\xdc\xfa\xf6\xfae\xb8a\xfb\xcd\x0c\xdf\xb0sb\\\x08\x8df\x0c\x836Ee+\x1f\xf7c\xc6\x1ctC4\xdc\x90\xf1\xe6\xf3~\x1a\xefn\x14\xebD\x89\xc4\x07\x9d\x14\x85iEY\xe4Jh\x9a\x1a\xd9I\x94 \xf1oJ\xe6\x10\xe5%\x1a\nuU\xf2\xf4\x8c5\xe4F[tuFg\x88$\xf6hl\x07\xdb/T\x98\xcbYT\xce\xc3qync\x93\tu\x12`)p\xa7@\x06\xa5x\xf1J\xa2\t\x85\xfd\xf0H\xe3"\xa8\x9cG\xd1\x81\x18\x9d[$\xc2\x15\xfc\x83Y\xd5\x1dU\x15\'\xc2\xea\xdf\xdd\xd4\x93*\x8aX\x83@9;-\xaf^m\xaf\x9f\x8bB\x0c\x9b\x8f 2\xb0\x81\x93OE\x10\xeac\x1c\xd3s\xd6\x01\xf4?\xb6\x81\xa3\xe3^\xcc\x86\xc3x\x14\x9f\xc0Cr\x1aJ!\x04\xde)\xdf \xa6\x1c\xc8\xe2\x00\x14\x02\xea\t\x8dmu\xba\xc5\x18\xe0\x98\xd2\\\x04qB\xbd\x91s\x99\xc1\xd2\xa8\x12 Fz\x89\xea\x0e9\x98P\x04\x0fPk:\xb2V\x8c\xaa\x11\xf1m\x1c\xb8\x0b\x02\xac\xad;<M\x1fS(\xaaQ\xa4\x17\x89\x03$\x17\x16\xfdG\xd3"f\x1eg\xbf\x87T\t\x86\x04]\xab\x9c\xc2\xe0\x82c\x135\'\xd9\xcb\xa4ua\xf5\x07.\xf2\x14/\xf04\x0e(Q\xc8\xc4Vc\xc7\xa2\xb9\x9c\x9d\x87\xcc\x9ck\xc5JL\x9c\x83\xa3f\xea\x91\x87,\x83\xddJA\xc9Bq\xe9\xf8\xc1\xb5|\xbc\xad"*\x9c90\xa8\x9ce*s\xa7za\xca\x9e\xe8\xd6\xe7/\xde]\xaa*\x90\xac\x92\xe0\xd9`\xf4w\n\xb6\xab\xbc\xcb\xde\x1e\xc0T\xa8\x9d\x96f+\x9d\xf8\xc3\xeb\xa4\xc9\xb3\xaeh\x89\xe2\xf2Y\xf20X\x98\xec\xb4\xbb\xe0\xb5b!\x85f;\x8b1}K\xcb,>PgP\xa008\x15\'vk%\xdbw\x12\x99\xbe\xb7\xc3\xeb\xba\xce\x8aNZ\x01?\xef\x01Ra\x0b\x15\xc1\xa8\xc7\xa0\xec\xc3<\x04\x17\xa5\x19\xdc\xaf\xb1\xd3"\xb8>B\x04\xf0yb"\xff\x0c4\x8a7\xbe\xc6\xf0{WE\x81\xce\xee\x0b\xd9\x91\x19#;\x14\xbfA\x9a<\'\x8f\x82<\x1e)\x02\xb0\xea^\x8e\xf3\x08\xf7\xf4\xc7\xec\xcb\xeccq\xea\xd6\xd3\xfb\x9a\x0c\xb9q\x81\xa9R\x96\x85\xedDt;lK =\\\xbdb\x9b\x06k\xa9\xb2\x9dF`\xba\xfb\nHf\x81\xc9\x9f\xb7\xcbe.\x07\x91F6H\xf6\x0c\x0e\xf1\xd2\xea\x08wX\x98\x92\xf2\xdaBt)\xbf\xbd\xf1\x00\xa4\xbc\x9a\x8c\xfe\xb3B\x056\x0c\xe5\x86\x8aA\x04\x02\xb2\xd9h\xbcH\xea\xa0\x89w\x14cP?\xb7\x8bZ4X\x04*EE0\x05K#S\xcbw\xdfP,\xbd\r\x05\x8cnc\xed\x15\x9e\x94A\xd2\x13\xd3Kko\xa2\'P\xed\xf4@Q\xf1\x977\xcf\xda\xf9\nF<\xc3K\xa5\xafQb\x86/\xd3\xd4\xa7\xf7v\x81R\xd5\x8e\xca\xa3QH\xaa`!\x93\xc8\xa4\x80keg\x87d\x9c<\x94\x16\xdca\xdd\xe3\x93\x1c\xa9\x15\xee\x1eY\x1dt8#\x0e\xe6\xe0cQ0\xd6pq\x86\x83\xd2\xd5\xb9\'\xf5\xef\xee\xee\xb6\xf4\xd8\xd2\xc7\xa4](c6\xaeh\xca\xc9]\xba\x15\xc8X\x8a\xbbK"\xf0\x8fF\xdb\xce\xb4 Jh\x8aT\'\xa5\xa9\x10<`\x8ai\x83\xa9\x01!\xc3i\x07\xd9,bL\x99\xe9j,\xf3\xee\xcf\x17U\xfea\xad#N\xce"z@4Q\x7f\x92P?\xac\x92\x81fv\'\x15\xa5y\x19\xc8N\x1a\xcd,\xd6\xbd\x0b1\x08}ZR\xddd\x98)\x90\'\xc3\xa8\xac\x04\xd1G\xb3Q\xb0\x06\xc8\x0fE\x17T\x14N\x89(\xa1\xf5s\xcb\xe6\xe3\xc4D\x8eB\xcbN\x04\x02aM$\xdf8\xb9\xe2_Y\xde\xde|\xf6\x8b\x16\\\x10\x03\xd3\xdb\x80Q\xec\x02n\xbe(\x8am\xd6\x10\xd2\xf4\xb38\xe8\xa6\x82\x18\xd1e\xc5\xd1\x043\x9d\xaep!\xca\xcf-\x8f\x07\xf6\xfb\xcb\x0b(\xcb\x80\xec9\xf4A\x16\xd4/ZC\xd4\\\n\xc5\xccw\xb2\x87\x82Q:\x16\xeaV>%\xac\xac\x85\xb6\xfbL\'\x93CQ\x0evn|\xadM\xa8\xab\x17\xec\xbc\'\xf3z\x80\x8b\x0fT\xa00\x08\x1a\xe6\xe7\xdf\xe8\x05\x15J\xc8\xed\x14Z\xd0B\xa4-FLb\xe4\x8f\x9d\xc6\xde}1!\x07\'#N\x90q1K\x93\xd6\x97\xf4e\xc5r\x86\x82\xa2\x9cL\x05C\xe7H\xb9*I\x1e\r\xf2\x10\x91m\xaf]\x02W\x83\xa3\x9c*>-\x929\x94\xdb\xe1\xd9*\\\xbd\x83u\xf7\xcf\x1b|\x8e\xd8\xc4\x8eX^\xd4G"\xcb\\@\x9e+\xe2\xa9\xb2Y\x95\x93\xc6\x8e\xd2\x0c\x0e\x83\xa0\x81@\xf2\x90&\xec\xbftcbTBXPY\x80\xb7x[\x1d\xbf\xe4\xdd\xfb\xc6\xbb\xb4\xa8\x8e\xe7\x90Uj\xa3\xb5\xb7_\xa1\x92\xa5t\x80\xf1Y\xb1/\xc6s\xe2\xe4:\x1d\xf6\x11\xe7"\xa9\xb4\x9e\x80\x9f\xf3\xe8\xbc\x0c\x0c^\x9b\x14\x17\xa1pE\xc66\xc1b\x00\xfeO\xbbq\x86\xb5J\xed\x01P\xbc\x02&\x11\x15\xd0\xae/\xdc\x07H\xe3\xd8#&\x15\x9d\x08\xd7H\xc8\xa5\xdf\xf8\xf7B\xc2x\xdf\xf8\xec\xe8\xe1V\\T\nc\x16IJ\xa4\xb8\x9d\t0;sf\x1a\x14\xe4\xa9\x96\xbd\x1f\xb6\xb5\xed;\xbd\x93Kn\xbb\xc1\xd0\xaf\xb8\xf4P\xbc\xfb\x1d:\xfdE\xa6\x86\xe3\x13B\xa1=\xe0\xdce|\xc7r\\\xec=(\xd9\xf8Hu.\xd3\xaf\xd4=\r\xeb\xa2\xca\x7f\x99\x04-b\xd1\x12\xd2\x1f\x85\x14\xcd \x07\x16\xbf8\xc1\xce\xd7(\xb7\xc9\x89YZ\x07l\x1aW{r\x90.\x86\xb4peq4%\xe7Hr\xc2S\x9eD)\xdc\xdeb\x81\x13_A\xbdI\xce\xe4\x14\xbd\xaa\xf6\x8bQ\xa2K\xcb\\x\xcc\x890L\x1e\xf0\x98\x14e\xfd\xcb\xb3`D\xd49\xb6\x1b\x81\xe27\xfc\xeeN\xe3T\x9e\xf3\xaf\xe8Q\xe1B\xd8\x94B/\xa0\x12\xb2\xa2e\x82\x96\x0f\xec\xc3e\xe7O\xb5\x9d\xd6\x1d\xf9b!EO\xc5\xaa\x838{\x93\xcf%\xa6\xb4\x10\xa0\xc2\x953}D\xe9\x1b9\xb0x(\x84\xad\xb5e\xefhF\xdc\xba\x8a\xa7=p%\xb6,*c\x93\x82\xca\'.\xe3\x01n1\xb5\x1eQW\x88\x90\xb0c\xa0\xa1B\x93\xea9t\x85w\x066\x0f7s\x02\xe4\xa2\xf3z\xf7\xe8\x82j\xcb\xc4\xe2M\x826\xf0\x0bv\xdc\xa0\x1d\xe4\x01:\x03\x0c\xf6\xa9\x96\xf6\xd3\xa7\xdaO7\x05\xf5)1\xfe\xb4\xd2\x93,\xcf\x14\x9f\xf3\xfe\xaa\x03,\xa9l\x9cC&\x81\x1f e\x01Q\xc5\x8eQ\xf7\xed\xcdu\x14M\x0b\xcb\xbcJ\xce\xaf\x07\xe1\xacK\x97Pd=\xbd\x8b\x89\x10^\xba<;\xba:\xf7\x12+\xa7DF\xb7\x11\x01r\x98\xed.4\xc9W\xc0eu\xc0\x8b\xfd/H\x89D\x7fz\x0f\x9a\xfe/\x9b\x08\x1f\x17/)\xe7\xc6\xb0z\xc9o\xf8\x91\xbf\xa1\x97\x8c$\x1a\x1e\x13\x8fe\x812v\x8a9i!\xdf$\x91\x98\xe6NQwW\xee\xe6\x80\xb1j\x84 \xf1#\x18\x9c\x9c(\x1f\xf8\xfd\xa7\xab\xfew\xb7\x1a\xf3\xb2\xf2_/\xa8\x833\xba\xa9$L\\:n\xa5L\xfaK9\xba\xd8\xde@e8\xd2\\\xa1T8\'\x90P\xf3\xa1+A>o(\xaa\xbd$\xcer\xf6\xef~\xa7\xea \x85s\x13N\x84\x0b\n\xa0$\xbe\xff\xe2zujR\x07X\x99\x7fX\xde\x97\xb5\xcf\xe2.\xa2>\x90\xb0|`|\xd4\xb6\xaf\xb5=;b\xe0\xa4\xbb\xab\xdf\xeej\x0f\x1e\xe7o\xc2\xa7\x87q\'e\x0b\xe9\xc2H\\\xde\x9b|\x85\'z]\xb9\xa5V\xc0\x968^\xbc\x0b\x12\xe5Q}b\x96OX\x07!\xda\x00\xb5\x80#\x19\x02\x81kq\xca8\xc6Si a\xffh5\xe8A\xc6\xd5\x0eU\xa7\xea\xf0i\xa1wY\xae\x89\xa5\xf9\xcb3\xa8ii\xe3\x03+n\xec\x8a\xb7z\xb1Q1\xa9\xc4k\\\xd2\xf5\xdf\xa8\x01\xfa\xc0\xb1\x93\xbc\xdf\x9c\x16\xd6\xa6(9\x95\x82\x1b\x9c\xb2\x8db\xa3\xd1\x0e\xf8&`{q\xcc\x1a&\x12\x11\x83\xb4\xdd\xdbo\xaea\x98J\x15n\xc1\xcb\x1aR\xb0X\xef\xe0\xe6,\xab\x94lW\x05\x7fT\xc5\xabG1\x85\x0b\x8eR\xcbv\xfb\xe59E\xccb\xd4k_\xc9\xe5H\xbf\x13[\x05\xc3\x85$3\x81\x1e@\xa1\xab\x9f9\xce\x9as\xeb2\xf6$>\x9bP%e\xca\x87\t\xaa\xadB\x9eV\xe0R\xa9rn\xb3\xd7\xe2\x035\x8545\xdd@zj\xfd]"\xd2C\x82\x91\x8f\xd9\x89yE\x83#\xc4\xf7\xec@\xf5a\x8ag\x91\x1e\xaah\x16\x81\rUI\xc3\xee\x80VM\x86\x14u\xe3\x02\xb8\xd7AM\xdfn\\\x99\xdc\xa9<\xa5\xe9\x972\xaa\x16\xb8gx\x95\x18\xa2\x88\xa3\xaaj\x97\x1a\x13\xad\xc9\x98Z\xf5(\x03b\x7f\x17\x8dr\x81+V\xd0\xbe\xa5EZ\xf9\x18X\xae\xacW\xdc\')\x08XD\xd4?V/\xdc\x01:\x16\xa7J\x13\x8d(\x95\xc5> W\xb0*\x81\x81\xf3\xf5\xe2R\xa0\xb4\xb8\x9c\xaa\x83"\x8b\x94\x82\xf4\r\xe7YE`\xd4\xaa\xcfQ_r\x0b\xb4~t\x95Hk\x12\x82B\xe8\x88T\xcdr\xbdpX\xf0\x90ue\x8bnhW\x1f\xd3\xadI\x9b\x8d`\x14\x9aH\xf6\xae\x9c\xdf\xdex\xc0\x0b\xc7x\xf9\xfa\xd5_]\x14\xcc\x15\xe1\xa2Y\xa3\xd2\xab\x08\xbc\xb16\x98\'\xfc\xe9\xb1\xfa\xddK\x84\xf1\x98\xebt\xe4\xe0E\x10\x104\xf7\x1f\xc5W\xe4\xfc\'9L\xcd\x16M+[7#\x879\xa9qc\xdc\x9e\x18\xf2r\xd4\xe1\x95<\x1b\x15\xac\xe6\x12wqS\xaaCMT\xe3W%\x9aZ\x1d<95\x99\xf8\x1d\x8f\xd5(\x05s\xe2\xf8\xe0\xc7\xd6h\x97\xbd<\n\xa1\x89\nv\xa9*\x86\x0f\xed\x06\xd9G:\x9bW%\x86\xea\x8a#\xd9\xc44\xb2\xd9U\xff\xeb\xa7x\xc25\x1dH\x17\x9a\x87\x1b\x07\xad\x98\xee\x8aP\xc5\x0e\xc6\xcd\x97\xf6\xca9\xb8y\xfa#_\x1cH4\x89\xa1Yr$<\xf9*\xd2\x94\x11\xa3\xdd\x8a\x02H\x9ca\xec\x9a\x03\xdc\t\xc2\xa5%\x8dCg0\xe6\xf2\xf8\xbe7y?$\r\x85\xc9G bWt\xe8\x184>$\xf6\xcdE\x7f\xe91\xee\xd0\xe43~^\xd8|\xe3\x1d\x08\xec7qvD$\xae\x02\x98\xa5H/\xc6#\xe8\x14\x00\xe4M\x81b;\'\x0e:\xc5#\x85\xf9\xc0\x05&\x05q\xe4\x17u\xfa\xb1\xc3\x8b\xd6\x82\xa2\x0c\xee+\xe7C\x7fD`N\xac\xe3\x8c5\xd4\x993\xf3\xbd\x19\x93\x86<\xef\xe33\xcbQ\xea%\xf1`\xb0T\x1ee+\xf7W\x98\x03\xc1<\xae\x16l\x9a\xa5\xe19L\xc0\xb0=\xa2\xb0\x0f\x13&\x1c\x95/:\x94\xda$5\x83B\xfd\xe6\x12\x17\x85\xe2\to\x91\x9b\x83\xb1\x80\xe2`\xb2\x83\xdd\xc7\x01\xce\xe3G\x8f\x9e@\xd2\xd5\x0e\x14\x128LDN\xd4\x8d\xf0\xd1\x90\x92%4\xbb\x04\x881\xc2FA\xd6=x\xe1/\x9a\xd5\x17\x99\xd1\'b\xd0\x8cXQ\x8f\x81\x03\x9d0\xd7\xe6\x8a\xae=q\xe2\x99Ff\x91\x93\x16\xbc\xa9\xd5`\x82\xb08\xfb\x0c\x03\xb9\x80\x01\r\x1d4\n1\xfe\xf0\xb6\xdd\x0eI\x1b\xd6\xf7^\x9b\x02\xad\x1f\xb9\xa8\xba7\xc4xsUn\xaa\x1fwXrl`S\x81\x7fB\xac\x88F<V\x0b\xe1U(\x03\xb0\xf6\xf0b\xed\x9biq\xfe\xad\xa2E\xb1U\x9d\x86\xb6R\x04\xe7\x08\x9f\x8c\x10\x84\x0f\xe5\xa1,\xd8\x04\r\n\x11\x89\xc0Ug\x9bQ\xf0\x1a\xe7\xb8\x9f[\xd7g0\xf3Y\x15\xb2\xcb\x9f\x07+\xab\x9e\xdc\x8ef<\xa7\xbd\xad)\xa4\xfe(\x10\x83\'E\xa8^,u(\xb3\x8d\xd1\xb5J!\xa3\xb7\x86\xe3\x00\xcd\xd5\xc8\x82U\x10&\xd42]\x16\x0bleR\x0c\x0f\xb2\x08\x11\x91,\x8d\x15]\xa1\xf2L@d\xc4\x86\xf9\x8e\xf20\x87U\x06\x80\xc9\xbaq\xfe\xb4\xe6iic\xbeI\xc3c\x01\x11\x1f\x0f\x01\x1b\x1e\x9c1(\xb4\xdd\xf8L\xf5\xcd\xca\xf6\xda\xb48\xb2\x92\xd0\x19~|\xdc\xfe\xb5\xa2\xa2 %\x91\x0c\x8e\xef\xa3/\x04y\xd2N\xeck\xdb\x93\x19\t\x7f+\xc8Qt\xf5\x89\xa9\xa3\x81\x97\xcf|%2\x1fb\x0e`\x94\xd5\x85\x84\x11\xc5\xebr\xee\xa8L[\x8c+$\x052&f\xe6%\xb3G\xb3x\xb8\xde\x82\xc59\x17<\x91\xba9\xe4\x8a\xf21\xc9A\x03b\xdaLc\xa2\xc6\xea7\x07\xec\xa2v\xc8\xa4<\xe3p\x8cmY<\x14\x82j\x1c\xbd\xb7OP\x86\x07\x15\xff\xa2e\x99U\t*\x0e\x86e\xe4\xdd\x9b%\xfe\xa9\xf1\xab\x0c\xdd\xb0\xcc\xc6\xf2f\xb7\x1c:C[\x9c\xb2-&\xb4I\xb3\x97J\xc7>\xc0\x91\x84T\x07\x95\xb6r9)\xc2I\xd1\xb0\xf5):,\x94R\xe7v\xa6\x9f\x1e\xea3\xbe\xb8n\x0c{;\xe9\xec\xf4\xc6\xd3\xb5y}\xb1\x93\xaa\x84\x9d\xe4rP\\Uca{S&\xc3\xa3O\xa5\x13\xa8\xf9\x86\x12X\x11\x95`Q\x8b\xf3:q\xfb\xa9y\x85N\xe8<\x00\xb7\x19\xb0\x9a\xda\xfa\x13\xd5\xf9\xc6\xc0\x06\xfb\x90\xf8_K\xdf\x18x\xee\xc0\'X<\xa4\xf6D=\x93\x1f\xc5\x07\xda\x06go\xb7\xb7Q\xa9Q\xc3\xc2a3\xfe7\xa3\x1f\x90\x99').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['t2v', 'i2v', 'flf2v', 'r2v']
REMOTE_MAX_IMAGES = 9

'''
Keep job directories isolated between different notebook services.
各 Notebook 使用獨立工作資料夾，避免不同服務互相干擾。
'''
REMOTE_ROOT = Path('/kaggle/working/remote_api_jobs') / REMOTE_SERVICE_ID
REMOTE_ROOT.mkdir(parents=True, exist_ok=True)

'''
Share existing state when the API cell is executed again.
重跑 API Cell 時保留原有工作及鎖定狀態。
'''
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('REMOTE_ENGINE_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=32))
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(2))

'''
Results remain available for up to 24 hours, subject to an 8 GiB storage limit.
結果最多保留 24 小時；若超過 8 GiB，會提前清除最舊結果。
'''
REMOTE_RESULT_TTL = 24 * 3600
REMOTE_RESULT_BYTES = 8 * 1024**3
REMOTE_MIN_FREE_BYTES = 2 * 1024**3

'''
Do not start a second GPU job when a long-running backend appears stalled.
當底層生成疑似卡住時，停止接收新工作，絕不啟動第二筆 GPU 推論。
'''
REMOTE_STALL_SECONDS = 2 * 3600

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        import torch
        if not torch.cuda.is_available():
            return False, 'Kaggle CUDA GPU 不可用；請啟用 GPU 並重新執行模型初始化 Cell。', []

        '''
        Use the model's actual GPU requirements, not the number of queued jobs.
        依各模型原本的 GPU 配置檢查，不因佇列工作數增加 GPU 使用量。
        '''
        qwen = REMOTE_KIND == 'image_multi'
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            selected_gpus = [globals().get('GPU_ID', 0)]
        elif qwen and globals().get('QWEN21_STATE', {}).get('mode') != 'bf16_dual':
            selected_gpus = [0]
        else:
            selected_gpus = [0, 1]
        available = torch.cuda.device_count()
        if any(not isinstance(index, int) or index < 0 or index >= available for index in selected_gpus):
            return False, f'需要 cuda:{selected_gpus}，但實際只有 {available} 張 CUDA GPU。', []
        devices = [f'cuda:{index} {torch.cuda.get_device_name(index)}' for index in selected_gpus]

        '''
        Validate resident PyTorch pipelines and their designated CUDA devices.
        檢查 PyTorch Pipeline 已初始化，且核心模型使用指定 CUDA 裝置。
        '''
        if qwen:
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／GPU 記憶體管理器尚未初始化，請先執行模型初始化 Cell。', devices
            if state.get('device') != 'cuda:0':
                return False, 'Qwen Transformer 沒有設定為 cuda:0。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa 雙 GPU Pipeline 尚未完整載入，請先執行模型載入 Cell。', devices
            for name, module, index in (('HeartMuLa', pipeline.mula, 0), ('HeartCodec', pipeline.codec, 1)):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{index}'):
                    return False, f'{name} 未位於指定的 cuda:{index}，請重新載入模型。', devices
        elif yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            if process is None or process.poll() is not None:
                return False, f'YuE2 CUDA{gpu_id} 常駐服務尚未啟動。', devices
            if f'CUDA{gpu_id}' not in [str(value) for value in process.args]:
                return False, 'YuE2 常駐服務沒有使用預期的 CUDA 啟動參數。', devices
        else:
            '''
            Query the actual ComfyUI process; a running HTTP server alone is insufficient.
            查詢真正的 ComfyUI 裝置資訊；HTTP 服務存活不等於正在使用 CUDA。
            '''
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動，請先執行模型服務 Cell。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any(str(item.get('type', '')).lower().startswith('cuda') or
                       str(item.get('name', '')).lower().startswith('cuda:')
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 沒有回報 CUDA 裝置，已拒絕 CPU-only 推論。', devices
        return True, 'CUDA 後端已就緒（不代表所有權重常駐顯存）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task

    '''
    Confirm that the requested video task matches the weights already loaded.
    確認影片模式與目前實際載入的模型權重及量化設定一致。
    '''
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 程序已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))

    '''
    Check numeric types, finite values, and limits used by the generator.
    檢查數值型別、有限值與生成程式的實際有效範圍。
    '''
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        matches = [int(number) for number in re.findall(r'<Picture\s+(\d+)>', str(fields['prompt']), re.I)]
        if any(number < 1 or number > int(fields.get('_image_count', 0)) for number in matches):
            raise ValueError('R2V Prompt 的 <Picture N> 編號超出已上傳的圖片數量')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]


'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return

        '''
        Reuse the same job ID for retry attempts after an uncertain response.
        網路回應不明時，重試會沿用同一個工作 ID，避免重複生成。
        '''
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)

            '''
            Reject invalid parameters and incomplete model configurations before queuing.
            在排隊前拒絕無效參數與尚未完成初始化的模型設定。
            '''
            remote_validate_fields(task, dict(fields, _image_count=len(files)))
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)

            '''
            Commit the validated upload and queue entry in one critical section.
            在同一個鎖定區段確認請求、保存資料並加入工作佇列。
            '''
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value

            '''
            Resolve -1 once on the server so the exact value can be reproduced.
            在服務端一次決定 -1 的實際種子，方便重現結果。
            '''
            for seed_key in ('seed', 'lm_seed'):

                '''
                Ignore missing seeds and validate the supported 32-bit range.
                未提供的種子沿用 Notebook 設定，並檢查 32-bit 範圍。
                '''
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')

                    '''
                    Translate -1 into an explicit random value before inference.
                    推論前將 -1 轉成明確的隨機數值。
                    '''
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)

            '''
            Save supplied seeds for reproducibility in the job status API.
            在工作狀態 API 中記錄請求所使用的實際種子。
            '''
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        '''
                        Point R2V to this job's uploaded reference images only.
                        讓 R2V 僅讀取這筆工作的參考圖片資料夾。
                        '''
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:

                    '''
                    LTX-2.5 uses a shared ACTIVE_JOB configuration.
                    LTX-2.5 使用共用 ACTIVE_JOB 參數。
                    '''
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'

            '''
            Clear old output metadata so a failed generation cannot reuse another job's file.
            清除前次輸出狀態，避免本次失敗時誤取其他工作的結果。
            '''
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)

            '''
            Execute a fixed local generation cell, never user-supplied code.
            僅執行 Notebook 內固定的生成程式，不會執行請求傳來的原始碼。
            '''
            '''
            Recheck the CUDA backend after queuing to prevent CPU-only generation.
            工作排入佇列後再次確認 CUDA，避免後端異常時改走純 CPU 生成。
            '''
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):

                '''
                Include YuE2 replay metadata and score with the single MP3.
                將 YuE2 的重播參數與 ABC 樂譜一起提供下載。
                '''
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None

            '''
            Remove only source files bearing this remote job's unique prefix.
            僅清理本次遠端工作專屬原始輸出，保留 Notebook 手動生成的檔案。
            '''
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

'''
Keep one worker and one maintenance thread even after rerunning this cell.
重跑 Cell 仍僅保留一個 worker 和一個清理 thread。
'''
if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()
if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

'''
Close only the previous ngrok tunnel; preserve the running API and its jobs.
只關閉舊的 ngrok 通道，保留正在運作的 API 與所有生成工作。
'''
old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)
NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(Path('/kaggle/working/ngrok'))
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

'''
Always request the auth token again when this cell is rerun.
每次重跑均重新要求輸入 ngrok Authtoken，不顯示內容。
'''
ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')
result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')
del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

'''
Reuse the same API server when this cell is executed again.
重跑本格時沿用同一個 API Server，不中斷生成工作或清空佇列。
'''
server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()
if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:

    '''
    Update the handler on rerun without restarting an active GPU job.
    重跑後更新 API handler，但不重新啟動正在執行的生成工作。
    '''
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')
with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)
NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)
if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])
print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')


## 13. Web API 使用說明

In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request

'''
Request the API schema directly from the local API server.
直接從本機 API Server 查詢支援的參數與模式。
'''
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)
print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本地 requests 範例：06_MiniMax_H3_影片生成_Kaggle_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')